# Quantum streaming bandit experiments

Implementation of the streaming KING/challenger algorithm, bounded-depth amplitude estimation, destructive-SWAP comparisons, and IBM hardware execution. All saved cell outputs are removed.

**For readers:** The archived simulation and hardware summary data are in `data/`. Run `python scripts/plot_simulator_results.py` to regenerate the statistical-simulation plots without hardware access.

**For hardware users:** Live QPU submission is opt-in and depends on private credentials and the original checkpoint. This notebook is an archived continuation implementation, not a clean-start one-click reproduction.


In [ ]:
# Optional installation cell. Uncomment in a fresh environment.
# %pip install -q "qiskit>=1.2" "qiskit-ibm-runtime>=0.30" pandas numpy matplotlib scipy


In [ ]:
# ============================================================
# Imports and global configuration
# ============================================================
import os
import math
import json
import time
import hashlib
import sqlite3
import threading
import functools
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from getpass import getpass
from collections import Counter, defaultdict, OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Frozen hardware campaign; never reuse physical counts from incompatible runs.
FIXED_KD = 5
DSWAP_BUDGET_FRACTION = 0.25
PILOT_RAW_FAR_CUTOFF = 2   # raw 11 >= 2 is FAR; raw 11 <= 1 is CLOSE
DSWAP_EARLY_BLOCK_SHOTS = 64  # independent Sampler jobs per block, last block 35
DSWAP_EARLY_MAX_BLOCKS = 4
PILOT_PAIR = (66, 67)      # V11's characterized locked physical pair
REVISION = "theory_faithful_system1_v14_kd5_f25_t2_early64_full_delta001"
LEGACY_AMPLITUDE_REVISION = REVISION
EXECUTION_PROTOCOL = "v14_full_campaign_early_far_block64_raw11_ge2_qpa_scoped"
OUTDIR = Path(os.getenv("NIGHTHAWK_THEORY_OUTDIR",
           f"results_{REVISION}"))
if os.getenv("NIGHTHAWK_PREVIOUS_OUTDIR", "").strip():
    raise RuntimeError("V14 requires a fresh checkpoint; unset NIGHTHAWK_PREVIOUS_OUTDIR")
if any(f"v{i}_" in str(OUTDIR).lower() for i in range(7,14)):
    raise RuntimeError("Refusing an older-results directory: use a new V14 output directory")
FIGDIR = OUTDIR / "figures"
DB_PATH = OUTDIR / "checkpoint.sqlite"
PREEXISTING_CHECKPOINT = DB_PATH.exists()
# This is a CONTINUATION notebook; never launch from an empty checkpoint.
if not PREEXISTING_CHECKPOINT:
    raise RuntimeError(
        "Continuation requires the existing campaign checkpoint.sqlite. "
        "Set NIGHTHAWK_THEORY_OUTDIR to the prior run directory (not a new one). "
        "Restore SQLite + WAL/SHM sidecars from the backup before continuing."
    )
OUTDIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)
REQUIRE_EXISTING_V5_CHECKPOINT = False
# Resume from this campaign checkpoint only; do not import earlier physical shot counts.

# Independent instances may overlap classical/Runtime work; jobs are not QPU-parallel.
# Execution optimizations: per-thread SQLite connections and heterogeneous Sampler PUB shots.
ENABLE_MIXED_SHOT_PUBS = True
ENABLE_LAZY_CIRCUIT_COMPILE = True  # avoid recompiling fully checkpointed circuits
MAX_PARALLEL_STREAMS = max(1, int(os.getenv("NIGHTHAWK_PARALLEL_STREAMS", "3")))
MAX_INFLIGHT_SUBMISSIONS = max(1,int(os.getenv("NIGHTHAWK_INFLIGHT_JOBS","3")))
# Keep the rolling Batch below the applicable IBM Runtime time-to-live;
# the environment can override this for accounts with smaller limits.
BATCH_MAX_TIME = os.getenv("NIGHTHAWK_BATCH_MAX_TIME","7h")
BATCH_ROTATE_SECONDS = max(120,int(os.getenv("NIGHTHAWK_BATCH_ROTATE_SECONDS","21600")))
_CSV_EXPORT_LOCK = threading.RLock()
_PROGRESS_EXPORT_INTERVAL_SEC = max(30,int(os.getenv("NIGHTHAWK_EXPORT_INTERVAL_SEC","600")))
_last_progress_export = 0.0
_SUBMISSION_GATE = threading.BoundedSemaphore(MAX_INFLIGHT_SUBMISSIONS)
_COMPILE_LOCK = threading.RLock()
# Bound the circuit-compile cache so huge campaigns do not accumulate millions of Qiskit objects.
COMPILE_CACHE_SIZE = 3000


# -----------------------------
# Paper settings
# -----------------------------
N = 1000
FAMILY = "clustered_lowvar"
DELTAS = [0.01, 0.05]  # group-major: each group finishes .01 then .05
SCHEDULER = "alap"
KR_VALUES = list(range(0, 11))
HARDWARE_KR_VALUES = [3, 6, 10, 0, 1, 2, 4, 5, 7, 8, 9]  # all 11 depths; useful depths first
KR_PLOT_VALUES = list(KR_VALUES)               # plot every measured k_R = 0..10
assert SCHEDULER == "alap"

# No algorithmic stream randomization: arms are processed by stored index.
STREAM_ORDER = "index"

# Independent problem instances for statistical evaluation.
SIM_SEEDS = list(range(7, 57))
# Three-seed groups: seed 10 is intentionally last (old IBM jobs need recovery).
HARDWARE_SEED_GROUPS = [
    [7, 8, 9],
    [11, 12, 13],
    [14, 15, 16],
    [17, 18, 19],
    [20, 21, 22],
    [23, 24, 25],
    [26, 10],
]
HARDWARE_EVAL_SEEDS = [seed for group in HARDWARE_SEED_GROUPS for seed in group]
assert len(HARDWARE_EVAL_SEEDS) == 20 and set(HARDWARE_EVAL_SEEDS) == set(range(7, 27))
assert all(1 <= len(group) <= 3 for group in HARDWARE_SEED_GROUPS)

DD_CALIBRATION_SEEDS = [1007, 1008]            # disjoint from evaluation

# Matched logical budgets. The display/plot order is ascending.
QUERIES_PER_ARM_GRID = [10,20,50,100,200,500,1000,2000,5000,10000,20000]
HARDWARE_QPA_GRID = list(QUERIES_PER_ARM_GRID)
# Execute the largest reservoir first; lower-QPA points replay/subsample actual hardware histograms.
HARDWARE_QPA_EXECUTION_ORDER = sorted(HARDWARE_QPA_GRID, reverse=True)
# Scheduler-only controls (no changes to circuits, budgets, decisions or checkpoint keys).
V143_BASELINE_TRIALS_PER_QPA = max(1, int(os.getenv("NIGHTHAWK_V143_BASELINE_TRIALS_PER_QPA", "2")))
V143_STOP_BASELINES_AFTER_FIRST_WIN = True
V143_RUN_NEW_BASELINES = False  # legacy adaptive baseline selector remains DISABLED
V143_STOP_BASELINES_AFTER_FIRST_WIN = False  # irrelevant to fixed 6-run protocol; no early success stopping
# Six predefined no-dSWAP hardware controls (fixed before delta=.05 results).
V1437_ENABLE_TARGETED_NO_DSWAP = True
V1437_TARGET_DELTA = 0.05
V1437_TARGET_QPA = 20000
V1437_TARGET_SEEDS = (7, 8, 9)
V1437_TARGET_KR_VALUES = (2, 4)
assert len(V1437_TARGET_SEEDS) * len(V1437_TARGET_KR_VALUES) == 6
assert V1437_TARGET_QPA in HARDWARE_QPA_GRID
assert set(V1437_TARGET_KR_VALUES).issubset(HARDWARE_KR_VALUES)
assert list(V1437_TARGET_SEEDS) == HARDWARE_SEED_GROUPS[0]
assert 0.05 in DELTAS
# A comparison counts as a win only when both methods identify the best arm and
# dSWAP uses strictly fewer logical queries for an identical seed/kR/QPA.
# Baseline candidate selection always ranks correct dSWAP results by fewest queries.


# Theorem confidence parameter. Exact Algorithm-3 M,T are reported separately.
DELTA_CONFIDENCE = 0.10

# Finite-shot NISQ dSWAP allocation. This is NOT the theorem Algorithm-3 M.
# Stage-A-based selection for high-QPA runs (not validated as universal at low QPA).
assert 0.0 < DSWAP_BUDGET_FRACTION < 1.0

# Directional challenge follows the Game-of-Arms geometric-level shape.
MAX_CHALLENGE_LEVELS = 8
MLAE_GRID_SIZE = 4001

# ------------------------------------------------------------
# LIVE PAPER HARDWARE SWITCHES
# ------------------------------------------------------------
# Hardware campaign is opt-in to avoid accidental large submissions.
# RUN_SIMULATOR_EXPERIMENTS stays False because it is not a hardware switch.
RUN_SIMULATOR_EXPERIMENTS = os.getenv("NIGHTHAWK_RUN_SIMULATOR", "0").strip() == "1"
RUN_QPU = os.getenv("NIGHTHAWK_ENABLE_QPU", "").strip() == "I_ACCEPT_QPU_COSTS"
RUN_DD_CALIBRATION = False  # reuse no-DD decisions frozen in V11
# Skip already-frozen policies; an explicit force is required to rerun DD calibration.
FORCE_DD_RECALIBRATION = os.getenv("NIGHTHAWK_FORCE_DD_RECALIBRATION", "0").strip() == "1"
# More upfront independent contexts can submit together; does not alter circuit shots.
DD_PREFLIGHT_WORKERS = max(1, min(3, MAX_INFLIGHT_SUBMISSIONS))
# Start with calibration ONLY. The FULL campaign requires two explicit opt-ins.
# Changing these switches never changes V11's fixed kD/fraction or its result identity.
RUN_FULL_CAMPAIGN = (RUN_QPU and os.getenv("NIGHTHAWK_RUN_FULL_CAMPAIGN", "").strip() == "I_ACCEPT_FULL_CAMPAIGN")  # explicit opt-in required
RUN_HARDWARE_TRACE = RUN_FULL_CAMPAIGN
RUN_END_TO_END_DD_ABLATION = False
GATE_CAL_SHOTS = 4096
GATE_CONTROL_SEEDS = [2007, 2008]  # different from evaluation seeds
GATE_FAR_MISS_TARGET = 0.01       # model-based per far-comparison false CLOSE goal
GATE_MIN_PREDICTED_CLOSE_SKIPS = 0.20
GATE_CHARACTERIZATION_BASES = [0.49,0.50,0.51]
GATE_CLOSE_GAPS = [0.05,0.10]      # as multiples of delta
GATE_FAR_GAPS = [0.50,1.0]        # protect materially different challengers
# Absence of demonstrated separation prevents starting a large production campaign.
BLOCK_UNSAFE_FULL_CAMPAIGN = False  # exploratory threshold override; not a safe gate certificate
# Larger guaranteed-first-level amplitude blocks amortize Runtime submission overhead.
# Every job is additionally bounded by the total PUB shot/execution ceiling below.
MAX_CIRCUITS_PER_JOB = 240
MAX_EXECUTIONS_PER_JOB = 9_000_000
QPU_TIMEOUT_SEC = None

# Resume/checkpoint policy.
RESUME_PENDING_QPU_JOBS = True
# Retry retrieval of existing IBM jobs; never blindly submit duplicates.
RUNTIME_RECOVERY_ATTEMPTS = max(1,int(os.getenv("NIGHTHAWK_RECOVERY_ATTEMPTS","5")))
RUNTIME_RECOVERY_RETRY_BASE_SEC = max(0.0,float(os.getenv("NIGHTHAWK_RECOVERY_RETRY_BASE_SEC","2")))
AUTO_RESUBMIT_CONFIRMED_TERMINAL_JOBS = True  # ERROR/FAILED/CANCELLED only, confirmed by IBM
# Look up pending IBM jobs in their original instance after a CRN switch.
# Supply IBM_PREVIOUS_INSTANCE_CRN and, if changing IBM account, IBM_PREVIOUS_QUANTUM_TOKEN.
# IMPORTANT: a not-found job is NOT a confirmed terminal failure.
# If the old IBM instance is retired/unavailable, STOP by default rather than blindly duplicate shots.
# Explicit opt-in "replace" is for manual-verified inaccessible jobs only; it keeps an audit archive.
INACCESSIBLE_JOB_POLICY = os.getenv("NIGHTHAWK_INACCESSIBLE_JOB_POLICY", "stop").strip().lower()
if INACCESSIBLE_JOB_POLICY not in ("stop", "replace"):
    raise ValueError("NIGHTHAWK_INACCESSIBLE_JOB_POLICY must be 'stop' or 'replace'.")
if INACCESSIBLE_JOB_POLICY == "replace" and os.getenv("NIGHTHAWK_ACCEPT_UNRECOVERED_JOB_RISK", "") != "I_ACCEPT_POSSIBLE_DUPLICATE_QPU_SHOTS":
    raise RuntimeError("Replacing inaccessible IBM jobs may duplicate shots already executed remotely. "
                       "After checking the old IBM instance/job status, explicitly set "
                       "NIGHTHAWK_ACCEPT_UNRECOVERED_JOB_RISK=I_ACCEPT_POSSIBLE_DUPLICATE_QPU_SHOTS to proceed.")


SKIP_COMPLETED_HARDWARE_CONFIGS = True
AUTOSAVE_PARTIAL_RESULTS = True  # checkpoint SQLite always; throttle CSV exports while working

# Batched circuit execution preserves the sequential decisions inside each problem instance.
# Low QPA may not afford one amplified dSWAP shot per comparison: explicitly report underfunded.
MIN_DSWAP_SHOTS = PILOT_RAW_FAR_CUTOFF  # fewer than two shots cannot establish a FAR event
# Transport-only execution optimizations. None of these alter logical query counts or decisions.
# MLAE: batch the currently required KING+CHALLENGER circuits (at most 2*(kR+1) <= 22 PUBs).
# dSWAP: prefetch only a small block; grow while KING is stable, reset immediately if KING changes.
DSWAP_PREFETCH_INITIAL = 1
DSWAP_PREFETCH_MAX = 1  # no speculative FULL dSWAP reservoir prefetch
# Speed knob: batch ONLY the FIRST 64-shot block for a few imminent arms
# after KING stays fixed for 8 challengers. This can save many IBM jobs at
# the cost of some unused physical shots when KING changes. Logical queries
# never charge those unused shots; physical manifest and counters do.
ENABLE_STABLE_KING_FIRST_BLOCK_PREFETCH = os.getenv("NIGHTHAWK_FIRST_BLOCK_PREFETCH", "1").strip()=="1"
DSWAP_STABLE_KING_MIN = 8
DSWAP_FIRST_BLOCK_PREFETCH_WINDOW = 4
DSWAP_PREFETCH_GROWTH = 2
# Zero-speculation first-level prefetch for the no-dSWAP baseline.
# This block is used only if b >= s_1, so every arriving arm necessarily runs level 1.
ENABLE_NO_DSWAP_FIRST_LEVEL_BLOCK_PREFETCH = True
# Safe because this prefetch is activated ONLY when every challenger is guaranteed
# to need the first MLAE level. With kR=3, up to 60 arms can now share one 240-PUB job.
NO_DSWAP_AMP_BLOCK_MAX = 240
# Submit independent circuit chunks concurrently if one batch exceeds the PUB limit.
ENABLE_CONCURRENT_PUB_CHUNKS = True

# Readout calibration and hardware-count reservoirs.
READOUT_CAL_SHOTS = 4096
ENABLE_READOUT_MITIGATION = True

# Avoid V5 excess-reservoir floor; large-high-QPA acquisitions are replayed at lower QPA.
# Low-QPA-only circuits use small physical top-ups to avoid paying hundreds of unused shots.
AMP_TOPUP_SHOTS = 64
DSWAP_TOPUP_SHOTS = 1  # EXACT adaptive blocks, including small low-QPA final blocks

# DD calibration across every actual MLAE depth used by the final sweep.
DD_CALIBRATION_K_VALUES = list(range(0, 11))
DD_MIN_RELATIVE_MAE_IMPROVEMENT = 0.05
DD_MIN_VALID_POINTS = 2

# Small end-to-end DD ablation; representative depths are sufficient here.
DD_ABLATION_KR_VALUES = [0, 3, 10]
DD_ABLATION_SEEDS = [7]
DD_ABLATION_DELTAS = [0.01, 0.05]  # group-major: each group finishes .01 then .05
DD_ABLATION_QPA = 10000

# Backend / physical pair. Leave pair blank to select deterministically from calibration data.
IBM_BACKEND_NAME = os.getenv("IBM_BACKEND_NAME", "ibm_rensselaer").strip()
LOCKED_PAIR_ENV = os.getenv("IBM_LOCKED_PAIR", "66,67").strip()
if LOCKED_PAIR_ENV.replace(" ", "") != "66,67":
    raise RuntimeError("Frozen V11 readout/policy calibration is ONLY for physical pair (66,67).")
if IBM_BACKEND_NAME != "ibm_rensselaer":
    raise RuntimeError("Frozen V11 readout/DD calibration is specific to ibm_rensselaer.")
assert DELTAS == [0.01, 0.05] and PILOT_RAW_FAR_CUTOFF == 2
# The same unvalidated empirical gate is also applied at delta=0.05.
# This is an exploratory hardware extension; no per-delta calibration was carried out.
V1436_DELTA005_EMPIRICAL_GATE_EXTENSION = True
assert V1436_DELTA005_EMPIRICAL_GATE_EXTENSION

assert len(HARDWARE_EVAL_SEEDS)>=1 and all(0<=int(k)<=10 for k in HARDWARE_KR_VALUES)
assert all(int(q)>0 for q in HARDWARE_QPA_GRID)
assert DSWAP_EARLY_BLOCK_SHOTS == 64 and DSWAP_EARLY_MAX_BLOCKS == 4
assert DSWAP_PREFETCH_INITIAL == DSWAP_PREFETCH_MAX == 1
assert FIXED_KD == 5 and DSWAP_BUDGET_FRACTION == 0.25

np.set_printoptions(suppress=True, precision=6)
print("Logical revision:", REVISION)
print("Execution protocol:", EXECUTION_PROTOCOL)
print("Existing/new checkpoint detected:", PREEXISTING_CHECKPOINT)
print("Output:", OUTDIR.resolve())
print("n:",N,"| family:",FAMILY,"| deltas:",DELTAS,"| hardware seeds:",len(HARDWARE_EVAL_SEEDS))
print("Hardware seed groups:", HARDWARE_SEED_GROUPS)
print("Predefined matched no-dSWAP controls:",
      {"delta": V1437_TARGET_DELTA, "qpa": V1437_TARGET_QPA,
       "seeds": V1437_TARGET_SEEDS, "kR": V1437_TARGET_KR_VALUES},
      "| enabled:", V1437_ENABLE_TARGETED_NO_DSWAP)
print("Gap order inside each group:", DELTAS,
      "| WARNING: delta=.05 reuses UNVALIDATED raw >= 2 gate")
print("k_D:", FIXED_KD, "| scheduler:", SCHEDULER, "| dSWAP Q fraction:", DSWAP_BUDGET_FRACTION)
print("k_R sweep:", HARDWARE_KR_VALUES)
print("QPA plot grid:", HARDWARE_QPA_GRID)
print("QPA execution order:", HARDWARE_QPA_EXECUTION_ORDER)
print("Campaign checkpoint:", DB_PATH, "| existing checkpoint:", PREEXISTING_CHECKPOINT)
print("EARLY-STOP cutoff: raw '11' hits >=", PILOT_RAW_FAR_CUTOFF,
      "means FAR; 0 or 1 means CLOSE. This is a post-hoc experimental threshold.")
print("Adaptive block size:", DSWAP_EARLY_BLOCK_SHOTS, "| maximum blocks:", DSWAP_EARLY_MAX_BLOCKS)
print("RUN_FULL_CAMPAIGN:", RUN_FULL_CAMPAIGN,
      "| planned dSWAP configs:", len(DELTAS)*len(HARDWARE_EVAL_SEEDS)*len(HARDWARE_KR_VALUES)*len(HARDWARE_QPA_GRID),
      "| no-dSWAP: selected successful candidates only")
print("RUN_QPU:", RUN_QPU, "| RUN_HARDWARE_TRACE:",RUN_HARDWARE_TRACE,
      "| RUN_DD_CALIBRATION:",RUN_DD_CALIBRATION,"| RUN_END_TO_END_DD_ABLATION:",RUN_END_TO_END_DD_ABLATION)


## Theory formulas and bounded-depth MLAE convention

For arm mean `p`, the weak oracle prepares

\[
|\psi(p)\rangle=\sqrt{1-p}|0\rangle+\sqrt{p}|1\rangle,
\qquad \theta(p)=\arcsin\sqrt p.
\]

At the fixed destructive-SWAP depth `k_D=5 (empirical amplification)`,

\[
P_{11}(p_i,p_j)=\frac12\sin^2(\theta_i-\theta_j).
\]

Algorithm 3 uses

\[
M=\left\lceil\frac{200}{\Delta^2}\log\frac{2}{\delta}\right\rceil,
\qquad
T=\left\lceil 10\log\frac{2}{\delta}\right\rceil.
\]

For the **directional challenge**, depth `k` has

\[
D(k)=2k+1,\qquad
q_k(p)=\sin^2\!\left((2k+1)\arcsin\sqrt p\right).
\]

A configuration labelled `k_R=r` uses the joint MLAE schedule

\[
k=0,1,\ldots,r,
\]

so one shot at every depth costs

\[
\sum_{k=0}^{r}(2k+1)=(r+1)^2
\]

logical oracle calls **per arm**. This is the meaning of the `k_R=0..10` sweep throughout the notebook. `k_D` and `k_R` are never conflated.


In [ ]:
# ============================================================
# Bandit instances and theory helpers
# ============================================================
def theta_from_p(p: float) -> float:
    p = float(np.clip(p, 1e-12, 1 - 1e-12))
    return float(np.arcsin(np.sqrt(p)))


def oracle_uses_from_k(k: int) -> int:
    return 2 * int(k) + 1


def amplified_probability(p: float, k: int) -> float:
    th = theta_from_p(float(p))
    return float(np.sin(oracle_uses_from_k(k) * th) ** 2)


def mlae_schedule_cost(kR: int) -> int:
    # sum_{k=0}^{kR} (2k+1) = (kR+1)^2
    return int((int(kR) + 1) ** 2)


def dswap_p11(p_a: float, p_b: float, kD: int = FIXED_KD) -> float:
    da = oracle_uses_from_k(kD)*(theta_from_p(p_a)-theta_from_p(p_b))
    return float(0.5*np.sin(da)**2)


_KD_THRESHOLD_CACHE = {}
def kd_threshold_spec(delta: float, kD: int = FIXED_KD):
    """V14 calibrated boundary rule (empirical; not an Algorithm-3 theorem guarantee)."""
    delta=float(delta); kD=int(kD)
    if kD==0:
        return {"threshold":theory_dswap_parameters(delta)["threshold_rate"],
                "semantics":"algorithm3_theory_k0"}
    key=(round(delta,12),kD)
    if key not in _KD_THRESHOLD_CACHE:
        ps=np.linspace(0.1,0.9,801)
        close=[dswap_p11(p,p+delta/10,kD) for p in ps if p+delta/10 <=0.9]
        far=[dswap_p11(p,p+delta/2,kD) for p in ps if p+delta/2<=0.9]
        mx=float(np.max(close)); mn=float(np.min(far))
        _KD_THRESHOLD_CACHE[key]={"threshold":float((mx+mn)/2),
             "max_close_boundary_p11":mx,"min_far_boundary_p11":mn,
             "global_alias_safe":bool(mx<mn),
             "semantics":"amplified_empirical_boundary_not_theoretical_guarantee"}
    return dict(_KD_THRESHOLD_CACHE[key])


def theory_dswap_parameters(delta: float, failure_prob: float = DELTA_CONFIDENCE):
    delta = float(delta)
    failure_prob = float(failure_prob)
    M = int(math.ceil((200.0 / (delta ** 2)) * math.log(2.0 / failure_prob)))
    T = int(math.ceil(10.0 * math.log(2.0 / failure_prob)))
    return {
        "M": M,
        "T": T,
        "threshold_rate": float(T / M),
        "asymptotic_rate": float(delta ** 2 / 20.0),
        "far_lower_p11": float(delta ** 2 / 8.0),
        "close_upper_p11": float(delta ** 2 / 72.0),
    }


def nisq_dswap_shots(total_query_cap: int, n: int, fraction: float = DSWAP_BUDGET_FRACTION) -> int:
    """Finite-shot dSWAP repetitions per KING–CHALLENGER pair.

    The theorem uses Algorithm-3 M. The NISQ experiment instead reserves a fixed fraction
    of the matched total logical-query cap Q for dSWAP:
        2*(2*k_D+1)*(n-1)*m_D <= fraction*Q.
    """
    Q = int(total_query_cap)
    n = int(n)
    if n <= 1 or Q <= 0:
        return 0
    return max(0, int(math.floor(float(fraction)*Q/(2.0*oracle_uses_from_k(FIXED_KD)*(n-1)))))


def nisq_dswap_allocation(total_query_cap: int, n: int, fraction: float = DSWAP_BUDGET_FRACTION):
    m_d = nisq_dswap_shots(total_query_cap, n, fraction)
    planned = 2 * oracle_uses_from_k(FIXED_KD) * max(0, int(n) - 1) * int(m_d)
    Q = int(total_query_cap)
    return {
        "m_d": int(m_d),
        "planned_queries": int(planned),
        "target_fraction": float(fraction),
        "realized_fraction": float(planned / Q) if Q > 0 else 0.0,
        "cost_per_pair_shot": 2*oracle_uses_from_k(FIXED_KD),
        "supported": bool(int(m_d)>=MIN_DSWAP_SHOTS),
    }


def generate_instance(n: int, delta: float, seed: int, family: str = FAMILY):
    rng = np.random.default_rng(int(seed))
    if family != "clustered_lowvar":
        raise ValueError("This conference notebook intentionally fixes FAMILY='clustered_lowvar'.")
    sigma = min(float(delta) / 10.0, 0.005)
    background = 0.5 + rng.normal(0.0, sigma, size=n - 1)
    background = np.clip(background, 0.05, 0.95 - float(delta))
    best_p = float(np.max(background) + float(delta))
    probs = np.concatenate([background, [best_p]])
    probs = probs[rng.permutation(n)]
    best_arm = int(np.argmax(probs))
    second_p = float(np.partition(probs, -2)[-2])
    return probs, {
        "best_arm": best_arm,
        "best_p": float(probs[best_arm]),
        "second_p": second_p,
        "realized_gap": float(probs[best_arm] - second_p),
        "sigma": sigma,
    }

rows=[]
for d in DELTAS:
    theory=theory_dswap_parameters(d)
    for qpa in QUERIES_PER_ARM_GRID:
        Q=int(N*qpa)
        alloc=nisq_dswap_allocation(Q,N)
        rows.append({
            "delta": d,
            "qpa": int(qpa),
            "total_query_cap": Q,
            **theory,
            "theory_dswap_queries_for_n1000": 2 * (N - 1) * theory["M"],  # theorem is kD=0
            "nisq_dswap_budget_fraction_target": DSWAP_BUDGET_FRACTION,
            "nisq_m_d": alloc["m_d"],
            "nisq_dswap_queries_planned": alloc["planned_queries"],
            "nisq_dswap_fraction_realized": alloc["realized_fraction"],
            "m_d_over_theory_M": alloc["m_d"] / theory["M"],
        })
feasibility = pd.DataFrame(rows)
display(feasibility)
feasibility.to_csv(OUTDIR / "theory_dswap_feasibility.csv", index=False)

kr_cost = pd.DataFrame({
    "kR": KR_VALUES,
    "depths_used": [f"0..{k}" for k in KR_VALUES],
    "oracle_queries_per_full_mlae_shot_per_arm": [mlae_schedule_cost(k) for k in KR_VALUES],
    "max_oracle_multiplier": [oracle_uses_from_k(k) for k in KR_VALUES],
})
display(kr_cost)
kr_cost.to_csv(OUTDIR / "kr_query_costs.csv", index=False)


In [ ]:
# ============================================================
# Exact Algorithm-3 property-test sanity check (simulation only)
# ============================================================
def exact_dswap_property_test_sanity(delta, trials=2000, seed=1234):
    pars=theory_dswap_parameters(delta)
    rng=np.random.default_rng(seed+int(round(delta*1e6)))
    p0=0.5
    p_close=p0+delta/10.0
    p_far=p0+delta
    rows=[]
    for label,p1 in [("close_boundary",p_close),("far_best_gap",p_far)]:
        p11=dswap_p11(p0,p1,0)
        x=rng.binomial(pars["M"],p11,size=int(trials))
        far=(x>=pars["T"])
        rows.append({
            "delta":delta,"case":label,"M":pars["M"],"T":pars["T"],"ideal_p11":p11,
            "far_rate":float(np.mean(far)),"close_rate":float(np.mean(~far)),
        })
    return rows

sanity=pd.DataFrame(sum([exact_dswap_property_test_sanity(d) for d in DELTAS],[]))
display(sanity)
sanity.to_csv(OUTDIR/"exact_theory_dswap_sanity.csv",index=False)


### Empirical versus theoretical comparison

The finite-budget hardware dSWAP rule uses empirical shot limits. It does not directly establish the theorem's property-testing guarantee. Compare success, regret, and realized logical-query use together.


In [ ]:
# ============================================================
# Theory-control-flow simulator with bounded-depth MLAE challenges
# ============================================================
_MLAE_CACHE = {}

def _mlae_cache(kR, grid_size=MLAE_GRID_SIZE):
    key=(int(kR),int(grid_size))
    if key not in _MLAE_CACHE:
        grid=np.linspace(1e-6,1.0-1e-6,int(grid_size))
        theta=np.arcsin(np.sqrt(grid))
        qs=[]
        for k in range(int(kR)+1):
            q=np.sin((2*k+1)*theta)**2
            qs.append(np.clip(q,1e-12,1-1e-12))
        qtab=np.asarray(qs)
        _MLAE_CACHE[key]=(grid,np.log(qtab),np.log(1.0-qtab))
    return _MLAE_CACHE[key]


def mlae_decode_successes(successes, shots_by_k, kR, grid_size=MLAE_GRID_SIZE):
    """Joint maximum-likelihood decode over depths 0..kR."""
    successes=np.asarray(successes,float)
    shots=np.asarray(shots_by_k,float)
    grid,log_q,log_one_minus_q=_mlae_cache(int(kR),int(grid_size))
    use=np.where(shots>0)[0]
    if len(use)==0:
        return np.nan
    # Same joint likelihood, with trigonometric and logarithmic tables cached once.
    x=successes[use,None]
    s=shots[use,None]
    ll=np.sum(x*log_q[use]+(s-x)*log_one_minus_q[use],axis=0)
    return float(grid[int(np.argmax(ll))])


def challenge_level_schedule(budget_injection: int, kR: int, max_levels: int = MAX_CHALLENGE_LEVELS):
    """Algorithm-2 geometric levels, rounded to fund a complete 0..kR MLAE schedule.

    s_l is a PER-ARM logical-query budget. Rounding only changes finite-budget engineering
    constants; the KING/CHALLENGER and geometric-level semantics are unchanged.
    """
    b=max(1,int(budget_injection))
    c=max(1,b//5)
    schedule_cost=mlae_schedule_cost(kR)
    out=[]
    for level in range(1,int(max_levels)+1):
        raw=max(1,int(c*(3**level)))
        rounded=int(math.ceil(raw/schedule_cost)*schedule_cost)
        out.append(rounded)
    return out


def directional_challenge_mlae(probs, king, challenger, bank_B, level_query_budgets, kR, rng):
    """Directional KING/CHALLENGER comparison using bounded-depth MLAE at depths 0..kR.

    At level l, s_l is the per-arm query budget. If B<s_l, Algorithm-2 semantics promote
    the challenger without executing that level. Otherwise the level executes equal shots at
    every depth 0..kR, where one shot at depth k costs 2k+1 oracle calls.
    """
    king=int(king); challenger=int(challenger); B=int(bank_B); kR=int(kR)
    S=mlae_schedule_cost(kR)
    queries=0; levels_run=0; max_k_used=-1
    for s_l in level_query_budgets:
        s_l=int(s_l)
        if B < s_l:
            return challenger,0,queries,levels_run,max_k_used,"budget_exhausted_challenger"
        shots=max(1,s_l//S)
        actual_per_arm=shots*S
        xs_k=[]; xs_c=[]; ss=[]
        for k in range(kR+1):
            qk=amplified_probability(float(probs[king]),k)
            qc=amplified_probability(float(probs[challenger]),k)
            xs_k.append(int(rng.binomial(shots,qk)))
            xs_c.append(int(rng.binomial(shots,qc)))
            ss.append(shots)
        pk=mlae_decode_successes(xs_k,ss,kR)
        pc=mlae_decode_successes(xs_c,ss,kR)
        queries += 2*actual_per_arm
        B -= s_l
        levels_run += 1
        max_k_used=max(max_k_used,kR)
        if pk >= pc:
            return king,B,queries,levels_run,max_k_used,"king_wins"
    return challenger,B,queries,levels_run,max_k_used,"max_level_challenger"


def run_streaming_trial(probs, delta, total_query_cap, rng_seed, kR, use_dswap=True,
                        dswap_mode="nisq_fixed_budget"):
    probs=np.asarray(probs,float); n=len(probs); kR=int(kR)
    best_arm=int(np.argmax(probs)); best_p=float(probs[best_arm])
    order=np.arange(n,dtype=int)
    theory=theory_dswap_parameters(delta)

    if use_dswap:
        if dswap_mode=="theory_exact":
            m_d=int(theory["M"]); T=int(theory["T"])
        elif dswap_mode=="nisq_fixed_budget":
            alloc=nisq_dswap_allocation(total_query_cap,n)
            m_d=int(alloc["m_d"])
            T=max(1,int(math.ceil(m_d*kd_threshold_spec(delta,FIXED_KD)["threshold"]))) if m_d else 0
        else:
            raise ValueError(dswap_mode)
        dswap_planned_queries=2*oracle_uses_from_k(FIXED_KD)*(n-1)*m_d
    else:
        m_d=0; T=0; dswap_planned_queries=0

    Q=int(total_query_cap)
    if use_dswap and dswap_mode=="theory_exact" and FIXED_KD!=0:
        raise ValueError("Theorem's exact M/T only applies to kD=0.")
    if (use_dswap and m_d<MIN_DSWAP_SHOTS) or dswap_planned_queries>Q:
        return {"supported":0,"status":"dswap_underfunded","kR":kR,"selected_arm":int(order[0]),
                "success":0,"simple_regret":float(best_p-probs[int(order[0])]),
                "total_queries_requested":Q,"total_queries_used":0,"dswap_queries_used":0,
                "challenge_queries_used":0,"close_skips":0,"far_challenges":0,
                "challenge_fraction":np.nan,"effective_candidate_fraction":np.nan,"best_false_close":0,
                "dswap_shots_per_pair":m_d,"dswap_threshold_count":T,"dswap_budget_fraction_target":DSWAP_BUDGET_FRACTION,
                "dswap_budget_fraction_realized":float(dswap_planned_queries/Q) if Q>0 else 0.0,
                "max_k_used":-1}

    challenge_cap=Q-dswap_planned_queries
    b=max(1,challenge_cap//max(1,2*n))
    levels=challenge_level_schedule(b,kR)
    rng_d=np.random.default_rng(int(rng_seed)+100003)
    rng_c=np.random.default_rng(int(rng_seed)+200003+1009*kR)

    king=int(order[0]); B=0; dswap_used=0; challenge_used=0
    close_skips=0; far_challenges=0; best_false_close=0; max_B=0; max_k_used=-1
    budget_exhaustions=0

    for challenger in order[1:]:
        challenger=int(challenger); B+=b; max_B=max(max_B,B); is_far=True
        if use_dswap:
            p11=dswap_p11(float(probs[king]),float(probs[challenger]),FIXED_KD)
            x11=int(rng_d.binomial(m_d,p11)); dswap_used+=2*oracle_uses_from_k(FIXED_KD)*m_d
            is_far=bool(x11>=T)
            if not is_far:
                close_skips+=1
                if challenger==best_arm and king!=best_arm:
                    best_false_close+=1
        if is_far:
            far_challenges+=1
            new_king,B,q,_,mk,status=directional_challenge_mlae(
                probs,king,challenger,B,levels,kR,rng_c
            )
            king=int(new_king); challenge_used+=int(q); max_k_used=max(max_k_used,mk)
            if status=="budget_exhausted_challenger": budget_exhaustions+=1
        if dswap_used+challenge_used>Q:
            return {"supported":0,"status":"query_cap_exceeded","kR":kR,"selected_arm":king,
                    "success":int(king==best_arm),"simple_regret":float(best_p-probs[king]),
                    "total_queries_requested":Q,"total_queries_used":dswap_used+challenge_used,
                    "dswap_queries_used":dswap_used,"challenge_queries_used":challenge_used,
                    "close_skips":close_skips,"far_challenges":far_challenges,
                    "challenge_fraction":far_challenges/max(1,n-1),
                    "effective_candidate_fraction":(1+far_challenges)/n,
                    "best_false_close":best_false_close,"dswap_shots_per_pair":m_d,
                    "dswap_threshold_count":T,"dswap_budget_fraction_target":DSWAP_BUDGET_FRACTION,
                    "dswap_budget_fraction_realized":float(dswap_planned_queries/Q) if Q>0 else 0.0,
                    "max_k_used":max_k_used,"budget_exhaustions":budget_exhaustions}

    used=dswap_used+challenge_used
    return {"supported":1,"status":"ok","kR":kR,"selected_arm":int(king),
            "success":int(king==best_arm),"simple_regret":float(best_p-probs[int(king)]),
            "total_queries_requested":Q,"total_queries_used":int(used),
            "dswap_queries_used":int(dswap_used),"challenge_queries_used":int(challenge_used),
            "close_skips":int(close_skips),"far_challenges":int(far_challenges),
            "challenge_fraction":float(far_challenges/max(1,n-1)),
            "effective_candidate_fraction":float((1+far_challenges)/n),
            "best_false_close":int(best_false_close),"dswap_shots_per_pair":int(m_d),
            "dswap_threshold_count":int(T),"dswap_budget_fraction_target":float(DSWAP_BUDGET_FRACTION),
            "dswap_budget_fraction_realized":float(dswap_planned_queries/Q) if Q>0 else 0.0,
            "budget_injection_b":int(b),"max_budget_bank":int(max_B),
            "max_k_used":int(max_k_used),"budget_exhaustions":int(budget_exhaustions),
            "mlae_schedule_cost":int(mlae_schedule_cost(kR))}


def sequential_halving(probs, total_query_cap, rng):
    probs=np.asarray(probs,float); arms=np.arange(len(probs),dtype=int)
    rounds=max(1,int(math.ceil(math.log2(len(arms))))); remaining=int(total_query_cap); used=0
    for r in range(rounds):
        if len(arms)<=1: break
        cap=remaining//max(1,rounds-r); shots=cap//len(arms)
        if shots<1: break
        x=rng.binomial(shots,probs[arms]); rates=x/shots
        q=len(arms)*shots; used+=q; remaining-=q
        keep=max(1,int(math.ceil(len(arms)/2))); arms=arms[np.argsort(-rates)[:keep]]
    return (int(arms[0]) if len(arms) else None),int(used)


def uniform_sampling(probs,total_query_cap,rng):
    probs=np.asarray(probs,float); shots=int(total_query_cap)//len(probs)
    if shots<1: return None,0
    x=rng.binomial(shots,probs)
    return int(np.argmax(x/shots)),int(shots*len(probs))


In [ ]:
# ============================================================
# Simulator campaign: full k_R sweep, paired dSWAP ablation, classical baselines
# ============================================================
def run_simulator_campaign(seeds=SIM_SEEDS,deltas=DELTAS,qpa_grid=QUERIES_PER_ARM_GRID,kr_values=KR_VALUES):
    rows=[]
    for delta in deltas:
        for seed in seeds:
            probs,meta=generate_instance(N,delta,seed)
            for qpa in qpa_grid:
                Q=int(N*qpa)
                for kR in kr_values:
                    for method,use_d in [("theory_stream_dswap",True),("theory_stream_no_dswap",False)]:
                        res=run_streaming_trial(probs,delta,Q,
                            rng_seed=seed*1000000+qpa*31+kR*997,
                            kR=kR,use_dswap=use_d,dswap_mode="nisq_fixed_budget")
                        rows.append({"method":method,"delta":delta,"seed":seed,"qpa":qpa,"kR":int(kR),
                                     "best_arm":meta["best_arm"],"realized_gap":meta["realized_gap"],**res})

                # Classical baselines are independent of k_R and are stored once with kR=-1.
                rng=np.random.default_rng(seed*100000+qpa+31)
                pred,used=sequential_halving(probs,Q,rng)
                rows.append({"method":"sequential_halving","delta":delta,"seed":seed,"qpa":qpa,"kR":-1,
                             "best_arm":meta["best_arm"],"realized_gap":meta["realized_gap"],
                             "supported":int(pred is not None),"status":"ok" if pred is not None else "underfunded",
                             "selected_arm":pred,"success":int(pred==meta["best_arm"]) if pred is not None else 0,
                             "simple_regret":float(meta["best_p"]-probs[pred]) if pred is not None else np.nan,
                             "total_queries_requested":Q,"total_queries_used":used,"dswap_queries_used":0,
                             "challenge_queries_used":used,"close_skips":0,"far_challenges":N-1,
                             "challenge_fraction":1.0,"effective_candidate_fraction":1.0,"best_false_close":0,
                             "max_k_used":-1,"budget_exhaustions":0})
                rng=np.random.default_rng(seed*100000+qpa+73)
                pred,used=uniform_sampling(probs,Q,rng)
                rows.append({"method":"uniform_sampling","delta":delta,"seed":seed,"qpa":qpa,"kR":-1,
                             "best_arm":meta["best_arm"],"realized_gap":meta["realized_gap"],
                             "supported":int(pred is not None),"status":"ok" if pred is not None else "underfunded",
                             "selected_arm":pred,"success":int(pred==meta["best_arm"]) if pred is not None else 0,
                             "simple_regret":float(meta["best_p"]-probs[pred]) if pred is not None else np.nan,
                             "total_queries_requested":Q,"total_queries_used":used,"dswap_queries_used":0,
                             "challenge_queries_used":used,"close_skips":0,"far_challenges":N-1,
                             "challenge_fraction":1.0,"effective_candidate_fraction":1.0,"best_false_close":0,
                             "max_k_used":-1,"budget_exhaustions":0})
    raw=pd.DataFrame(rows)
    raw.to_csv(OUTDIR/"simulator_raw.csv",index=False)
    summary=(raw[raw.supported==1].groupby(["delta","qpa","method","kR"],as_index=False)
             .agg(trials=("success","size"),success_rate=("success","mean"),
                  mean_simple_regret=("simple_regret","mean"),median_simple_regret=("simple_regret","median"),
                  mean_queries_used=("total_queries_used","mean"),
                  mean_challenge_fraction=("challenge_fraction","mean"),
                  mean_effective_candidate_fraction=("effective_candidate_fraction","mean"),
                  best_false_close_rate=("best_false_close",lambda x:float(np.mean(np.asarray(x)>0))),
                  mean_max_k_used=("max_k_used","mean")))
    summary.to_csv(OUTDIR/"simulator_summary.csv",index=False)
    return raw,summary

if RUN_SIMULATOR_EXPERIMENTS:
    sim_raw,sim_summary=run_simulator_campaign()
else:
    raw_path=OUTDIR/"simulator_raw.csv"; sum_path=OUTDIR/"simulator_summary.csv"
    sim_raw=pd.read_csv(raw_path) if raw_path.exists() else pd.DataFrame()
    sim_summary=pd.read_csv(sum_path) if sum_path.exists() else pd.DataFrame()
    if len(sim_summary)==0:
        print("Simulator campaign not run. Set RUN_SIMULATOR_EXPERIMENTS=True for the full 50-seed k_R sweep.")
    else:
        display(sim_summary.head(30))


In [ ]:
# ============================================================
# Paired dSWAP ablation table and paper figures
# ============================================================
def paired_dswap_ablation(raw):
    if raw is None or len(raw)==0:
        return pd.DataFrame()
    q=raw[raw.method.isin(["theory_stream_dswap","theory_stream_no_dswap"]) & (raw.supported==1)].copy()
    p=q.pivot_table(index=["delta","seed","qpa","kR"],columns="method",
                    values=["success","simple_regret","total_queries_used","challenge_fraction","best_false_close"])
    p.columns=["__".join(c) for c in p.columns]; p=p.reset_index()
    p["regret_delta_dswap_minus_no"]=p["simple_regret__theory_stream_dswap"]-p["simple_regret__theory_stream_no_dswap"]
    p["query_delta_dswap_minus_no"]=p["total_queries_used__theory_stream_dswap"]-p["total_queries_used__theory_stream_no_dswap"]
    p["success_delta_dswap_minus_no"]=p["success__theory_stream_dswap"]-p["success__theory_stream_no_dswap"]
    p.to_csv(OUTDIR/"paired_dswap_ablation.csv",index=False)
    return p

paired=paired_dswap_ablation(sim_raw)
if len(paired):
    display(paired.groupby(["delta","qpa","kR"],as_index=False).agg(
        paired_trials=("seed","size"),mean_regret_delta=("regret_delta_dswap_minus_no","mean"),
        mean_query_delta=("query_delta_dswap_minus_no","mean"),mean_success_delta=("success_delta_dswap_minus_no","mean")))

if sim_summary is not None and len(sim_summary):
    for delta in DELTAS:
        s=sim_summary[(sim_summary.delta==delta) & (sim_summary.method=="theory_stream_dswap")]
        # Full k_R depth tradeoff: one line per selected display depth; all k_R remain in CSV.
        fig,ax=plt.subplots(figsize=(7.4,4.8))
        for kR in KR_PLOT_VALUES:
            g=s[s.kR==kR].sort_values("qpa")
            if len(g): ax.plot(g.qpa,g.success_rate,marker="o",label=f"dSWAP kR={kR}")
        sh=sim_summary[(sim_summary.delta==delta)&(sim_summary.method=="sequential_halving")].sort_values("qpa")
        if len(sh): ax.plot(sh.qpa,sh.success_rate,"--",marker="^",label="Sequential Halving")
        ax.set_xscale("log"); ax.set_ylim(-0.02,1.02); ax.set_xlabel("Queries per arm")
        ax.set_ylabel("Exact best-arm success rate"); ax.set_title(f"Bounded-MLAE depth sweep, Delta={delta}")
        ax.legend(ncol=2,fontsize=8); fig.tight_layout(); fig.savefig(FIGDIR/f"success_kr_sweep_delta_{delta}.png",dpi=300); plt.show()

        fig,ax=plt.subplots(figsize=(7.4,4.8))
        for kR in KR_PLOT_VALUES:
            g=s[s.kR==kR].sort_values("qpa")
            if len(g): ax.plot(g.qpa,g.mean_simple_regret,marker="o",label=f"dSWAP kR={kR}")
        if len(sh): ax.plot(sh.qpa,sh.mean_simple_regret,"--",marker="^",label="Sequential Halving")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Queries per arm")
        ax.set_ylabel("Mean simple regret"); ax.set_title(f"Regret vs bounded MLAE depth, Delta={delta}")
        ax.legend(ncol=2,fontsize=8); fig.tight_layout(); fig.savefig(FIGDIR/f"regret_kr_sweep_delta_{delta}.png",dpi=300); plt.show()


In [ ]:
# ============================================================
# Local no-QPU correctness smoke tests for k_R semantics
# ============================================================
def local_kr_smoke():
    checks=[]
    # Query-cost identity
    for kR in KR_VALUES:
        checks.append({"check":f"schedule_cost_kR{kR}","passed":mlae_schedule_cost(kR)==sum(2*k+1 for k in range(kR+1)),
                       "value":mlae_schedule_cost(kR)})
    # Noise-free MLAE decode should recover representative p values approximately.
    for kR in [0,1,3,10]:
        for p in [0.2,0.5,0.73]:
            shots=20000
            successes=[amplified_probability(p,k)*shots for k in range(kR+1)]
            est=mlae_decode_successes(successes,[shots]*(kR+1),kR)
            checks.append({"check":f"mlae_p{p}_kR{kR}","passed":abs(est-p)<0.002,"value":est})
    df=pd.DataFrame(checks); display(df.tail(20))
    if not bool(df.passed.all()): raise AssertionError("Local k_R/MLAE smoke test failed")
    return df

local_kr_checks=local_kr_smoke()


In [ ]:
# ============================================================
# Offline consistency checks: amplified dSWAP math and allocations
# ============================================================
def validate_v11_parameters():
    assert SCHEDULER=="alap" and FIXED_KD in (4,5) and DSWAP_BUDGET_FRACTION in (0.20,0.25)
    assert oracle_uses_from_k(FIXED_KD) == (2*FIXED_KD+1)
    assert abs(dswap_p11(0.5,0.5,FIXED_KD))<1e-12
    for delta in DELTAS:
        t=kd_threshold_spec(delta,FIXED_KD)
        assert 0 <= t["threshold"] <= 0.5
    for qpa in HARDWARE_QPA_GRID:
        Q=N*qpa; allocation=nisq_dswap_allocation(Q,N)
        assert allocation["planned_queries"]<=int(DSWAP_BUDGET_FRACTION*Q)
        assert allocation["planned_queries"]==2*(2*FIXED_KD+1)*(N-1)*allocation["m_d"]
        if qpa<50: assert allocation["m_d"]==0
    print("Offline parameter accounting checks passed; no QPU work performed.")
validate_v11_parameters()


In [ ]:
# ============================================================
# Historical offline feasibility using completed 227-shot raw hit totals
# Optional and read-only; uses the specifically named historical audit CSV or ZIP.
# ============================================================
import zipfile
from scipy.stats import hypergeom

V12_RESULT_STEM="results_theory_faithful_system1_v12_kd5_f25_t2_seed9_kr3_qpa20000"
V12_AUDIT_NAME="hardware_pilot_dswap_decisions_delta0.01_seed9_qpa20000_kr3.csv"
V12_AUDIT_SOURCE=os.getenv("NIGHTHAWK_V12_AUDIT", "").strip()

def find_v12_audit():
    choices=[Path(V12_AUDIT_SOURCE)] if V12_AUDIT_SOURCE else []
    choices.extend([Path.cwd()/V12_RESULT_STEM/V12_AUDIT_NAME,
                    Path.cwd()/(V12_RESULT_STEM+'.zip'),
                    Path.cwd()/(V12_RESULT_STEM+'(1).zip')])
    for path in choices:
        if not path.exists() or not path.is_file():
            continue
        if path.suffix.lower()=='.csv':
            return pd.read_csv(path),str(path)
        if path.suffix.lower()=='.zip':
            with zipfile.ZipFile(path) as z:
                matches=[n for n in z.namelist() if n.endswith('/'+V12_AUDIT_NAME)
                         and not n.startswith('__MACOSX/')]
                if matches:
                    import io
                    return pd.read_csv(io.BytesIO(z.read(matches[0]))),str(path)
    return None,None

def expected_shots_by_conditional_shuffle(full_shot_counts,block_size=64,cap=227,threshold=2):
    """E[physical shots] under exchangeability given a fixed total of raw-11 hits.

    This is OFFLINE modeling, not a recoverable measurement order or end-to-end
    pilot claim. With raw H hits among cap shots, the prefix count is
    Hypergeometric(N=cap, K=H, n=prefix). A following block is required iff
    the previous cumulative prefix has fewer than threshold hits.
    """
    all_sizes=[]; remaining=cap
    while remaining:
        x=min(int(block_size),remaining)
        all_sizes.append(x);remaining-=x
    totals=np.asarray(full_shot_counts,dtype=int)
    if np.any((totals<0)|(totals>cap)):
        raise ValueError("Expected raw hit totals between 0 and cap")
    expected=np.full(len(totals),float(all_sizes[0]))
    prefix=all_sizes[0]
    for size in all_sizes[1:]:
        expected+=size*hypergeom.cdf(threshold-1,cap,totals,prefix)
        prefix+=size
    return expected

_v12_audit,_v12_source=find_v12_audit()
if _v12_audit is not None:
    assert len(_v12_audit)==999 and set(_v12_audit['shots'])=={227}
    reports=[]
    for block_size in [32,48,64,80,96,112]:
        x=expected_shots_by_conditional_shuffle(
            _v12_audit['raw_11_hits'],block_size=block_size)
        cost=float(x.sum()*2*oracle_uses_from_k(FIXED_KD))
        reports.append({"shot_block_size":block_size,"expected_avg_shots":float(x.mean()),
                        "modeled_dswap_queries":int(round(cost)),
                        "modeled_total_queries_if_V12_MLAE_unchanged":int(round(cost))+10194848})
    offline=pd.DataFrame(reports)
    offline.to_csv(OUTDIR/'v13_offline_early_stop_feasibility.csv',index=False)
    print("V12 trace source:",_v12_source)
    display(offline)
    print("Note: This conditions on the V12 histograms and assumes exchangeable shot order;")
    print("new hardware sampling, KING trajectories, and IBM wall time may differ.")
else:
    print('V12 audit not found locally: offline trace-dependent feasibility skipped.')
    print('Optional: set NIGHTHAWK_V12_AUDIT to a V12 results ZIP or audit CSV before Run All.')

## IBM Quantum connection

For authorized hardware use, supply `IBM_QUANTUM_TOKEN` and `IBM_INSTANCE_CRN` privately via environment variables. Do not commit credentials or local checkpoint databases.


In [ ]:
# IBM credentials are provided at runtime via environment variables or getpass.
# Do not commit or share live IBM API tokens.


In [ ]:
# ============================================================
# IBM Runtime connection
# ============================================================
def get_ibm_service():
    from qiskit_ibm_runtime import QiskitRuntimeService
    token = os.getenv("IBM_QUANTUM_TOKEN", "").strip()
    instance = os.getenv("IBM_INSTANCE_CRN", "").strip()
    if not token:
        token = getpass("IBM Quantum API token: ").strip()
    if not instance:
        instance = getpass("IBM Quantum NEW instance CRN: ").strip()
    if not token or not instance:
        raise RuntimeError("IBM credentials are required for RUN_QPU=True")
    # Permit the previous-instance recovery service to reuse the same-account token.
    os.environ["IBM_QUANTUM_TOKEN"] = token
    return QiskitRuntimeService(channel="ibm_quantum_platform",token=token,instance=instance)


class RollingBatchSampler:
    """Thread-safe IBM Runtime Batch proxy; jobs may remain in flight concurrently.

    Only submission/rotation is serialized. Each independent worker waits for its own
    job result outside the lock. Batch.close() is called only after all workers finish,
    or when rotating before the configured TTL.
    """
    def __init__(self,backend,max_time=BATCH_MAX_TIME,rotate_seconds=BATCH_ROTATE_SECONDS):
        self.backend=backend; self.max_time=max_time; self.rotate_seconds=int(rotate_seconds)
        self._lock=threading.RLock(); self._batch=None; self._sampler=None; self._opened_at=0
        with self._lock: self._rotate_locked()

    def _rotate_locked(self):
        if self._batch is not None:
            try: self._batch.close()
            except Exception: pass
        from qiskit_ibm_runtime import Batch, SamplerV2
        try: self._batch=Batch(backend=self.backend,max_time=self.max_time)
        except TypeError: self._batch=Batch(backend=self.backend)
        self._sampler=SamplerV2(mode=self._batch)
        self._opened_at=time.time()
        try: self._sampler.options.environment.job_tags=["nighthawk","v14","alap-kd5-f25","early-far-stop"]
        except Exception: pass
        print("Opened rolling IBM Runtime Batch")

    def run(self,*args,**kwargs):
        with self._lock:
            if time.time()-self._opened_at>=self.rotate_seconds:
                self._rotate_locked()
            try:
                return self._sampler.run(*args,**kwargs)
            except Exception as exc:
                msg=str(exc).lower()
                if any(x in msg for x in ("expired","batch closed","not accepting","session closed","ttl")):
                    self._rotate_locked()
                    return self._sampler.run(*args,**kwargs)
                raise

    def close(self):
        with self._lock:
            if self._batch is not None:
                try: self._batch.close()
                finally: self._batch=None; self._sampler=None


def get_sampler(backend):
    # A real Batch rather than Sampler(mode=backend), which creates independent jobs.
    return RollingBatchSampler(backend)


def extract_counts(pub_result):
    data=pub_result.data
    for name in ["meas","c"]:
        if hasattr(data,name):
            obj=getattr(data,name)
            if hasattr(obj,"get_counts"):
                return {str(k).replace(" ",""): int(v) for k,v in obj.get_counts().items()}
    for name in dir(data):
        if name.startswith("_"): continue
        obj=getattr(data,name)
        if hasattr(obj,"get_counts"):
            return {str(k).replace(" ",""): int(v) for k,v in obj.get_counts().items()}
    raise RuntimeError("Could not locate counts in Sampler result")


def safe_job_id(job):
    try: return str(job.job_id())
    except Exception:
        try: return str(job.job_id)
        except Exception: return "<unknown>"

In [ ]:
# ============================================================
# Qiskit circuit builders: k_D=0 dSWAP + bounded-depth MLAE amplitude circuits
# ============================================================
def build_amplitude_circuit(p: float, k: int = 0):
    """One-arm bounded-depth amplitude circuit with amplified probability q_k(p)."""
    from qiskit import QuantumCircuit
    k=int(k); theta=theta_from_p(float(p))
    qc=QuantumCircuit(1,1,name=f"amp_k{k}")
    qc.ry(2.0*theta,0)
    qc.barrier()
    # Linear bounded-depth schedule: k Grover-style amplification blocks -> D(k)=2k+1 oracle uses.
    for _ in range(k):
        qc.z(0); qc.ry(-2.0*theta,0); qc.x(0); qc.z(0); qc.x(0); qc.ry(2.0*theta,0)
        qc.barrier()
    qc.measure(0,0)
    return qc


def build_dswap_circuit(p_a: float, p_b: float, kD: int = FIXED_KD):
    """kD=0 reproduces Algorithm-3 circuit; higher kD uses V14 amplified oracle blocks."""
    from qiskit import QuantumCircuit
    kD=int(kD)
    qc=QuantumCircuit(2,2,name=f"dswap_kD{kD}")
    for q,p in ((0,p_a),(1,p_b)):
        th=theta_from_p(float(p))
        qc.ry(2.0*th,q); qc.barrier(q)
        for _ in range(kD):
            qc.z(q); qc.ry(-2.0*th,q); qc.x(q); qc.z(q); qc.x(q); qc.ry(2.0*th,q)
            qc.barrier(q)
    qc.barrier(); qc.cx(0,1); qc.h(0); qc.barrier(); qc.measure([0,1],[0,1])
    return qc



In [ ]:
# ============================================================
# Deterministic physical-pair selection from backend calibration
# ============================================================
def _safe_qubit_props(backend, q):
    try:
        p=backend.qubit_properties(int(q))
        return float(getattr(p,"t1",np.nan)), float(getattr(p,"t2",np.nan))
    except Exception:
        return np.nan, np.nan


def _readout_error(backend,q):
    try:
        props=backend.properties()
        return float(props.readout_error(int(q)))
    except Exception:
        try:
            x=backend.target["measure"][(int(q),)]
            return float(getattr(x,"error",np.nan))
        except Exception:
            return np.nan


def _twoq_error(backend,a,b):
    for name in ["cz","ecr","cx"]:
        try:
            prop=backend.target[name][(int(a),int(b))]
            err=getattr(prop,"error",None)
            if err is not None: return float(err), name
        except Exception: pass
        try:
            prop=backend.target[name][(int(b),int(a))]
            err=getattr(prop,"error",None)
            if err is not None: return float(err), name
        except Exception: pass
    return np.nan, "unknown"


def choose_physical_pair(backend):
    if LOCKED_PAIR_ENV:
        a,b=[int(x.strip()) for x in LOCKED_PAIR_ENV.split(",")]
        return (a,b), pd.DataFrame()
    edges=sorted({tuple(sorted(map(int,e))) for e in backend.coupling_map.get_edges()})
    rows=[]
    for a,b in edges:
        ro=_readout_error(backend,a)+_readout_error(backend,b)
        e2,g2=_twoq_error(backend,a,b)
        t1a,t2a=_safe_qubit_props(backend,a); t1b,t2b=_safe_qubit_props(backend,b)
        tmin=np.nanmin([t1a,t2a,t1b,t2b])
        rows.append({"q0":a,"q1":b,"readout_sum":ro,"twoq_error":e2,"twoq_gate":g2,"min_t1_t2_s":tmin})
    df=pd.DataFrame(rows)
    for col in ["readout_sum","twoq_error"]:
        med=float(np.nanmedian(df[col])) if np.isfinite(df[col]).any() else 1.0
        df[col]=df[col].fillna(med)
    coh=1.0/np.maximum(df["min_t1_t2_s"].fillna(df["min_t1_t2_s"].median()),1e-12)
    def norm(x):
        x=np.asarray(x,float); lo=np.nanmin(x); hi=np.nanmax(x)
        return np.zeros_like(x) if hi<=lo else (x-lo)/(hi-lo)
    df["score"]=norm(df.readout_sum)+norm(df.twoq_error)+norm(coh)
    df=df.sort_values(["score","q0","q1"]).reset_index(drop=True)
    return (int(df.iloc[0].q0),int(df.iloc[0].q1)), df


In [ ]:
# ============================================================
# ALAP scheduling, DD eligibility, and physical-resource audit
# ============================================================
class DDInsertionError(RuntimeError):
    pass


def _dd_sequence(name):
    from qiskit.circuit.library import XGate, YGate
    if name=="xx": return [XGate(),XGate()]
    if name=="xy4": return [XGate(),YGate(),XGate(),YGate()]
    if name=="no_dd": return []
    raise ValueError(name)



def _target_with_y(backend):
    import copy
    from qiskit.circuit.library import YGate
    try:
        from qiskit.transpiler import InstructionProperties
    except Exception:
        InstructionProperties=None
    target=copy.deepcopy(backend.target)
    basis=list(getattr(target,"operation_names",[]))
    if "y" in basis or InstructionProperties is None:
        return target,basis
    props={}
    for q in range(int(target.num_qubits)):
        try:
            xprop=target["x"][(q,)]
            props[(q,)]=InstructionProperties(duration=getattr(xprop,"duration",None),error=getattr(xprop,"error",None))
        except Exception:
            pass
    if props:
        target.add_instruction(YGate(),props)
    return target,basis


def _layout(logical,pair):
    return [int(pair[0])] if logical.num_qubits==1 else [int(pair[0]),int(pair[1])]


def _duration_seconds(circuit, backend):
    d=getattr(circuit,"duration",None); unit=getattr(circuit,"unit","dt")
    if d is None: return np.nan
    scale={"s":1.0,"ms":1e-3,"us":1e-6,"µs":1e-6,"ns":1e-9}
    if unit in scale: return float(d)*scale[unit]
    dt=getattr(backend,"dt",None)
    return float(d)*float(dt) if dt is not None else np.nan


def _delay_seconds(op,backend):
    d=getattr(op,"duration",None); unit=getattr(op,"unit","dt")
    if d is None: return np.nan
    scale={"s":1.0,"ms":1e-3,"us":1e-6,"µs":1e-6,"ns":1e-9}
    if unit in scale: return float(d)*scale[unit]
    dt=getattr(backend,"dt",None)
    return float(d)*float(dt) if dt is not None else np.nan


def _replace_y(circuit):
    if not any(getattr(x.operation,"name","")=="y" for x in circuit.data): return circuit
    new=circuit.copy_empty_like()
    for item in circuit.data:
        if getattr(item.operation,"name","")=="y":
            new.rz(math.pi,item.qubits[0]); new.x(item.qubits[0])
        else:
            new.append(item.operation,item.qubits,item.clbits)
    return new


def compile_alap_variant(logical, backend, pair, treatment="no_dd"):
    from qiskit import transpile
    from qiskit.transpiler import PassManager
    try:
        from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling, PadDelay
    except Exception:
        from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling, PadDelay

    isa=transpile(logical,backend=backend,optimization_level=1,initial_layout=_layout(logical,pair))
    before=Counter(isa.count_ops())
    target=backend.target
    try:
        scheduler=ALAPScheduleAnalysis(target=target)
    except Exception:
        scheduler=ALAPScheduleAnalysis(target.durations())

    if treatment=="no_dd":
        try: raw=PassManager([scheduler,PadDelay(target=target)]).run(isa)
        except Exception: raw=PassManager([scheduler,PadDelay(target.durations())]).run(isa)
        verified=True
    else:
        seq=_dd_sequence(treatment)
        if treatment=="xy4":
            target,_basis=_target_with_y(backend)
            try: scheduler=ALAPScheduleAnalysis(target=target)
            except Exception: scheduler=ALAPScheduleAnalysis(target.durations())
        try: raw=PassManager([scheduler,PadDynamicalDecoupling(target=target,dd_sequence=seq)]).run(isa)
        except Exception: raw=PassManager([scheduler,PadDynamicalDecoupling(target.durations(),seq)]).run(isa)
        after=Counter(raw.count_ops())
        ins_x=max(0,int(after.get("x",0)-before.get("x",0)))
        ins_y=max(0,int(after.get("y",0)-before.get("y",0)))
        if treatment=="xx": verified=(ins_x>=2 and ins_x%2==0)
        else: verified=(ins_x>=2 and ins_y>=2 and ins_x==ins_y)
        if not verified: raise DDInsertionError(f"{treatment}: complete sequence not inserted")
        if treatment=="xy4": raw=_replace_y(raw)

    after=Counter(raw.count_ops())
    inserted_x=max(0,int(after.get("x",0)-before.get("x",0)))
    inserted_y=max(0,int(after.get("y",0)-before.get("y",0)))
    delays=[_delay_seconds(item.operation,backend) for item in raw.data if getattr(item.operation,"name","")=="delay"]
    delays=[x for x in delays if np.isfinite(x)]
    dur=_duration_seconds(raw,backend)
    q_used=list(pair[:logical.num_qubits])
    tvals=[]
    for q in q_used:
        t1,t2=_safe_qubit_props(backend,q); tvals.extend([t1,t2])
    finite=[x for x in tvals if np.isfinite(x) and x>0]
    min_t=min(finite) if finite else np.nan
    t2s=[_safe_qubit_props(backend,q)[1] for q in q_used]
    t2s=[x for x in t2s if np.isfinite(x) and x>0]
    min_t2=min(t2s) if t2s else np.nan

    oneq=twoq=meas=barriers=0
    for item in raw.data:
        name=getattr(item.operation,"name","")
        nq=len(item.qubits)
        if name=="measure": meas+=1
        elif name=="barrier": barriers+=1
        elif name in {"delay","reset"}: pass
        elif nq==1: oneq+=1
        elif nq==2: twoq+=1

    stats={
        "scheduler":"alap","treatment":treatment,"verified_dd":bool(verified),
        "logical_qubits":int(logical.num_qubits),"physical_qubits":"-".join(map(str,q_used)),
        "depth":int(raw.depth()),"size":int(raw.size()),"oneq_gates":int(oneq),"twoq_gates":int(twoq),
        "measurements":int(meas),"barriers":int(barriers),"inserted_x":inserted_x,"inserted_y":inserted_y,
        "max_idle_us":float(max(delays,default=0.0)*1e6),"total_idle_us":float(sum(delays)*1e6),
        "scheduled_duration_us":float(dur*1e6) if np.isfinite(dur) else np.nan,
        "min_t1_t2_us":float(min_t*1e6) if np.isfinite(min_t) else np.nan,
        "min_t2_us":float(min_t2*1e6) if np.isfinite(min_t2) else np.nan,
        "rho_t2":float(dur/min_t2) if np.isfinite(dur) and np.isfinite(min_t2) and min_t2>0 else np.nan,
        "rho_t1_t2":float(dur/min_t) if np.isfinite(dur) and np.isfinite(min_t) and min_t>0 else np.nan,
        "retention_proxy_t2":float(np.exp(-dur/min_t2)) if np.isfinite(dur) and np.isfinite(min_t2) and min_t2>0 else np.nan,
        "ops_json":json.dumps(dict(after),sort_keys=True),
    }
    return raw,stats


def eligible_variants(logical, backend, pair):
    """Always include no-DD. Include XX/XY4 only if a complete DD sequence is physically inserted."""
    out=[]
    exe,st=compile_alap_variant(logical,backend,pair,"no_dd"); out.append(("no_dd",exe,st))
    for tr in ["xx","xy4"]:
        try:
            exe,st=compile_alap_variant(logical,backend,pair,tr)
            out.append((tr,exe,st))
        except Exception:
            pass
    return out


In [ ]:
# ============================================================
# Resume-safe storage for QPU jobs, counts, physical manifests, and end-to-end results
# ============================================================
_DB_SETUP_LOCK=threading.RLock()
_DB_WAL_READY=False
# V9: sqlite connection/context reuse, one connection per thread, never share across threads.
# sqlite3.Connection's context manager commits on exit but does NOT close connections;
# V8 previously created a new connection for each short read/write.
_DB_THREAD_LOCAL=threading.local()
def db():
    global _DB_WAL_READY
    con=getattr(_DB_THREAD_LOCAL,"connection",None)
    if con is not None:
        return con
    with _DB_SETUP_LOCK:
        con=sqlite3.connect(DB_PATH,timeout=120)
        con.execute("PRAGMA busy_timeout=120000")
        if not _DB_WAL_READY:
            con.execute("PRAGMA journal_mode=WAL")
            _DB_WAL_READY=True
        con.execute("PRAGMA synchronous=FULL")
        _DB_THREAD_LOCAL.connection=con
    return con


def _table_columns(con, table):
    return {r[1] for r in con.execute(f"PRAGMA table_info({table})").fetchall()}


def _ensure_column(con, table, name, decl):
    if name not in _table_columns(con,table):
        con.execute(f"ALTER TABLE {table} ADD COLUMN {name} {decl}")


def init_db():
    with db() as con:
        con.executescript("""
        CREATE TABLE IF NOT EXISTS qpu_counts(
          key TEXT PRIMARY KEY, role TEXT, seed INTEGER, arm_a INTEGER, arm_b INTEGER,
          delta REAL, treatment TEXT, shots INTEGER, counts_json TEXT, job_id TEXT,
          submit_wall_sec REAL, created_at REAL, circuit_sig TEXT);

        CREATE TABLE IF NOT EXISTS physical_manifest(
          key TEXT PRIMARY KEY, role TEXT, seed INTEGER, arm_a INTEGER, arm_b INTEGER,
          delta REAL, treatment TEXT, shots INTEGER, job_id TEXT,
          logical_oracle_queries INTEGER, circuit_executions INTEGER,
          estimated_scheduled_qpu_time_sec REAL, submit_wall_sec REAL,
          stats_json TEXT, created_at REAL, circuit_sig TEXT);

        CREATE TABLE IF NOT EXISTS dd_policy(
          policy_key TEXT PRIMARY KEY, chosen_treatment TEXT, calibration_json TEXT, created_at REAL);

        CREATE TABLE IF NOT EXISTS hardware_results(
          key TEXT PRIMARY KEY, method TEXT, delta REAL, seed INTEGER, qpa INTEGER, kR INTEGER,
          dd_mode TEXT, selected_arm INTEGER, best_arm INTEGER, success INTEGER, simple_regret REAL,
          logical_queries_used INTEGER, dswap_queries_used INTEGER, challenge_queries_used INTEGER,
          close_skips INTEGER, far_challenges INTEGER, best_false_close INTEGER,
          challenge_fraction REAL, effective_candidate_fraction REAL, max_k_used INTEGER,
          budget_exhaustions INTEGER, status TEXT, created_at REAL,
          dswap_shots_per_pair INTEGER,
          dswap_budget_fraction_target REAL,
          dswap_budget_fraction_realized REAL,
          prefetch_not_consumed_circuits INTEGER DEFAULT 0,
          prefetch_not_consumed_shots INTEGER DEFAULT 0,
          prefetch_not_consumed_oracle_queries INTEGER DEFAULT 0,
          execution_protocol TEXT);

        CREATE TABLE IF NOT EXISTS qpu_jobs(
          job_id TEXT PRIMARY KEY, batch_key TEXT, role TEXT, seed INTEGER, delta REAL, shots INTEGER,
          state TEXT, item_count INTEGER, submitted_at REAL, completed_at REAL,
          last_error TEXT, updated_at REAL, execution_protocol TEXT);

        CREATE TABLE IF NOT EXISTS qpu_job_items(
          item_key TEXT PRIMARY KEY, job_id TEXT, batch_index INTEGER,
          arm_a INTEGER, arm_b INTEGER, treatment TEXT, stats_json TEXT,
          created_at REAL);

        CREATE INDEX IF NOT EXISTS idx_qpu_job_items_job_id ON qpu_job_items(job_id);

        CREATE TABLE IF NOT EXISTS run_meta(
          meta_key TEXT PRIMARY KEY, value_json TEXT, created_at REAL, updated_at REAL);

        CREATE TABLE IF NOT EXISTS replay_samples(
          sample_key TEXT PRIMARY KEY, config_key TEXT, circuit_sig TEXT, draw_index INTEGER,
          shots INTEGER, counts_json TEXT, source_total_shots INTEGER, created_at REAL);
        CREATE INDEX IF NOT EXISTS idx_replay_config_sig
          ON replay_samples(config_key,circuit_sig,draw_index);
        CREATE INDEX IF NOT EXISTS idx_qpu_counts_circuit_sig
          ON qpu_counts(circuit_sig);
        CREATE TABLE IF NOT EXISTS adaptive_dswap_decisions(
          config_key TEXT, challenger INTEGER, king_before INTEGER,
          raw_11_hits INTEGER, shots_used INTEGER, shots_cap INTEGER,
          is_far INTEGER, best_false_close INTEGER,
          block_sizes_json TEXT, job_ids_json TEXT,
          elapsed_wall_sec REAL, circuit_sig TEXT, created_at REAL,
          PRIMARY KEY(config_key,challenger));
        """)
        # Expression index on the job item's saved signature is available on
        # modern SQLite builds with JSON1. Maintain compatibility with older ones.
        try:
            con.execute("""CREATE INDEX IF NOT EXISTS idx_qpu_job_items_circuit_sig
                ON qpu_job_items(json_extract(stats_json, '$.circuit_sig'))""")
        except sqlite3.OperationalError:
            print("SQLite JSON expression index unavailable; retaining slower compatibility scan.")

        # Migration safety if this notebook is pointed at an older checkpoint DB.
        _ensure_column(con,"hardware_results","dswap_shots_per_pair","INTEGER")
        _ensure_column(con,"hardware_results","dswap_budget_fraction_target","REAL")
        _ensure_column(con,"hardware_results","dswap_budget_fraction_realized","REAL")
        _ensure_column(con,"hardware_results","prefetch_not_consumed_circuits","INTEGER DEFAULT 0")
        _ensure_column(con,"hardware_results","prefetch_not_consumed_shots","INTEGER DEFAULT 0")
        _ensure_column(con,"hardware_results","prefetch_not_consumed_oracle_queries","INTEGER DEFAULT 0")
        _ensure_column(con,"hardware_results","execution_protocol","TEXT")
        _ensure_column(con,"qpu_counts","circuit_sig","TEXT")
        _ensure_column(con,"physical_manifest","circuit_sig","TEXT")
        _ensure_column(con,"qpu_jobs","execution_protocol","TEXT")


def get_run_meta(key, default=None):
    with db() as con:
        row=con.execute("SELECT value_json FROM run_meta WHERE meta_key=?",(str(key),)).fetchone()
    if row is None:
        return default
    try:
        return json.loads(row[0])
    except Exception:
        return row[0]


def set_run_meta(key, value):
    now=time.time()
    payload=json.dumps(value,default=str)
    with db() as con:
        con.execute("""
        INSERT INTO run_meta(meta_key,value_json,created_at,updated_at)
        VALUES(?,?,?,?)
        ON CONFLICT(meta_key) DO UPDATE SET value_json=excluded.value_json,updated_at=excluded.updated_at
        """,(str(key),payload,now,now))


def _backend_name(backend):
    name=getattr(backend,"name","unknown")
    try:
        name=name() if callable(name) else name
    except Exception:
        pass
    return str(name)


def load_or_freeze_physical_pair(backend):
    """Freeze backend + physical pair in the checkpoint DB on the first live run.

    A restart reuses the same pair even if a later IBM calibration snapshot would rank another pair first.
    """
    bname=_backend_name(backend)
    stored_backend=get_run_meta("backend_name")
    if stored_backend is not None and str(stored_backend)!=bname:
        raise RuntimeError(
            f"Checkpoint DB is frozen to backend {stored_backend!r}, but current backend is {bname!r}. "
            "Use the same backend or a new output directory."
        )

    stored_pair=get_run_meta("physical_pair")
    explicit=None
    if LOCKED_PAIR_ENV:
        explicit=tuple(int(x.strip()) for x in LOCKED_PAIR_ENV.split(","))

    if stored_pair is not None:
        pair=tuple(int(x) for x in stored_pair)
        if explicit is not None and tuple(explicit)!=pair:
            raise RuntimeError(
                f"Checkpoint DB is frozen to physical pair {pair}, but IBM_LOCKED_PAIR={explicit}. "
                "Do not mix pairs inside one checkpoint DB."
            )
        return pair,pd.DataFrame()

    pair,ranking=choose_physical_pair(backend) if explicit is None else (explicit,pd.DataFrame())
    set_run_meta("backend_name",bname)
    set_run_meta("physical_pair",[int(pair[0]),int(pair[1])])
    set_run_meta("revision",REVISION)
    return tuple(pair),ranking


def hardware_result_key(method,delta,seed,qpa,kR,dd_mode):
    return hashlib.sha256(
        repr((REVISION,str(method),float(delta),int(seed),int(qpa),int(kR),str(dd_mode))).encode()
    ).hexdigest()[:28]


def load_hardware_result(key):
    with db() as con:
        cur=con.execute("SELECT * FROM hardware_results WHERE key=?",(str(key),))
        row=cur.fetchone()
        cols=[x[0] for x in cur.description] if cur.description else []
    return None if row is None else dict(zip(cols,row))


def export_checkpoint_ledger():
    """Refresh human-readable checkpoint CSVs without waiting for the campaign to finish."""
    with db() as con:
        jobs=pd.read_sql_query("SELECT * FROM qpu_jobs ORDER BY submitted_at",con)
        counts=pd.read_sql_query("""
            SELECT role,seed,delta,treatment,shots,job_id,COUNT(*) AS circuit_records,
                   SUM(CASE WHEN counts_json IS NOT NULL THEN 1 ELSE 0 END) AS completed_records,
                   MIN(created_at) AS first_created_at
            FROM qpu_counts
            GROUP BY role,seed,delta,treatment,shots,job_id
            ORDER BY first_created_at
        """,con)
        hw=pd.read_sql_query("SELECT * FROM hardware_results ORDER BY created_at",con)
        replay=pd.read_sql_query("""
            SELECT config_key,circuit_sig,COUNT(*) AS draws,SUM(shots) AS logical_replay_shots,
                   MAX(source_total_shots) AS max_source_reservoir_shots
            FROM replay_samples GROUP BY config_key,circuit_sig
        """,con)
    jobs.to_csv(OUTDIR/"qpu_job_ledger.csv",index=False)
    counts.to_csv(OUTDIR/"qpu_counts_progress.csv",index=False)
    hw.to_csv(OUTDIR/"hardware_algorithm_results.csv",index=False)
    replay.to_csv(OUTDIR/"hardware_replay_progress.csv",index=False)
    return {"jobs":jobs,"counts_progress":counts,"hardware_results":hw,"replay_progress":replay}



def checkpoint_resume_summary():
    """Human-readable checkpoint and optional legacy-import summary."""
    with db() as con:
        jobs=int(con.execute("SELECT COUNT(*) FROM qpu_jobs").fetchone()[0])
        completed_jobs=int(con.execute("SELECT COUNT(*) FROM qpu_jobs WHERE state='completed'").fetchone()[0])
        counts=int(con.execute("SELECT COUNT(*) FROM qpu_counts").fetchone()[0])
        hw=int(con.execute("SELECT COUNT(*) FROM hardware_results WHERE status='ok'").fetchone()[0])
        replay=int(con.execute("SELECT COUNT(*) FROM replay_samples").fetchone()[0])
        pending=int(con.execute("""
            SELECT COUNT(DISTINCT i.job_id)
            FROM qpu_job_items i
            LEFT JOIN qpu_counts c ON c.key=i.item_key
            WHERE c.key IS NULL
        """).fetchone()[0])
    info={
        "preexisting_checkpoint_before_v6":bool(PREEXISTING_CHECKPOINT),
        "saved_qpu_jobs":jobs,
        "completed_qpu_jobs":completed_jobs,
        "saved_qpu_count_records":counts,
        "completed_algorithm_configs":hw,
        "saved_replay_draws":replay,
        "pending_remote_jobs_to_recover":pending,
    }
    print("Resume checkpoint summary:",info)
    return info


if REQUIRE_EXISTING_V5_CHECKPOINT and not PREEXISTING_CHECKPOINT:
    raise RuntimeError("Expected old checkpoint; refusing to create a new campaign")

init_db()
previous_campaign=get_run_meta("v7_campaign_revision")
if previous_campaign is not None and previous_campaign!=REVISION:
    raise RuntimeError(f"Checkpoint belongs to a different campaign: {previous_campaign}")
set_run_meta("v7_campaign_revision",REVISION)
set_run_meta("v7_config",{"scheduler":SCHEDULER,"kD":FIXED_KD,
                           "fraction":DSWAP_BUDGET_FRACTION,"exec":EXECUTION_PROTOCOL,
                           "fingerprint_legacy_amplitude":LEGACY_AMPLITUDE_REVISION})
RESUME_SUMMARY=checkpoint_resume_summary()
print("Checkpoint DB:",DB_PATH.resolve())
print("Resume pending IBM jobs:",RESUME_PENDING_QPU_JOBS,
      "| skip completed configs:",SKIP_COMPLETED_HARDWARE_CONFIGS,
      "| autosave partial results:",AUTOSAVE_PARTIAL_RESULTS)


In [ ]:
# ============================================================
# Raw count helpers and lightweight readout mitigation
# ============================================================
def count_prob(counts,target):
    total=max(1,sum(int(v) for v in counts.values()))
    return float(int(counts.get(str(target),0))/total)


def make_readout_cal_circuits(num_qubits):
    from qiskit import QuantumCircuit
    circs=[]; labels=[]
    for state in range(2**num_qubits):
        qc=QuantumCircuit(num_qubits,num_qubits,name=f"rocal_{num_qubits}q_{state}")
        bits=[(state>>q)&1 for q in range(num_qubits)]
        for q,b in enumerate(bits):
            if b: qc.x(q)
        qc.barrier(); qc.measure(range(num_qubits),range(num_qubits))
        circs.append(qc); labels.append("".join(str(bits[q]) for q in reversed(range(num_qubits))))
    return circs,labels


def confusion_from_counts(counts_list, labels):
    m=len(labels); mat=np.zeros((m,m),float)
    for j,c in enumerate(counts_list):
        tot=max(1,sum(c.values()))
        for i,label in enumerate(labels): mat[i,j]=c.get(label,0)/tot
    return mat


def mitigate_target_probability(counts, labels, confusion, target):
    y=np.array([counts.get(lbl,0) for lbl in labels],float)
    if y.sum()<=0: return np.nan
    y/=y.sum()
    x=np.linalg.pinv(confusion)@y
    x=np.clip(x,0,None)
    if x.sum()>0: x/=x.sum()
    return float(x[labels.index(str(target))])


In [ ]:
# ============================================================
# Generic Runtime submission + immediate job-ID checkpointing + restart recovery
# ============================================================
def _runtime_service():
    svc=globals().get("service",None)
    if svc is None:
        raise RuntimeError("IBM Runtime service is not initialized.")
    return svc


def _is_ibm_auth_error(exc):
    # Retry of a job is not a solution to invalid/expired credentials.
    name=type(exc).__name__.lower()
    msg=str(exc).lower()
    return ("notauthorized" in name or "unauthorized" in name
            or "authentication" in name or "401 client error" in msg
            or "http 401" in msg or "status code 401" in msg)


def _raise_on_ibm_auth_error(exc,job_id=None):
    if _is_ibm_auth_error(exc):
        jid=(f" for job {job_id}" if job_id else "")
        raise RuntimeError(
            "IBM Quantum authentication failed (HTTP 401)"+jid+". "
            "All saved measurement counts and pending IBM job IDs are preserved. "
            "Restore/rotate IBM_QUANTUM_TOKEN and verify IBM_INSTANCE_CRN, "
            "restart the kernel, then Run All against the SAME checkpoint directory. "
            "Do not delete the checkpoint or submit replacement measurements for this job."
        ) from exc


# Legacy job retrieval: recovery may require its ORIGINAL IBM instance, even when
# new submissions use a different CRN. The service is initialized lazily and ONLY
# for recovery, never for new QPU submissions.
_PREVIOUS_RECOVERY_SERVICE = None
_PREVIOUS_RECOVERY_SERVICE_LOCK = threading.RLock()
_RECOVERY_PROCESS_STARTED_AT = time.time()

def _previous_runtime_service():
    """Use the verified old-instance service cached by the recovery preflight first."""
    global _PREVIOUS_RECOVERY_SERVICE
    with _PREVIOUS_RECOVERY_SERVICE_LOCK:
        if _PREVIOUS_RECOVERY_SERVICE is not None:
            return _PREVIOUS_RECOVERY_SERVICE
        crn = os.getenv("IBM_PREVIOUS_INSTANCE_CRN", "").strip()
        if not crn:
            return None
        token = (os.getenv("IBM_PREVIOUS_QUANTUM_TOKEN", "").strip()
                 or os.getenv("IBM_QUANTUM_TOKEN", "").strip())
        if not token:
            raise RuntimeError("Set IBM_QUANTUM_TOKEN for previous-instance job recovery")
        from qiskit_ibm_runtime import QiskitRuntimeService
        _PREVIOUS_RECOVERY_SERVICE = QiskitRuntimeService(
            channel="ibm_quantum_platform", token=token, instance=crn)
        return _PREVIOUS_RECOVERY_SERVICE


def _is_ibm_job_not_found(exc):
    name=type(exc).__name__.lower()
    msg=str(exc).lower()
    return ("runtimejobnotfound" in name or
            "job not found" in msg or "jobnotfound" in name)


def _retrieve_from_service(svc,job_id):
    try:
        return svc.job(str(job_id))
    except TypeError:
        return svc.job(job_id=str(job_id))


def _prepare_inaccessible_job_archive():
    with db() as con:
        con.execute("""CREATE TABLE IF NOT EXISTS inaccessible_job_items_archive (
            item_key TEXT NOT NULL, old_job_id TEXT NOT NULL, batch_index INTEGER,
            arm_a INTEGER,arm_b INTEGER,treatment TEXT,stats_json TEXT,created_at REAL,
            reason TEXT,archived_at REAL NOT NULL,
            PRIMARY KEY(item_key,old_job_id))""")


def _release_inaccessible_job_after_user_optin(job_id,reason):
    """Explicit recovery escape hatch. NEVER claim IBM confirmed terminal failure.

    Completed qpu_counts/physical_manifest are untouched. Every missing item is
    archived, with its old job id, before removal of ACTIVE ownership. Future
    replacement measurements have a NEW job id; old jobs may still have consumed
    QPU time/physical shots that are not recoverable through this account.
    """
    if INACCESSIBLE_JOB_POLICY != "replace":
        raise RuntimeError("Inaccessible job replacement was not enabled.")
    _prepare_inaccessible_job_archive()
    jid=str(job_id); now=time.time()
    with db() as con:
        row=con.execute("SELECT state,submitted_at FROM qpu_jobs WHERE job_id=?",(jid,)).fetchone()
        if row is None:
            raise RuntimeError(f"Unknown IBM job {jid}; refusing release.")
        if str(row[0]).lower()=="completed":
            raise RuntimeError(f"IBM job {jid} is already completed; refusing release.")
        # A transient IBM lookup failure on a job submitted in THIS kernel must
        # not silently cause a duplicate even with manual orphan policy enabled.
        if row[1] is None or float(row[1])>=_RECOVERY_PROCESS_STARTED_AT:
            raise RuntimeError(f"IBM job {jid} was submitted during this notebook execution, "
                               "so replacing it is prohibited. Restart and verify its "
                               "remote status/ownership before making any manual decision.")
        rows=con.execute("""
            SELECT i.item_key,i.batch_index,i.arm_a,i.arm_b,i.treatment,
                   i.stats_json,i.created_at
            FROM qpu_job_items i LEFT JOIN qpu_counts c ON c.key=i.item_key
            WHERE i.job_id=? AND c.key IS NULL
        """,(jid,)).fetchall()
        if rows:
            con.executemany("""INSERT OR IGNORE INTO inaccessible_job_items_archive
                (item_key,old_job_id,batch_index,arm_a,arm_b,treatment,stats_json,
                 created_at,reason,archived_at)
                VALUES(?,?,?,?,?,?,?,?,?,?)""",
                [(r[0],jid,*r[1:],str(reason)[:1200],now)
                 for r in rows])
            con.execute("""DELETE FROM qpu_job_items WHERE job_id=? AND
                NOT EXISTS (SELECT 1 FROM qpu_counts WHERE qpu_counts.key=qpu_job_items.item_key)
            """,(jid,))
        con.execute("""UPDATE qpu_jobs
            SET state='inaccessible_abandoned',last_error=?,updated_at=?
            WHERE job_id=?""",(str(reason)[:1200],now,jid))
    print(f"WARNING: old IBM job {jid} could not be retrieved; user explicitly chose "
          f"to replace {len(rows)} MISSING item(s). Previous job is archived, "
          "not declared failed. Possible extra physical QPU shots must be disclosed.")
    return len(rows)


def _handle_missing_previous_instance_job(job_id,error):
    if INACCESSIBLE_JOB_POLICY == "replace":
        return _release_inaccessible_job_after_user_optin(job_id,repr(error))
    raise RuntimeError(
        f"Job {job_id} is not visible via the current/available IBM instance(s). "
        "Its saved counts are preserved and no replacement shots were submitted. "
        "Set IBM_PREVIOUS_INSTANCE_CRN (and IBM_PREVIOUS_QUANTUM_TOKEN if accounts differ) "
        "to recover the ORIGINAL job, then restart with the same checkpoint. "
        "If the old instance truly cannot be accessed, review the original job status first; "
        "only then consider NIGHTHAWK_INACCESSIBLE_JOB_POLICY=replace and "
        "NIGHTHAWK_ACCEPT_UNRECOVERED_JOB_RISK=I_ACCEPT_POSSIBLE_DUPLICATE_QPU_SHOTS."
    ) from error


def _retrieve_runtime_job(job_id):
    svc=_runtime_service()
    try:
        return _retrieve_from_service(svc,job_id)
    except Exception as exc:
        # The new instance can be healthy for submitting fresh jobs while old
        # job IDs are invisible there. Try the ORIGINAL instance before stopping.
        if _is_ibm_job_not_found(exc):
            previous=_previous_runtime_service()
            if previous is not None:
                try:
                    return _retrieve_from_service(previous,job_id)
                except Exception as old_exc:
                    _raise_on_ibm_auth_error(old_exc,job_id)
                    if _is_ibm_job_not_found(old_exc):
                        raise _CrossInstanceJobNotFound(job_id,exc,old_exc) from old_exc
                    raise
            raise _CrossInstanceJobNotFound(job_id,exc,None) from exc
        _raise_on_ibm_auth_error(exc,job_id)
        raise


class _CrossInstanceJobNotFound(RuntimeError):
    def __init__(self,job_id,current_error,previous_error):
        self.job_id=str(job_id)
        self.current_error=current_error
        self.previous_error=previous_error
        super().__init__(f"IBM job {job_id} not found on current instance"
                         +(" or configured previous instance" if previous_error else ""))


def _job_status_text(job,job_id=None):
    try:
        x=job.status()
        raw=str(getattr(x,"name",x)).strip().upper()
        return raw.rsplit(".",1)[-1]  # handles Enum 'JobStatus.ERROR'
    except Exception as exc:
        _raise_on_ibm_auth_error(exc,job_id)
        return "UNKNOWN"


def _terminal_ibm_status(status):
    return str(status).strip().upper() in {"ERROR","FAILED","CANCELLED","CANCELED"}


def _mark_confirmed_terminal_failure(job_id,status,reason=""):
    """Only a confirmed terminal IBM job can release missing item keys for re-submission.

    Finished measurement records are never deleted. Prior job metadata remains in the
    ledger; qpu_job_items keys for still-missing shots are unlinked atomically.
    """
    status=str(status).strip().upper()
    if not _terminal_ibm_status(status):
        raise RuntimeError(f"Refusing to replace IBM job {job_id} with unverified status {status}.")
    if not AUTO_RESUBMIT_CONFIRMED_TERMINAL_JOBS:
        raise RuntimeError(f"IBM job {job_id} is terminal ({status}); manual review required.")
    with db() as con:
        con.execute("""
            UPDATE qpu_jobs SET state=?,last_error=?,updated_at=? WHERE job_id=?
        """,("terminal_"+status,str(reason or f"IBM job ended in {status}")[:1200],
             time.time(),str(job_id)))
        missing=con.execute("""
            SELECT COUNT(*) FROM qpu_job_items i
            LEFT JOIN qpu_counts c ON c.key=i.item_key
            WHERE i.job_id=? AND c.key IS NULL
        """,(str(job_id),)).fetchone()[0]
        con.execute("""
            DELETE FROM qpu_job_items WHERE job_id=?
              AND NOT EXISTS (SELECT 1 FROM qpu_counts WHERE qpu_counts.key=qpu_job_items.item_key)
        """,(str(job_id),))
    print(f"IBM job {job_id} confirmed terminal ({status}); released {missing} MISSING circuit item(s). "
          "Saved results remain intact; only those items may be submitted again.")
    maybe_export_progress()
    return int(missing)


class _ConfirmedIBMTerminalRerunNeeded(RuntimeError):
    def __init__(self,job_id,count):
        super().__init__(f"IBM job {job_id} ended terminally; {count} missing item(s) "
                         "may be resubmitted on restart. Existing counts are preserved.")


def _serialize_stats(stats):
    return json.dumps(dict(stats),default=str,sort_keys=True)


def _persist_job_submission(job,items,shots,role,seed,delta):
    """Write the remote IBM job ID to SQLite immediately after sampler.run returns."""
    jid=safe_job_id(job)
    if not jid or jid=="<unknown>":
        raise RuntimeError("IBM Runtime returned a job without a recoverable job ID; refusing unsafe execution.")
    keys=[x[0] for x in items]
    shot_plan=[int(item[2].get("reservoir_chunk_shots",shots)) if shots == 0 else int(shots)
               for item in items]
    batch_key=hashlib.sha256(
        repr((REVISION,role,int(seed),float(delta),int(shots),keys,shot_plan)).encode()
    ).hexdigest()[:28]
    now=time.time()
    with db() as con:
        con.execute("""
            INSERT OR REPLACE INTO qpu_jobs
            (job_id,batch_key,role,seed,delta,shots,state,item_count,submitted_at,completed_at,last_error,updated_at,execution_protocol)
            VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)
        """,(jid,batch_key,role,int(seed),float(delta),int(shots),"submitted",len(items),now,None,None,now,EXECUTION_PROTOCOL))
        for idx,item in enumerate(items):
            key,exe,stats,arm_a,arm_b,treatment=item
            con.execute("""
                INSERT OR REPLACE INTO qpu_job_items
                (item_key,job_id,batch_index,arm_a,arm_b,treatment,stats_json,created_at)
                VALUES(?,?,?,?,?,?,?,?)
            """,(key,jid,int(idx),int(arm_a),None if arm_b is None else int(arm_b),
                 str(treatment),_serialize_stats(stats),now))
    maybe_export_progress()
    return jid,now


def _job_item_records(job_id):
    with db() as con:
        rows=con.execute("""
            SELECT item_key,batch_index,arm_a,arm_b,treatment,stats_json
            FROM qpu_job_items WHERE job_id=? ORDER BY batch_index
        """,(str(job_id),)).fetchall()
        jrow=con.execute("""
            SELECT role,seed,delta,shots,submitted_at,state FROM qpu_jobs WHERE job_id=?
        """,(str(job_id),)).fetchone()
    if jrow is None:
        raise RuntimeError(f"Checkpoint metadata missing for IBM job {job_id}.")
    role,seed,delta,shots,submitted_at,state=jrow
    items=[{
        "key":r[0],"batch_index":int(r[1]),"arm_a":int(r[2]),
        "arm_b":None if r[3] is None else int(r[3]),"treatment":str(r[4]),
        "stats":json.loads(r[5])
    } for r in rows]
    return {"role":role,"seed":int(seed),"delta":float(delta),"shots":int(shots),
            "submitted_at":float(submitted_at),"state":state,"items":items}


def _persist_completed_remote_job(job,job_id,result):
    meta=_job_item_records(job_id)
    pubs=list(result)
    if len(pubs)!=len(meta["items"]):
        raise RuntimeError(
            f"IBM job {job_id} returned {len(pubs)} publication results for "
            f"{len(meta['items'])} checkpointed items."
        )
    now=time.time()
    wall=max(0.0,now-meta["submitted_at"])
    runtime_metrics=None
    try:
        runtime_metrics=json.loads(json.dumps(job.metrics(),default=str))
    except Exception:
        pass

    with db() as con:
        for rec,pub in zip(meta["items"],pubs):
            counts=extract_counts(pub)
            stats=dict(rec["stats"])
            item_role = str(stats.get("record_role", meta["role"]))
            if runtime_metrics is not None:
                stats["runtime_metrics"]=runtime_metrics
            # A qpu_jobs.shots=0 sentinel identifies PUB-wise shot counts, which
            # were checkpointed individually in each item's stats_json.
            item_shots=(int(stats["reservoir_chunk_shots"])
                        if int(meta["shots"])==0 else int(meta["shots"]))
            if item_shots<=0:
                raise RuntimeError(f"Invalid shots for recovered job {job_id}")
            dur=stats.get("scheduled_duration_us",np.nan)
            try: dur=float(dur)
            except Exception: dur=np.nan
            est_time=dur*1e-6*item_shots if np.isfinite(dur) else np.nan
            try: oqps=int(float(stats.get("oracle_queries_per_shot",0)))
            except Exception: oqps=0
            logical_queries=oqps*item_shots

            con.execute("""
                INSERT OR REPLACE INTO qpu_counts
                (key,role,seed,arm_a,arm_b,delta,treatment,shots,counts_json,job_id,submit_wall_sec,created_at,circuit_sig)
                VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)
            """,(rec["key"],item_role,meta["seed"],rec["arm_a"],rec["arm_b"],
                 meta["delta"],rec["treatment"],item_shots,json.dumps(counts),
                 str(job_id),wall,now,stats.get("circuit_sig")))

            con.execute("""
                INSERT OR REPLACE INTO physical_manifest
                (key,role,seed,arm_a,arm_b,delta,treatment,shots,job_id,
                 logical_oracle_queries,circuit_executions,estimated_scheduled_qpu_time_sec,
                 submit_wall_sec,stats_json,created_at,circuit_sig)
                VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
            """,(rec["key"],item_role,meta["seed"],rec["arm_a"],rec["arm_b"],
                 meta["delta"],rec["treatment"],item_shots,str(job_id),
                 logical_queries,item_shots,
                 float(est_time) if np.isfinite(est_time) else None,
                 wall,json.dumps(stats,default=str),now,stats.get("circuit_sig")))

        con.execute("""
            UPDATE qpu_jobs SET state=?,completed_at=?,last_error=?,updated_at=? WHERE job_id=?
        """,("completed",now,None,now,str(job_id)))

    maybe_export_progress()


def recover_qpu_job(job_id,initial_job=None):
    """Recover existing IBM job, with bounded transient retries and terminal-only replacement.

    401/unknown states: do NOT resubmit. Confirmed ERROR/FAILED/CANCELLED: release only
    uncompleted items for the caller to resubmit. COMPLETED/DONE: persist original data.
    """
    jid=str(job_id)
    last_exc=None
    for attempt in range(1,RUNTIME_RECOVERY_ATTEMPTS+1):
        try:
            job=initial_job if attempt==1 and initial_job is not None else _retrieve_runtime_job(jid)
            status=_job_status_text(job,jid)
            if _terminal_ibm_status(status):
                _mark_confirmed_terminal_failure(jid,status,
                                                f"IBM returned terminal status {status}; {last_exc!r}")
                return None
            with db() as con:
                con.execute("UPDATE qpu_jobs SET state=?,updated_at=? WHERE job_id=?",
                            (status,time.time(),jid))
            result=job.result(timeout=QPU_TIMEOUT_SEC) if QPU_TIMEOUT_SEC else job.result()
            if result is None:
                raise RuntimeError(f"IBM job {jid} result is None")
            _persist_completed_remote_job(job,jid,result)
            return job
        except Exception as exc:
            _raise_on_ibm_auth_error(exc,jid)
            last_exc=exc
            # If a local worker has already recovered this same job, accept its result.
            with db() as con:
                total=con.execute("SELECT COUNT(*) FROM qpu_job_items WHERE job_id=?",(jid,)).fetchone()[0]
                missing=con.execute("""
                    SELECT COUNT(*) FROM qpu_job_items i LEFT JOIN qpu_counts c ON c.key=i.item_key
                    WHERE i.job_id=? AND c.key IS NULL
                """,(jid,)).fetchone()[0]
                if total>0 and missing==0:
                    return None
                con.execute("UPDATE qpu_jobs SET last_error=?,updated_at=? WHERE job_id=?",
                            (repr(exc)[:1200],time.time(),jid))
            # A job can become terminal while the result API is reporting 1319/out-of-sync.
            try:
                refreshed=_retrieve_runtime_job(jid)
                after=_job_status_text(refreshed,jid)
            except Exception as status_exc:
                _raise_on_ibm_auth_error(status_exc,jid)
                after="UNKNOWN"
            if _terminal_ibm_status(after):
                _mark_confirmed_terminal_failure(jid,after,repr(exc))
                return None
            if attempt>=RUNTIME_RECOVERY_ATTEMPTS:
                maybe_export_progress()
                if isinstance(exc,_CrossInstanceJobNotFound) and after == "UNKNOWN":
                    _handle_missing_previous_instance_job(jid,exc)
                    return None
                raise RuntimeError(
                    f"IBM job {jid} could not be recovered after {attempt} attempts "
                    f"(last status {after}). Its checkpoint ownership is preserved; "
                    "no replacement shots were submitted. Recheck IBM status and credentials "
                    f"before resuming. Last retrieval error: {type(exc).__name__}"
                ) from exc
            wait=min(30.0,RUNTIME_RECOVERY_RETRY_BASE_SEC*(2**(attempt-1)))
            print(f"Retrying existing IBM job {jid} after retrieval error "
                  f"({type(exc).__name__}); attempt {attempt+1}/{RUNTIME_RECOVERY_ATTEMPTS} "
                  f"in {wait:.1f}s; remote status={after}.")
            if wait: time.sleep(wait)
    raise AssertionError("Unreachable recovery path")


def _pending_job_ids_for_item_keys(keys):
    if not keys: return []
    marks=",".join("?" for _ in keys)
    with db() as con:
        rows=con.execute(f"""
            SELECT DISTINCT i.job_id
            FROM qpu_job_items i
            LEFT JOIN qpu_counts c ON c.key=i.item_key
            WHERE i.item_key IN ({marks}) AND c.key IS NULL
            ORDER BY i.created_at
        """,tuple(keys)).fetchall()
    return [r[0] for r in rows]


def run_compiled_batch(sampler,compiled_items,shots,role,seed,delta):
    """Resume-safe Runtime execution.

    Every completed circuit is reused from qpu_counts. If a circuit belongs to an IBM job that
    was submitted before an interruption, that exact job is recovered by job ID before any new
    submission is considered. Newly submitted job IDs are committed immediately after sampler.run().
    compiled_items may contain many PUBs; the function chunks them at MAX_CIRCUITS_PER_JOB.
    """
    if not compiled_items:
        return []

    out=[]
    for start in range(0,len(compiled_items),MAX_CIRCUITS_PER_JOB):
        chunk=compiled_items[start:start+MAX_CIRCUITS_PER_JOB]

        # 1) Recover any already-submitted remote jobs that contain still-missing items.
        with db() as con:
            missing_keys=[item[0] for item in chunk
                          if con.execute("SELECT 1 FROM qpu_counts WHERE key=?",(item[0],)).fetchone() is None]

        if missing_keys and RESUME_PENDING_QPU_JOBS:
            for jid in _pending_job_ids_for_item_keys(missing_keys):
                print("Recovering IBM job:",jid)
                recover_qpu_job(jid)

        # 2) Recheck after recovery; submit only genuinely unseen items.
        with db() as con:
            new_items=[item for item in chunk
                       if con.execute("SELECT 1 FROM qpu_counts WHERE key=?",(item[0],)).fetchone() is None]

        if new_items:
            # Safety: if an unresolved checkpointed job still owns any item, do not duplicate it.
            pending=_pending_job_ids_for_item_keys([x[0] for x in new_items])
            if pending:
                raise RuntimeError(
                    "Unresolved checkpointed IBM job(s) still own requested circuits: "
                    + ", ".join(map(str,pending))
                )
            if not RUN_QPU:
                raise RuntimeError("QPU data missing while RUN_QPU=False")

            # Keep concurrency bounded to the account/backend capacity, not unlimited submissions.
            with _SUBMISSION_GATE:
                if shots is None:
                    # SamplerV2 PUBs can specify individual shots as (circuit,None,shots).
                    # Use zero only in the qpu_jobs parent record as a mixed-shot sentinel.
                    pubs=[]
                    for x in new_items:
                        per_pub_shots=int(x[2].get("reservoir_chunk_shots",0))
                        if per_pub_shots<=0:
                            raise RuntimeError("Mixed-shot PUB missing its exact shot count")
                        pubs.append((x[1],None,per_pub_shots))
                    job=sampler.run(pubs)
                    stored_shots=0
                else:
                    job=sampler.run([x[1] for x in new_items],shots=int(shots))
                    stored_shots=int(shots)
                # ALWAYS persist the recoverable job id before waiting or allowing another submit.
                jid,_submitted_at=_persist_job_submission(job,new_items,stored_shots,role,seed,delta)
                print("Submitted IBM Batch job:",jid,"| circuits:",len(new_items),"| role:",role)
                # Reuse exactly the same recovery/error policy for freshly submitted jobs.
                # A 1319 transient can recover; a confirmed ERROR can release only missing
                # measurements. 401 never causes duplicate submission.
                recover_qpu_job(jid,initial_job=job)
                # If IBM confirmed terminal failure, acquire just the released items.
                with db() as con:
                    unresolved=[x for x in new_items if con.execute(
                        "SELECT 1 FROM qpu_counts WHERE key=?",(x[0],)).fetchone() is None]
                if unresolved:
                    raise _ConfirmedIBMTerminalRerunNeeded(jid,len(unresolved))

        # 3) Return records in the same order as the requested chunk.
        with db() as con:
            for item in chunk:
                row=con.execute(
                    "SELECT counts_json,treatment,job_id FROM qpu_counts WHERE key=?",(item[0],)
                ).fetchone()
                if row is None:
                    raise RuntimeError(f"No completed QPU result available for circuit key {item[0]}.")
                out.append((item[0],json.loads(row[0]),row[1],row[2]))
    return out



# V8: for >MAX_CIRCUITS_PER_JOB items, submit separate, nonoverlapping PUB
# chunks concurrently. Each serial worker persists its own IBM job ID before
# waiting; _SUBMISSION_GATE still bounds total in-flight IBM Runtime requests.
_run_compiled_batch_serial=run_compiled_batch
def run_compiled_batch(sampler,compiled_items,shots,role,seed,delta):
    # Packing by both PUB count and circuit executions avoids oversized jobs
    # when first-level MLAE prefetch grows. No circuit or shot count is changed.
    chunks=[]
    current=[]
    current_executions=0
    for item in compiled_items:
        pub_shots=(int(item[2].get("reservoir_chunk_shots",0))
                   if shots is None else int(shots))
        if pub_shots<=0:
            raise RuntimeError("Cannot batch a PUB with nonpositive shots")
        if pub_shots>MAX_EXECUTIONS_PER_JOB:
            raise RuntimeError("Single PUB exceeds execution limit; adjust acquisition size")
        if current and (len(current)>=MAX_CIRCUITS_PER_JOB or
                        current_executions+pub_shots>MAX_EXECUTIONS_PER_JOB):
            chunks.append(current)
            current=[]
            current_executions=0
        current.append(item)
        current_executions+=pub_shots
    if current:
        chunks.append(current)
    if len(chunks)<=1:
        return _run_compiled_batch_serial(sampler,compiled_items,shots,role,seed,delta)
    if not ENABLE_CONCURRENT_PUB_CHUNKS:
        return [record for chunk in chunks for record in
                _run_compiled_batch_serial(sampler,chunk,shots,role,seed,delta)]
    limit=min(len(chunks),MAX_INFLIGHT_SUBMISSIONS)
    results=[None]*len(chunks)
    with ThreadPoolExecutor(max_workers=limit,thread_name_prefix="nighthawk-pub") as executor:
        futures={executor.submit(_run_compiled_batch_serial,sampler,chunk,shots,role,seed,delta):idx
                 for idx,chunk in enumerate(chunks)}
        # Wait for every submitted job even if an earlier worker failed; leaving
        # futures alive after closing the executor could lose local completion
        # metadata, although the remote job IDs remain checkpointed.
        problems=[]
        for future in as_completed(futures):
            i=futures[future]
            try: results[i]=future.result()
            except Exception as exc: problems.append((i,repr(exc)))
    if problems:
        raise RuntimeError(f"{len(problems)} Runtime PUB chunk(s) need recovery: {problems}")
    return [record for chunk in results for record in chunk]

def export_physical_manifest():
    with db() as con:
        df=pd.read_sql_query("SELECT * FROM physical_manifest ORDER BY created_at",con)
    if len(df):
        stats=pd.json_normalize(df.pop("stats_json").apply(json.loads))
        stats=stats.loc[:, ~stats.columns.isin(df.columns)]
        df=pd.concat([df.reset_index(drop=True),stats.reset_index(drop=True)],axis=1)
        df.to_csv(OUTDIR/"physical_manifest.csv",index=False)
    else:
        # Still create an empty file so progress inspection is predictable.
        df.to_csv(OUTDIR/"physical_manifest.csv",index=False)
    return df

# Serializing CSV snapshots avoids partially overwritten outputs when two streams finish together.
_raw_export_checkpoint_ledger = export_checkpoint_ledger
_raw_export_physical_manifest = export_physical_manifest
def export_checkpoint_ledger(*args,**kwargs):
    with _CSV_EXPORT_LOCK: return _raw_export_checkpoint_ledger(*args,**kwargs)
def export_physical_manifest(*args,**kwargs):
    with _CSV_EXPORT_LOCK: return _raw_export_physical_manifest(*args,**kwargs)


def maybe_export_progress():
    """Best-effort periodic human-readable export; SQLite remains source of truth."""
    global _last_progress_export
    if not AUTOSAVE_PARTIAL_RESULTS:
        return
    if time.monotonic()-_last_progress_export < _PROGRESS_EXPORT_INTERVAL_SEC:
        return
    with _CSV_EXPORT_LOCK:
        if time.monotonic()-_last_progress_export < _PROGRESS_EXPORT_INTERVAL_SEC:
            return
        _last_progress_export=time.monotonic()
        try:
            if "export_partial_hardware_artifacts" in globals():
                export_partial_hardware_artifacts()
            else:
                export_checkpoint_ledger()
                export_physical_manifest()
        except Exception as exc:
            print("Nonfatal CSV progress export problem:",repr(exc))


In [ ]:
# ============================================================
# Readout calibration on the locked pair (1q and 2q)
# ============================================================
MITIGATION = {}

def calibrate_readout(sampler,backend,pair):
    """Resume-safe combined one- and two-qubit readout calibration.

    Preserves V9 item keys and all shot counts. Previously completed or pending
    work is found by the original per-circuit key; only missing PUBs are sent.
    """
    specs={}
    combined=[]
    for nq in (1,2):
        logicals,labels=make_readout_cal_circuits(nq)
        keys=[]
        for j,logical in enumerate(logicals):
            exe,stats=compile_alap_variant(logical,backend,pair,"no_dd")
            stats=dict(stats)
            stats.update({"circuit_k":-1,"policy_key":"readout",
                          "oracle_queries_per_shot":0,"execution_protocol":EXECUTION_PROTOCOL})
            key=hashlib.sha256(repr((REVISION,_backend_name(backend),tuple(pair),
                   "readout_cal",nq,j)).encode()).hexdigest()[:28]
            combined.append((key,exe,stats,j,None,"no_dd"))
            keys.append(key)
        specs[nq]=(labels,keys)

    # Both groups have the same role, seed, delta and shots: safe to submit together.
    got=run_compiled_batch(sampler,combined,READOUT_CAL_SHOTS,"readout_cal",-1,0.0)
    by_key={key:counts for key,counts,_,_ in got}
    out={}
    for nq,(labels,keys) in specs.items():
        counts=[by_key[key] for key in keys]
        out[f"{nq}q"]={"labels":labels,"confusion":confusion_from_counts(counts,labels)}
    np.savez(OUTDIR/"readout_mitigation.npz",
             labels1=np.array(out["1q"]["labels"]), conf1=out["1q"]["confusion"],
             labels2=np.array(out["2q"]["labels"]), conf2=out["2q"]["confusion"])
    return out
def load_readout_mitigation():
    p=OUTDIR/"readout_mitigation.npz"
    if not p.exists(): return {}
    z=np.load(p,allow_pickle=True)
    return {"1q":{"labels":z["labels1"].tolist(),"confusion":z["conf1"]},
            "2q":{"labels":z["labels2"].tolist(),"confusion":z["conf2"]}}

def corrected_target(counts,target,nq):
    m=MITIGATION.get(f"{nq}q") if isinstance(MITIGATION,dict) else None
    if not ENABLE_READOUT_MITIGATION or not m:
        return count_prob(counts,target)
    return mitigate_target_probability(counts,m["labels"],m["confusion"],target)


## Hardware scheduling

The circuits use ALAP scheduling. The available idle windows did not permit insertion of complete XX/XY4 sequences in the reported production circuits.


In [ ]:

# ============================================================
# DD calibration and frozen production policy, separately for every MLAE depth
# Batched calibration PUBs; amplitude policy may reuse legacy hardware counts.
# ============================================================
def representative_calibration_specs(delta):
    specs=[]
    for k in DD_CALIBRATION_K_VALUES:
        for p in [0.50,0.50+delta]:
            specs.append((f"amplitude_k{k}","amplitude",int(k),p,None,"1",amplified_probability(p,k)))
    specs.extend([
        (f"dswap_k{FIXED_KD}","dswap",FIXED_KD,0.50,0.50+delta/20.0,"11",dswap_p11(0.50,0.50+delta/20.0,FIXED_KD)),
        (f"dswap_k{FIXED_KD}","dswap",FIXED_KD,0.50,0.50+delta,"11",dswap_p11(0.50,0.50+delta,FIXED_KD)),
    ])
    return specs


def load_dd_policy():
    with db() as con:
        rows=con.execute("SELECT policy_key,chosen_treatment FROM dd_policy").fetchall()
    return {r:t for r,t in rows}


def collect_dd_calibration_context(backend,sampler,pair,delta,seed):
    """One combined context submission, preserving role-specific PUB identity.

    The merged job contains the same independently prepared dSWAP/amplitude circuits
    and shots as V9. Its per-item `record_role` ensures qpu_counts and physical
    manifests still reflect the real circuit type. Existing V9 items are reused.
    """
    meta_by_key={}
    combined=[]
    for policy_key,role,k,p_a,p_b,target,ideal in representative_calibration_specs(delta):
        logical=(build_amplitude_circuit(p_a,k) if role=="amplitude"
                 else build_dswap_circuit(p_a,p_b,FIXED_KD))
        with _COMPILE_LOCK:
            variants=eligible_variants(logical,backend,pair)
        for tr,exe,stats in variants:
            stats=dict(stats)
            stats.update({"circuit_k":int(k),"policy_key":policy_key,
                          "record_role":role,
                          "oracle_queries_per_shot":oracle_uses_from_k(k) if role=="amplitude"
                              else 2*oracle_uses_from_k(FIXED_KD),
                          "execution_protocol":EXECUTION_PROTOCOL})
            ver=LEGACY_AMPLITUDE_REVISION if role=="amplitude" else REVISION
            key=hashlib.sha256(repr((ver,_backend_name(backend),tuple(pair),
                 "ddcal",delta,seed,policy_key,p_a,p_b,tr)).encode()).hexdigest()[:28]
            combined.append((key,exe,stats,0,None if p_b is None else 1,tr))
            meta_by_key[key]={"delta":delta,"seed":seed,"policy_key":policy_key,
                "role":role,"k":k,"target":target,"ideal":ideal}

    got=run_compiled_batch(sampler,combined,READOUT_CAL_SHOTS,
                           "dd_calibration",seed,delta)
    local_rows=[]
    for key,counts,tr,jid in got:
        m=meta_by_key[key]
        obs=corrected_target(counts,m["target"],1 if m["role"]=="amplitude" else 2)
        local_rows.append({"delta":m["delta"],"seed":m["seed"],
            "policy_key":m["policy_key"],"role":m["role"],"k":m["k"],
            "treatment":tr,"ideal":m["ideal"],"observed":obs,
            "abs_error":abs(obs-m["ideal"]),"job_id":jid})
    return local_rows
def calibrate_dd(service,backend,sampler,pair):
    contexts=[(d,s) for d in DELTAS for s in DD_CALIBRATION_SEEDS]
    rows=[]
    workers=min(DD_PREFLIGHT_WORKERS,len(contexts))
    if workers==1:
        for d,s in contexts:
            rows.extend(collect_dd_calibration_context(backend,sampler,pair,d,s))
    else:
        with ThreadPoolExecutor(max_workers=workers,thread_name_prefix="nighthawk-dd") as pool:
            tasks={pool.submit(collect_dd_calibration_context,backend,sampler,pair,d,s):(d,s)
                   for d,s in contexts}
            for f in as_completed(tasks):
                rows.extend(f.result())
                print("DD calibration context complete:",tasks[f])

    df=pd.DataFrame(rows)
    df.to_csv(OUTDIR/"dd_calibration_raw.csv",index=False)

    summaries=[]
    for policy_key,g in df.groupby("policy_key"):
        agg=g.groupby("treatment").agg(
            mae=("abs_error","mean"),n=("abs_error","size")
        ).reset_index()
        no=float(agg.loc[agg.treatment=="no_dd","mae"].iloc[0])
        candidates=agg[agg.n>=DD_MIN_VALID_POINTS].sort_values("mae")
        best=candidates.iloc[0]
        improvement=(no-float(best.mae))/max(no,1e-12)
        chosen=(str(best.treatment)
                if str(best.treatment)!="no_dd"
                and improvement>=DD_MIN_RELATIVE_MAE_IMPROVEMENT
                else "no_dd")
        payload={"table":agg.to_dict("records"),"relative_improvement":float(improvement)}
        with db() as con:
            con.execute(
                "INSERT OR REPLACE INTO dd_policy VALUES (?,?,?,?)",
                (policy_key,chosen,json.dumps(payload),time.time())
            )
        summaries.append({
            "policy_key":policy_key,
            "chosen_treatment":chosen,
            "relative_improvement":improvement
        })

    pd.DataFrame(summaries).to_csv(
        OUTDIR/"dd_policy_summary.csv",index=False
    )
    print("Frozen DD policy:",load_dd_policy())
    return df


if RUN_QPU:
    service=get_ibm_service()
    backend=service.backend(IBM_BACKEND_NAME)
    sampler=get_sampler(backend)
    PAIR,pair_ranking=load_or_freeze_physical_pair(backend)
    print("Backend:",_backend_name(backend),"frozen pair:",PAIR)
    if len(pair_ranking):
        pair_ranking.to_csv(OUTDIR/"physical_pair_ranking.csv",index=False)

    # V12 fixed calibration values copied from completed V11 characterization on
    # ibm_rensselaer, physical qubits (66,67), with every depth selecting no_dd.
    # These are constants, not imported V11 quantum measurement results.
    if tuple(PAIR) != PILOT_PAIR:
        raise RuntimeError(f"V11 calibration was on {PILOT_PAIR}, got {PAIR}")
    frozen_policy_keys=[f"amplitude_k{k}" for k in DD_CALIBRATION_K_VALUES]
    frozen_policy_keys.append(f"dswap_k{FIXED_KD}")
    existing_policies=load_dd_policy()
    if existing_policies and any(v!="no_dd" for v in existing_policies.values()):
        raise RuntimeError("V14 checkpoint has DD treatments inconsistent with V11 no-DD policy.")
    if not existing_policies:
        with db() as con:
            for key in frozen_policy_keys:
                con.execute("INSERT OR REPLACE INTO dd_policy VALUES (?,?,?,?)",
                            (key,"no_dd",json.dumps({"source":"V11 completed hardware calibration",
                                                      "pair":[66,67],"backend":"ibm_rensselaer"}),time.time()))
    if not (OUTDIR/"readout_mitigation.npz").exists():
        labels1=np.array(["0","1"])
        conf1=np.array([[1.0,0.0048828125],[0.0,0.9951171875]],dtype=float)
        labels2=np.array(["00","01","10","11"])
        conf2=np.array([
            [0.998779296875,0.004150390625,0.00732421875,0.0],
            [0.00048828125,0.9951171875,0.0,0.008544921875],
            [0.000732421875,0.0,0.99267578125,0.00439453125],
            [0.0,0.000732421875,0.0,0.987060546875]
        ],dtype=float)
        np.savez(OUTDIR/"readout_mitigation.npz",
                 labels1=labels1,conf1=conf1,labels2=labels2,conf2=conf2)
    set_run_meta("v14_frozen_calibration",{
        "source":"V11 completed DD/readout calibration, 2026-10-18",
        "pair":[66,67],"backend":IBM_BACKEND_NAME,
        "readout_matrix_reused":True,"chosen_dd":"no_dd",
        "raw_gate_threshold":PILOT_RAW_FAR_CUTOFF,
        "gate_threshold_is_certified":False,
        "measurement_reuse_from_v11":False})
    print("V14: reusing V11 frozen readout + no-DD policies (NO calibration hardware jobs).")

    MITIGATION=load_readout_mitigation()
    if ENABLE_READOUT_MITIGATION and not MITIGATION:
        MITIGATION=calibrate_readout(sampler,backend,PAIR)

    # Re-running calibration is resume-safe: already completed V5 calibration circuits are reused,
    # pending V5 jobs are recovered, and only genuinely missing items are submitted.
    required_policy_keys={f"amplitude_k{k}" for k in DD_CALIBRATION_K_VALUES}
    required_policy_keys.add(f"dswap_k{FIXED_KD}")
    frozen_keys=set(load_dd_policy())
    # No re-calibration of frozen policies unless explicitly requested. Incomplete
    # checkpoints still recover pending jobs and finish calibration as before.
    needs_dd = not required_policy_keys.issubset(frozen_keys)
    if FORCE_DD_RECALIBRATION or (RUN_DD_CALIBRATION and needs_dd):
        dd_calibration=calibrate_dd(service,backend,sampler,PAIR)
    elif needs_dd:
        raise RuntimeError("Missing DD policies but RUN_DD_CALIBRATION=False. Enable DD calibration.")
    else:
        print("V10: Reusing complete frozen DD policies; skipping upfront QPU calibration.")
else:
    service=backend=sampler=None
    PAIR=None
    print("QPU cells are configured but not executed. Set RUN_QPU=True when ready.")


## Hardware campaign

The following implementation includes the published circuit generation, query accounting, streaming comparisons, and checkpointed QPU measurement workflow. Execution requires private archived state and explicit opt-in.


In [ ]:

# ============================================================
# Live hardware helper: reusable QPU count reservoirs + exact replay/subsampling
# V6 execution optimization: batch only circuits the logical algorithm actually needs.
# ============================================================
def _policy_key(role,k=0):
    return f"amplitude_k{int(k)}" if role=="amplitude" else f"dswap_k{FIXED_KD}"


def production_treatment(role,k=0,override=None):
    if override is not None:
        return str(override)
    return load_dd_policy().get(_policy_key(role,k),"no_dd")


def _stable_seed(*parts):
    h=hashlib.sha256(repr(parts).encode()).hexdigest()[:16]
    return int(h,16) % (2**63-1)


def _circuit_signature(backend,pair,role,seed,delta,arm_a,arm_b,k,treatment,p_a,p_b):
    # Amplitude histograms stay compatible with V5 after a deliberate DB snapshot import.
    # New amplified dSWAP histograms have a NEW revision; no accidental k0 reuse.
    signature_revision = LEGACY_AMPLITUDE_REVISION if str(role)=="amplitude" else REVISION
    payload=(signature_revision,_backend_name(backend),tuple(pair),"reservoir",str(role),int(seed),float(delta),
             int(arm_a),None if arm_b is None else int(arm_b),int(k),str(treatment),
             round(float(p_a),14),None if p_b is None else round(float(p_b),14))
    return hashlib.sha256(repr(payload).encode()).hexdigest()[:32]


def _aggregate_many_reservoir_counts(circuit_sigs):
    """Bulk-load immutable IBM shot histograms using the indexed signature column."""
    wanted=list(dict.fromkeys(str(x) for x in circuit_sigs))
    if not wanted:
        return {}
    accum={sig:Counter() for sig in wanted}
    parts={sig:0 for sig in wanted}
    with db() as con:
        for start in range(0,len(wanted),400):
            batch=wanted[start:start+400]
            marks=",".join("?" for _ in batch)
            rows=con.execute(
                f"SELECT circuit_sig,counts_json FROM qpu_counts WHERE circuit_sig IN ({marks}) ORDER BY created_at",
                tuple(batch)
            ).fetchall()
            for sig,counts_json in rows:
                sig=str(sig)
                accum[sig].update({str(k):int(v) for k,v in json.loads(counts_json).items()})
                parts[sig]+=1
    return {s:(dict(accum[s]),int(sum(accum[s].values())),int(parts[s])) for s in wanted}


def _aggregate_reservoir_counts(circuit_sig):
    return _aggregate_many_reservoir_counts([circuit_sig])[str(circuit_sig)]


_COMPILE_CACHE = OrderedDict()
def _compile_reservoir_circuit(backend,pair,role,p_a,p_b,k,requested_treatment,
                               strict_override=False):
    # Cache transpilation by actual circuit+DD settings, never by logical query budget.
    tr=str(requested_treatment)
    ck=(_backend_name(backend),tuple(pair),str(role),int(k),
        round(float(p_a),14),None if p_b is None else round(float(p_b),14),
        tr,bool(strict_override), FIXED_KD if role=="dswap" else -1)
    with _COMPILE_LOCK:
        hit=_COMPILE_CACHE.get(ck)
        if hit is not None:
            _COMPILE_CACHE.move_to_end(ck)
            return hit[0],dict(hit[1]),hit[2]
        logical=build_amplitude_circuit(p_a,k) if role=="amplitude" else build_dswap_circuit(p_a,p_b,FIXED_KD)
        try:
            exe,stats=compile_alap_variant(logical,backend,pair,tr)
        except Exception:
            if strict_override and tr!="no_dd": raise
            tr="no_dd"
            exe,stats=compile_alap_variant(logical,backend,pair,tr)
        ret=(exe,dict(stats),tr)
        _COMPILE_CACHE[ck]=ret
        if len(_COMPILE_CACHE)>COMPILE_CACHE_SIZE:
            _COMPILE_CACHE.popitem(last=False)
        return exe,dict(stats),tr


def _pending_job_ids_for_circuit_sigs(circuit_sigs):
    """Indexed search for unfinished jobs; no full checkpoint-table scan."""
    wanted=list(dict.fromkeys(str(s) for s in circuit_sigs))
    if not wanted:
        return []
    found=[]; seen=set()
    with db() as con:
        for start in range(0,len(wanted),400):
            batch=wanted[start:start+400]
            marks=",".join("?" for _ in batch)
            try:
                rows=con.execute(f"""
                    SELECT DISTINCT i.job_id
                    FROM qpu_job_items i
                    LEFT JOIN qpu_counts c ON c.key=i.item_key
                    WHERE json_extract(i.stats_json,'$.circuit_sig') IN ({marks})
                      AND c.key IS NULL
                    ORDER BY i.created_at
                """,tuple(batch)).fetchall()
            except sqlite3.OperationalError:
                # Old SQLite without JSON1: keep a correct, slower fallback.
                all_rows=con.execute("""SELECT i.job_id,i.stats_json,c.key
                    FROM qpu_job_items i LEFT JOIN qpu_counts c ON c.key=i.item_key
                    WHERE c.key IS NULL ORDER BY i.created_at""").fetchall()
                rows=[]
                for jid,stats_json,_ in all_rows:
                    try: sig=str(json.loads(stats_json).get('circuit_sig',''))
                    except Exception: sig=''
                    if sig in batch:rows.append((jid,))
            for (jid,) in rows:
                if jid not in seen:
                    seen.add(jid); found.append(str(jid))
    return found


def _make_reservoir_request(backend,pair,role,seed,delta,arm_a,p_a,min_shots,k=0,
                            arm_b=None,p_b=None,dd_override=None,strict_override=False,
                            request_id=None,dswap_replay_scope=None):
    """Compile enough metadata to identify one reusable physical histogram."""
    k=int(k); min_shots=max(1,int(min_shots))
    req_tr=production_treatment(role,k,dd_override)
    # Frozen no-DD policy lets us derive the exact identity without compiling.
    # A circuit is compiled lazily only if its reservoir lacks physical shots.
    # All non-no-DD requests take the existing eager compile path, since their
    # chosen circuit treatment may fall back on compilation failure.
    if ENABLE_LAZY_CIRCUIT_COMPILE and req_tr=="no_dd" and not strict_override:
        exe=None; stats={}; tr="no_dd"
    else:
        exe,stats,tr=_compile_reservoir_circuit(
            backend,pair,role,float(p_a),None if p_b is None else float(p_b),k,req_tr,
            strict_override=strict_override
        )
    sig=_circuit_signature(
        backend,pair,role,seed,delta,arm_a,arm_b,k,tr,p_a,p_b
    )
    if str(role)=="dswap":
        if dswap_replay_scope is None:
            raise RuntimeError("V14 dSWAP requests require explicit QPA/max-shots replay scope")
        sig=hashlib.sha256(repr((sig,"v14_block_plan",tuple(dswap_replay_scope))).encode()).hexdigest()[:32]
    return {
        "request_id": str(request_id) if request_id is not None else sig,
        "role": str(role), "seed": int(seed), "delta": float(delta),
        "arm_a": int(arm_a), "arm_b": None if arm_b is None else int(arm_b),
        "p_a": float(p_a), "p_b": None if p_b is None else float(p_b),
        "k": int(k), "min_shots": int(min_shots),
        "dd_override": dd_override, "strict_override": bool(strict_override),
        "exe": exe, "stats": dict(stats), "treatment": str(tr), "circuit_sig": str(sig),
    }


def ensure_count_reservoir_batch(sampler,backend,pair,requests):
    """Ensure multiple physical count reservoirs, batching equal-size top-ups into Runtime jobs.

    This changes *transport only*. Each logical circuit, treatment, physical pair, and logical shot
    requirement is unchanged. Existing V5 count reservoirs are reused first.

    Returns one record per request_id with aggregate counts and the number of NEW shots submitted
    by this call. Multiple requests can be sent in the same Runtime job when their top-up shot counts
    match; run_compiled_batch still caps each Runtime submission at MAX_CIRCUITS_PER_JOB PUBs.
    """
    if not requests:
        return {}

    prepared=[]
    for req in requests:
        if isinstance(req,dict) and "exe" in req and "circuit_sig" in req:
            prepared.append(dict(req))
        else:
            prepared.append(_make_reservoir_request(backend,pair,**req))

    # First recover any older submitted job for these exact circuit signatures. This prevents a V6
    # 512-shot top-up from duplicating an unfinished V5 8192-shot job.
    sigs=[r["circuit_sig"] for r in prepared]
    if RESUME_PENDING_QPU_JOBS:
        for jid in _pending_job_ids_for_circuit_sigs(sigs):
            print("Recovering pre-existing reservoir IBM job:",jid)
            recover_qpu_job(jid)

    # Deduplicate identical physical circuit signatures within the same preparation call.
    # Use the maximum required reservoir size for that signature.
    merged={}
    request_ids_by_sig=defaultdict(list)
    for r in prepared:
        sig=r["circuit_sig"]
        request_ids_by_sig[sig].append(r["request_id"])
        if sig not in merged or int(r["min_shots"])>int(merged[sig]["min_shots"]):
            merged[sig]=dict(r)

    # Bulk reservoir lookup replaces one SQLite connection/query per circuit.
    existing=_aggregate_many_reservoir_counts(merged.keys())

    # Build missing top-up items. V9 uses SamplerV2's per-PUB shot counts, so
    # different top-up sizes within one (role,seed,delta) can share a Runtime job.
    # Every item still records its own exact physical/logical shot costs.
    groups=defaultdict(list)
    submitted_shots_by_sig=defaultdict(int)

    for sig,r in merged.items():
        counts,total,_=existing[sig]
        if total >= int(r["min_shots"]):
            continue

        minimum_chunk=AMP_TOPUP_SHOTS if r["role"]=="amplitude" else DSWAP_TOPUP_SHOTS
        topup=max(int(minimum_chunk),int(r["min_shots"])-int(total))
        target_total=int(total)+int(topup)

        if r["exe"] is None:
            exe,compiled_stats,actual_tr=_compile_reservoir_circuit(
                backend,pair,r["role"],float(r["p_a"]),
                None if r["p_b"] is None else float(r["p_b"]),
                int(r["k"]),r["treatment"],strict_override=r["strict_override"]
            )
            if actual_tr!=r["treatment"]:
                raise RuntimeError("Frozen lazy-compile treatment mismatch; refusing incorrect circuit reuse")
            r["exe"]=exe
            r["stats"]=dict(compiled_stats)
        stats=dict(r["stats"])
        stats.update({
            "circuit_k":int(r["k"]),
            "policy_key":_policy_key(r["role"],r["k"]),
            "oracle_queries_per_shot":oracle_uses_from_k(r["k"]) if r["role"]=="amplitude" else 2*oracle_uses_from_k(FIXED_KD),
            "kD":FIXED_KD if r["role"]=="dswap" else None,
            "circuit_sig":sig,
            "reservoir_target_total":int(target_total),
            "reservoir_chunk_shots":int(topup),
            "execution_protocol":EXECUTION_PROTOCOL,
        })

        # Item keys are new, but legacy pending reservoirs are recovered first by circuit_sig.
        item_key=hashlib.sha256(
            repr((REVISION,"reservoir_chunk",sig,int(target_total))).encode()
        ).hexdigest()[:28]

        item=(item_key,r["exe"],stats,int(r["arm_a"]),
              None if r["arm_b"] is None else int(r["arm_b"]),r["treatment"])

        group_key=((r["role"],int(r["seed"]),float(r["delta"]))
                   if ENABLE_MIXED_SHOT_PUBS else
                   (r["role"],int(r["seed"]),float(r["delta"]),int(topup)))
        groups[group_key].append((sig,item))
        submitted_shots_by_sig[sig]+=int(topup)

    # Submit grouped items. run_compiled_batch internally chunks at MAX_CIRCUITS_PER_JOB.
    for group_key,sig_items in groups.items():
        role,seed,delta=group_key[:3]
        items=[item for _,item in sig_items]
        shots_arg=None if ENABLE_MIXED_SHOT_PUBS else int(group_key[3])
        run_compiled_batch(sampler,items,shots_arg,role,int(seed),float(delta))

    # Verify and return aggregate histograms with another bulk query.
    completed=_aggregate_many_reservoir_counts(merged.keys())
    by_req={}
    for sig,r in merged.items():
        counts,total,nparts=completed[sig]
        if total < int(r["min_shots"]):
            raise RuntimeError(
                f"Reservoir {sig} has {total} shots after batched top-up; need {r['min_shots']}."
            )
        record={
            "counts":counts,
            "treatment":r["treatment"],
            "circuit_sig":sig,
            "total_shots":int(total),
            "new_physical_shots":int(submitted_shots_by_sig.get(sig,0)),
            "parts":int(nparts),
        }
        for rid in request_ids_by_sig[sig]:
            by_req[str(rid)]=dict(record)
    return by_req


def ensure_count_reservoir(sampler,backend,pair,role,seed,delta,arm_a,p_a,min_shots,k=0,
                           arm_b=None,p_b=None,dd_override=None,strict_override=False):
    """Backward-compatible single-reservoir wrapper around the V6 batched implementation."""
    rid="single"
    req=_make_reservoir_request(
        backend,pair,role,seed,delta,arm_a,p_a,min_shots,k=k,
        arm_b=arm_b,p_b=p_b,dd_override=dd_override,
        strict_override=strict_override,request_id=rid
    )
    rec=ensure_count_reservoir_batch(sampler,backend,pair,[req])[rid]
    return rec["counts"],rec["treatment"],rec["circuit_sig"],rec["total_shots"]


def _draw_multivariate_hypergeom(counts,shots,rng):
    counts={str(k):int(v) for k,v in counts.items() if int(v)>0}
    shots=int(shots); total=sum(counts.values())
    if shots<0 or shots>total:
        raise ValueError(f"Cannot draw {shots} samples from reservoir of size {total}.")
    if shots==0:
        return {k:0 for k in counts}
    keys=sorted(counts)
    out={k:0 for k in keys}
    rem_draw=shots; rem_total=total
    for k in keys[:-1]:
        good=counts[k]; bad=rem_total-good
        x=int(rng.hypergeometric(good,bad,rem_draw)) if rem_draw>0 else 0
        out[k]=x
        rem_draw-=x; rem_total-=good
    if keys:
        out[keys[-1]]=rem_draw
    return out


def _existing_replay_draws(config_key,circuit_sig):
    with db() as con:
        rows=con.execute("""
            SELECT draw_index,shots,counts_json FROM replay_samples
            WHERE config_key=? AND circuit_sig=? ORDER BY draw_index
        """,(str(config_key),str(circuit_sig))).fetchall()
    return [(int(i),int(s),json.loads(c)) for i,s,c in rows]


def _replay_sample_key(config_key,circuit_sig,draw_index,shots):
    return hashlib.sha256(
        repr((REVISION,"replay",str(config_key),str(circuit_sig),int(draw_index),int(shots))).encode()
    ).hexdigest()[:32]


def _replay_sample_exists(config_key,circuit_sig,draw_index,shots):
    key=_replay_sample_key(config_key,circuit_sig,draw_index,shots)
    with db() as con:
        return con.execute("SELECT 1 FROM replay_samples WHERE sample_key=?",(key,)).fetchone() is not None


def replay_draw_counts(config_key,circuit_sig,draw_index,shots,ensure_fn):
    """Return one deterministic, checkpointed hypergeometric draw from a physical histogram.

    Previous draws from the same circuit within the same algorithm configuration are removed before
    sampling, so repeated MLAE uses obey without-replacement accounting. Different QPA/k_R configs
    may replay the same physical histogram, which is exactly how the paper reconstructs query curves.
    """
    shots=int(shots); draw_index=int(draw_index)
    sample_key=_replay_sample_key(config_key,circuit_sig,draw_index,shots)
    with db() as con:
        row=con.execute("SELECT counts_json FROM replay_samples WHERE sample_key=?",(sample_key,)).fetchone()
    if row is not None:
        return json.loads(row[0])

    prior=_existing_replay_draws(config_key,circuit_sig)
    prior=[x for x in prior if x[0] < draw_index]
    prior_counts=Counter()
    prior_shots=0
    for _,s,c in prior:
        prior_shots+=int(s)
        prior_counts.update({str(k):int(v) for k,v in c.items()})

    required=prior_shots+shots
    aggregate,total=ensure_fn(required)
    aggregate=Counter({str(k):int(v) for k,v in aggregate.items()})
    aggregate.subtract(prior_counts)
    remaining={k:max(0,int(v)) for k,v in aggregate.items()}
    if sum(remaining.values()) < shots:
        raise RuntimeError("Replay reservoir accounting underflow; inspect checkpoint DB.")

    rng=np.random.default_rng(_stable_seed(sample_key))
    draw=_draw_multivariate_hypergeom(remaining,shots,rng)
    now=time.time()
    with db() as con:
        con.execute("""
            INSERT OR REPLACE INTO replay_samples
            (sample_key,config_key,circuit_sig,draw_index,shots,counts_json,source_total_shots,created_at)
            VALUES(?,?,?,?,?,?,?,?)
        """,(sample_key,str(config_key),str(circuit_sig),draw_index,shots,json.dumps(draw),int(total),now))
    return draw


def replay_dswap_counts(config_key,sampler,backend,pair,probs,seed,delta,king,challenger,shots,
                        dd_override=None,strict_override=False):
    req_tr=production_treatment("dswap",FIXED_KD,dd_override)
    exe,stats,tr=_compile_reservoir_circuit(
        backend,pair,"dswap",float(probs[king]),float(probs[challenger]),FIXED_KD,req_tr,
        strict_override=strict_override
    )
    sig=_circuit_signature(backend,pair,"dswap",seed,delta,king,challenger,FIXED_KD,tr,
                           float(probs[king]),float(probs[challenger]))
    def ensure(required):
        counts,tr2,sig2,total=ensure_count_reservoir(
            sampler,backend,pair,"dswap",seed,delta,int(king),float(probs[king]),required,k=FIXED_KD,
            arm_b=int(challenger),p_b=float(probs[challenger]),dd_override=dd_override,
            strict_override=strict_override
        )
        if sig2!=sig:
            raise RuntimeError("dSWAP treatment/signature changed during reservoir construction.")
        return counts,total
    return replay_draw_counts(config_key,sig,0,int(shots),ensure),tr,sig


def prefetch_dswap_block(config_key,sampler,backend,pair,probs,seed,delta,king,challengers,shots,
                         dd_override=None,strict_override=False):
    """Acquire only the next small adaptive block of dSWAP reservoirs against the current KING.

    The caller still processes challengers strictly in stream order. If KING changes, unused
    old-KING results are never consumed by the logical algorithm.
    """
    requests=[]
    for challenger in challengers:
        challenger=int(challenger)
        rid=f"dswap:{int(king)}:{challenger}"
        requests.append(_make_reservoir_request(
            backend,pair,"dswap",seed,delta,int(king),float(probs[king]),int(shots),k=FIXED_KD,
            arm_b=challenger,p_b=float(probs[challenger]),dd_override=dd_override,
            strict_override=strict_override,request_id=rid
        ))
    recs=ensure_count_reservoir_batch(sampler,backend,pair,requests)
    by_challenger={}
    for challenger in challengers:
        challenger=int(challenger)
        by_challenger[challenger]=recs[f"dswap:{int(king)}:{challenger}"]
    return by_challenger


class ArmMLAEPool:
    """Per-configuration replay pool over reusable physical amplitude histograms.

    V6 batches the currently required KING+CHALLENGER depth schedule before decoding. It never
    pre-measures later challenge levels or future challengers, so logical challenge semantics are unchanged.
    """
    def __init__(self,sampler,backend,pair,probs,seed,delta,config_key,dd_override=None,strict_override=False):
        self.sampler=sampler; self.backend=backend; self.pair=pair; self.probs=np.asarray(probs,float)
        self.seed=int(seed); self.delta=float(delta); self.config_key=str(config_key)
        self.dd_override=dd_override; self.strict_override=strict_override
        self.draw_index=defaultdict(int)

    def _draw_descriptor(self,arm,k,shots):
        arm=int(arm); k=int(k); shots=int(shots)
        req_tr=production_treatment("amplitude",k,self.dd_override)
        if ENABLE_LAZY_CIRCUIT_COMPILE and req_tr=="no_dd" and not self.strict_override:
            # Same circuit signature as an eager no-DD compilation, but avoid
            # transpiling when either the replay draw or QPU histogram exists.
            exe,stats,tr=None,{},"no_dd"
        else:
            exe,stats,tr=_compile_reservoir_circuit(
                self.backend,self.pair,"amplitude",float(self.probs[arm]),None,k,req_tr,
                strict_override=self.strict_override
            )
        sig=_circuit_signature(
            self.backend,self.pair,"amplitude",self.seed,self.delta,arm,None,k,tr,
            float(self.probs[arm]),None
        )
        idx=int(self.draw_index[sig])
        prior=_existing_replay_draws(self.config_key,sig)
        prior_shots=sum(int(s) for i,s,c in prior if int(i)<idx)
        return {
            "arm":arm,"k":k,"shots":shots,"treatment":tr,"sig":sig,"draw_index":idx,
            "prior_shots":int(prior_shots),"exe":exe,"stats":stats
        }

    def prefetch_guaranteed_challengers(self,challengers,kR,first_level_shots):
        """Batch the first MLAE level for future no-dSWAP challengers.

        Caller must prove that every challenger receives level one (b >= s_1).
        Challenger amplitudes are independent of which arm becomes KING, so
        these acquisitions remain valid through KING changes. No additional
        logical comparisons or future refinement levels are introduced.
        """
        reqs=[]
        for arm in challengers:
            arm=int(arm)
            for k in range(int(kR)+1):
                d=self._draw_descriptor(arm,k,int(first_level_shots))
                if _replay_sample_exists(self.config_key,d['sig'],d['draw_index'],d['shots']):
                    continue
                reqs.append(_make_reservoir_request(
                    self.backend,self.pair,'amplitude',self.seed,self.delta,
                    arm,float(self.probs[arm]),d['prior_shots']+int(first_level_shots),
                    k=k,dd_override=self.dd_override,
                    strict_override=self.strict_override,
                    request_id=f"guaranteed:amp:{arm}:{k}:{d['draw_index']}"
                ))
        return ensure_count_reservoir_batch(self.sampler,self.backend,self.pair,reqs) if reqs else {}

    def prepare_pair(self,arm_a,arm_b,kR,shots_per_depth):
        """Batch all *currently required* 0..kR circuits for both arms.

        Exactly the same per-depth logical shot count is retained. This only changes how missing
        physical histograms are submitted to IBM Runtime.
        """
        desc=[]
        for arm in [int(arm_a),int(arm_b)]:
            for k in range(int(kR)+1):
                d=self._draw_descriptor(arm,k,int(shots_per_depth))
                # If the exact replay draw already exists from a previous interrupted run, no reservoir
                # top-up is needed for that draw.
                if _replay_sample_exists(self.config_key,d["sig"],d["draw_index"],d["shots"]):
                    continue
                rid=f"amp:{arm}:{k}:{d['draw_index']}"
                desc.append(_make_reservoir_request(
                    self.backend,self.pair,"amplitude",self.seed,self.delta,arm,float(self.probs[arm]),
                    d["prior_shots"]+int(shots_per_depth),k=k,arm_b=None,p_b=None,
                    dd_override=self.dd_override,strict_override=self.strict_override,request_id=rid
                ))
        if desc:
            ensure_count_reservoir_batch(self.sampler,self.backend,self.pair,desc)

    def draw_rate(self,arm,k,shots):
        arm=int(arm); k=int(k); shots=int(shots)
        req_tr=production_treatment("amplitude",k,self.dd_override)
        if ENABLE_LAZY_CIRCUIT_COMPILE and req_tr=="no_dd" and not self.strict_override:
            # Same circuit signature as an eager no-DD compilation, but avoid
            # transpiling when either the replay draw or QPU histogram exists.
            exe,stats,tr=None,{},"no_dd"
        else:
            exe,stats,tr=_compile_reservoir_circuit(
                self.backend,self.pair,"amplitude",float(self.probs[arm]),None,k,req_tr,
                strict_override=self.strict_override
            )
        sig=_circuit_signature(self.backend,self.pair,"amplitude",self.seed,self.delta,arm,None,k,tr,
                               float(self.probs[arm]),None)
        idx=int(self.draw_index[sig]); self.draw_index[sig]+=1
        def ensure(required):
            counts,tr2,sig2,total=ensure_count_reservoir(
                self.sampler,self.backend,self.pair,"amplitude",self.seed,self.delta,arm,
                float(self.probs[arm]),required,k=k,dd_override=self.dd_override,
                strict_override=self.strict_override
            )
            if sig2!=sig:
                raise RuntimeError("Amplitude treatment/signature changed during reservoir construction.")
            return counts,total
        counts=replay_draw_counts(self.config_key,sig,idx,shots,ensure)
        return corrected_target(counts,"1",1)

    def estimate(self,arm,kR,shots_per_depth):
        rates=[]; shots=[]
        for k in range(int(kR)+1):
            rates.append(self.draw_rate(arm,k,int(shots_per_depth)))
            shots.append(int(shots_per_depth))
        successes=[r*s for r,s in zip(rates,shots)]
        return mlae_decode_successes(successes,shots,int(kR))

    def estimate_pair(self,arm_a,arm_b,kR,shots_per_depth):
        self.prepare_pair(arm_a,arm_b,kR,shots_per_depth)
        pa=self.estimate(arm_a,kR,shots_per_depth)
        pb=self.estimate(arm_b,kR,shots_per_depth)
        return pa,pb


In [ ]:
# ============================================================
# EMPIRICAL RAW-COUNT THRESHOLD (NO NEW GATE CALIBRATION)
# ============================================================
# V11 controls for delta=0.01 did not demonstrate useful CLOSE/FAR separation.
# The following threshold is intentionally more permissive. It is not certified.
# FAR when raw '11' hits >= 2; CLOSE (skip MLAE) for 0 or 1 hits.
# Preserve the original gate-characterization evidence; this empirical threshold is not certified.
_GATE_LOOKUP = {}
def required_hardware_gate(delta,qpa,dd_override=None):
    if not any(np.isclose(float(delta), d) for d in DELTAS) or dd_override is not None:
        raise RuntimeError("Exploratory raw threshold is only enabled for configured delta=0.01/0.05, production no-DD.")
    return PILOT_RAW_FAR_CUTOFF

assert nisq_dswap_allocation(1000*20000,1000)["m_d"] == 227
print("Exploratory raw >=2 FAR gate at both deltas; UNVALIDATED for delta=0.05")
print("Not a validated best-arm guarantee; old V11 gate controls remain archival.")


In [ ]:
# ============================================================
# Early-FAR stopping: physically measured, ordered Sampler blocks
# ============================================================
# The sampler returns one unordered histogram per PUB. We must NEVER split a
# single 227-shot histogram into fictitious chronological blocks. Every block
# below is independently submitted (or recovered from V13 checkpoint), and
# qpu_counts stores its own measured histogram. The circuit signature stays
# fixed across top-ups so compiled circuits remain reusable.

def early_dswap_block_plan(m_d):
    m_d=int(m_d)
    if m_d<=0:
        raise ValueError("dSWAP maximum shots must be positive")
    out=[]
    remaining=m_d
    while remaining>0:
        shots=min(int(DSWAP_EARLY_BLOCK_SHOTS),remaining)
        out.append(shots)
        remaining-=shots
    return out


def _ordered_physical_dswap_chunks(circuit_sig):
    """Preserve per-PUB histograms in physical submission order for restarts."""
    with db() as con:
        rows=con.execute("""
            SELECT c.shots,c.counts_json,c.job_id,c.key
            FROM qpu_counts c
            LEFT JOIN qpu_jobs j ON j.job_id=c.job_id
            WHERE c.circuit_sig=?
            ORDER BY COALESCE(j.submitted_at,c.created_at),c.created_at,c.key
        """,(str(circuit_sig),)).fetchall()
    parts=[]
    for shots,raw,jid,key in rows:
        counts={str(k):int(v) for k,v in json.loads(raw).items()}
        if sum(counts.values())!=int(shots):
            raise RuntimeError(f"QPU count/shot mismatch for {key}: {counts}, shots {shots}")
        parts.append({"shots":int(shots),"counts":counts,"job_id":jid,"key":key})
    return parts


def adaptive_dswap_measure(config_key,sampler,backend,pair,probs,seed,delta,
                           king,challenger,max_shots,qpa=None,dd_override=None,
                           strict_override=False):
    """Acquire up to max_shots, stopping once raw 11 hits >= 2.

    Every block is its own physical PUB. On a restart, read already-completed
    blocks in original physical order; recover previously-submitted IBM jobs
    using ensure_count_reservoir_batch before making any new submission.
    Logical queries count only the actually acquired blocks. The stopped
    decision has the same full-horizon threshold event under stationary iid
    Bernoulli shots, not a new theorem-level guarantee on noisy hardware.
    """
    plan=early_dswap_block_plan(max_shots)
    if qpa is None: raise ValueError("V14 early dSWAP requires query budget qpa for shot-plan isolation")
    req=_make_reservoir_request(
        backend,pair,"dswap",int(seed),float(delta),int(king),float(probs[king]),
        min_shots=int(plan[0]),k=FIXED_KD,arm_b=int(challenger),
        p_b=float(probs[challenger]),dd_override=dd_override,
        strict_override=strict_override,
        request_id=f"early:{int(king)}:{int(challenger)}",
        dswap_replay_scope=(int(qpa),int(max_shots),int(DSWAP_EARLY_BLOCK_SHOTS))
    )
    sig=str(req["circuit_sig"])
    total=0
    hits=0
    all_counts=Counter()
    job_ids=[]
    started=time.monotonic()
    for index,block_shots in enumerate(plan):
        parts=_ordered_physical_dswap_chunks(sig)
        if len(parts)>len(plan):
            raise RuntimeError("Unexpected extra physical blocks in V14 adaptive reservoir")
        if len(parts)<=index:
            # If a job is already submitted, ensure_count_reservoir_batch
            # recovers it by its saved job ID instead of submitting twice.
            expected_total=sum(plan[:index+1])
            ensure_count_reservoir_batch(
                sampler,backend,pair,[dict(req,min_shots=expected_total)]
            )
            parts=_ordered_physical_dswap_chunks(sig)
        if len(parts)<=index:
            raise RuntimeError(f"Missing physical shot block {index} for {sig}")
        part=parts[index]
        if int(part["shots"])!=int(block_shots):
            raise RuntimeError(
                f"Adaptive physical block {index} contains {part['shots']} shots, "
                f"but V14 expects {block_shots}. Use a fresh output directory."
            )
        total+=int(block_shots)
        hits+=int(part["counts"].get("11",0))
        all_counts.update(part["counts"])
        job_ids.append(part["job_id"])
        if hits>=int(PILOT_RAW_FAR_CUTOFF):
            break
    if total>int(max_shots) or len(job_ids)>len(plan):
        raise RuntimeError("Adaptive dSWAP exceeded its per-pair shot budget")
    if sum(all_counts.values())!=total:
        raise RuntimeError("Adaptive logical count accounting mismatch")
    return {
        "counts":dict(all_counts),"raw_11_hits":int(hits),"shots_used":int(total),
        "shots_cap":int(max_shots),"block_sizes":plan[:len(job_ids)],
        "num_blocks":len(job_ids),"job_ids":job_ids,"circuit_sig":sig,
        "is_far":bool(hits>=int(PILOT_RAW_FAR_CUTOFF)),
        "elapsed_wall_seconds":float(time.monotonic()-started),
        "treatment":req["treatment"],
    }


def prefetch_early_dswap_first_blocks(sampler,backend,pair,probs,seed,delta,qpa,
                                      king,challengers,max_shots,
                                      dd_override=None,strict_override=False):
    """Speculatively batch first *physical* adaptive blocks for nearby arms.

    Safe logical behavior: measured future-arm blocks are used only if the KING
    remains unchanged. If KING changes, these shots remain in physical_manifest
    as unused physical work; they are NEVER counted as logical dSWAP queries.
    Call only after a stable-KING streak, and limit the window for bounded waste.
    """
    if not challengers:
        return {}
    block_shots=early_dswap_block_plan(int(max_shots))[0]
    reqs=[]
    for challenger in challengers:
        ch=int(challenger)
        reqs.append(_make_reservoir_request(
            backend,pair,"dswap",int(seed),float(delta),int(king),float(probs[int(king)]),
            min_shots=int(block_shots),k=FIXED_KD,
            arm_b=ch,p_b=float(probs[ch]),dd_override=dd_override,
            strict_override=strict_override,request_id=f"pref0:{int(king)}:{ch}",
            dswap_replay_scope=(int(qpa),int(max_shots),int(DSWAP_EARLY_BLOCK_SHOTS))
        ))
    got=ensure_count_reservoir_batch(sampler,backend,pair,reqs)
    return {int(ch):got[f"pref0:{int(king)}:{int(ch)}"] for ch in challengers}


def save_adaptive_dswap_decision(config_key,record):
    with db() as con:
        con.execute("""
            INSERT OR REPLACE INTO adaptive_dswap_decisions
             (config_key,challenger,king_before,raw_11_hits,shots_used,shots_cap,
              is_far,best_false_close,block_sizes_json,job_ids_json,elapsed_wall_sec,
              circuit_sig,created_at)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
        """,(
            str(config_key),int(record["challenger"]),int(record["king_before"]),
            int(record["raw_11_hits"]),int(record["shots"]),int(record["shots_cap"]),
            int(record["is_far"]),int(record["best_false_close_this_encounter"]),
            json.dumps(record["block_sizes"]),json.dumps(record["job_ids"]),
            float(record["elapsed_wall_seconds"]),str(record["circuit_sig"]),time.time()
        ))
    if int(record["challenger"])%100==0 or int(record.get("true_best_encounter",0)):
        # Periodic, lightweight progress snapshot for sharing without waiting
        # for a 1000-arm stream to finish. SQLite remains authoritative.
        with db() as con:
            progress=pd.read_sql_query(
                "SELECT * FROM adaptive_dswap_decisions WHERE config_key=? ORDER BY challenger",
                con,params=(str(config_key),)
            )
        progress.to_csv(OUTDIR / (f"hardware_adaptive_dswap_delta{float(record['delta']):.2f}"
                                   f"_seed{int(record['seed'])}_qpa{int(record['qpa'])}"
                                   f"_kr{int(record['kR'])}.csv"),index=False)
        # Lightweight campaign-wide progress summary, sourced from committed SQLite rows.
        with db() as con:
            summary=pd.read_sql_query("""SELECT config_key,COUNT(*) AS challengers_processed,
                SUM(1-is_far) AS filtered, SUM(shots_used) AS shots_used,
                MAX(challenger) AS last_challenger,MAX(created_at) AS updated_at
                FROM adaptive_dswap_decisions GROUP BY config_key ORDER BY updated_at DESC""",con)
        summary.to_csv(OUTDIR/"hardware_adaptive_progress_summary.csv",index=False)

# Local mathematical invariant: the adaptive rule always yields the same
# FAR/CLOSE decision as a 227-shot fixed-horizon test on the same ordered shots.
assert early_dswap_block_plan(227)==[64,64,64,35]
for fake_shots in (["00"]*227, ["11"]+["00"]*226,
                   ["00"]*70+["11"]+["00"]*5+["11"]+["00"]*150):
    obtained=0; observed=0
    for block in early_dswap_block_plan(len(fake_shots)):
        observed+=fake_shots[obtained:obtained+block].count("11")
        obtained+=block
        if observed>=PILOT_RAW_FAR_CUTOFF: break
    assert (observed>=PILOT_RAW_FAR_CUTOFF)==(fake_shots.count("11")>=PILOT_RAW_FAR_CUTOFF)
print("Adaptive dSWAP: exact per-QPA 64-shot blocks; raw 11 >= 2 => stop FAR")

### Checkpointed job recovery

Previously submitted hardware jobs can be recovered from the original private checkpoint. No job resubmission is needed merely to inspect the archived results.


In [ ]:
# ============================================================
# Recover seed 8; optional replacement for verified closed-batch orphan
# Uses the ORIGINAL General-dedicated IBM instance (not the new submission CRN).
# Recovery may archive one old missing item ONLY with explicit opt-in; actual quantum submissions occur in normal seed runner.
# ============================================================
SEED8_RECOVERY_JOB_ID = os.getenv("NIGHTHAWK_RECOVERY_JOB_ID", "")
SEED8_ORIGINAL_BATCH_ID = os.getenv("NIGHTHAWK_RECOVERY_BATCH_ID", "")
SEED8_ORIGINAL_INSTANCE_NAME = os.getenv("NIGHTHAWK_RECOVERY_INSTANCE_NAME", "")
# When IBM reports an old job as QUEUED, do not claim it completed, wait
# indefinitely, or discard its saved circuit ownership. Let seeds 7/9 progress.
V143_SEED8_DEFERRED = False
# Default OFF. After verifying the exact job is QUEUED in its original CLOSED Batch,
# change this to True to approve replacing ONLY its MISSING checkpointed item(s).
# The job may already have consumed physical time even without accessible counts.
# NEVER enable globally for any other pending jobs.
SEED8_APPROVE_CLOSED_BATCH_ORPHAN_REPLACEMENT = False  # historical orphan already archived in existing checkpoint

def _release_only_seed8_closed_batch_orphan(job, batch_details):
    """Surgically detach one verified orphaned measurement; archive it first.

    Never treat a QUEUED job as an IBM-confirmed terminal failure. Executed
    physical shots may have been consumed; original job is preserved as an
    auditable entry and completed counts/physical manifests are untouched.
    """
    jid=SEED8_RECOVERY_JOB_ID
    if not SEED8_APPROVE_CLOSED_BATCH_ORPHAN_REPLACEMENT:
        raise RuntimeError("Explicit seed-8 replacement approval is OFF")
    if str(job.job_id()) != jid or _job_status_text(job,jid) != "QUEUED":
        raise RuntimeError("Job identity/status changed; refusing replacement")
    job_batch=getattr(job,"session_id",None)
    if callable(job_batch): job_batch=job_batch()
    if str(job_batch) != SEED8_ORIGINAL_BATCH_ID:
        raise RuntimeError(f"Job belongs to {job_batch!r}, not verified original batch. Refusing replacement")
    if not isinstance(batch_details,dict) or (str(batch_details.get("state","")).lower()!="closed"
            or batch_details.get("accepting_jobs") is not False
            or str(batch_details.get("id",""))!=SEED8_ORIGINAL_BATCH_ID):
        raise RuntimeError("Original batch is not confirmed CLOSED/NOT ACCEPTING; refusing replacement")
    _prepare_inaccessible_job_archive()
    reason=(f"MANUAL VERIFIED CLOSED-BATCH ORPHAN (not a confirmed IBM failure): "
            f"job {jid} status=QUEUED, batch={SEED8_ORIGINAL_BATCH_ID} state=CLOSED; "
            f"user explicitly approved possible duplicate physical shots")
    now=time.time()
    with db() as con:
        saved=con.execute("SELECT seed,state,submitted_at FROM qpu_jobs WHERE job_id=?",(jid,)).fetchone()
        if saved is None or int(saved[0]) != 8 or str(saved[1]).lower()=="completed":
            raise RuntimeError("Unexpected ledger seed, missing job, or already completed job")
        if saved[2] is None or float(saved[2]) >= _RECOVERY_PROCESS_STARTED_AT:
            raise RuntimeError("Not a job from a prior notebook execution; refusing duplicate-submission risk")
        rows=con.execute("""SELECT i.item_key,i.batch_index,i.arm_a,i.arm_b,
                               i.treatment,i.stats_json,i.created_at
                               FROM qpu_job_items i
                               LEFT JOIN qpu_counts c ON c.key=i.item_key
                               WHERE i.job_id=? AND c.key IS NULL""",(jid,)).fetchall()
        if len(rows)!=1:
            raise RuntimeError(f"Expected exactly 1 missing item for seed 8; found {len(rows)}. Manual review required")
        con.executemany("""INSERT OR IGNORE INTO inaccessible_job_items_archive
                     (item_key,old_job_id,batch_index,arm_a,arm_b,treatment,stats_json,
                      created_at,reason,archived_at)
                      VALUES(?,?,?,?,?,?,?,?,?,?)""",
                      [(r[0],jid,*r[1:],reason,now) for r in rows])
        con.execute("""DELETE FROM qpu_job_items WHERE job_id=? AND NOT EXISTS
                   (SELECT 1 FROM qpu_counts WHERE qpu_counts.key=qpu_job_items.item_key)""",(jid,))
        con.execute("""UPDATE qpu_jobs SET state='closed_batch_orphan_replaced',
                   last_error=?,updated_at=? WHERE job_id=?""",(reason[:1200],now,jid))
    print("AUDIT: detached 1 seed-8 measurement from CLOSED-BATCH orphan job",jid)
    print("Original job remains in qpu_jobs; archived missing item in inaccessible_job_items_archive")
    print("WARNING: replacement QPU shots may duplicate physical work from original job")
    return 1


def recover_verified_seed8_batch_job():
    global _PREVIOUS_RECOVERY_SERVICE, V143_SEED8_DEFERRED
    if not (RUN_QPU and RUN_HARDWARE_TRACE):
        print("Seed-8 recovery preflight skipped: QPU campaign disabled")
        return
    if INACCESSIBLE_JOB_POLICY != "stop":
        raise RuntimeError("Disable inaccessible-job replacement before recovering known completed IBM jobs")

    jid = SEED8_RECOVERY_JOB_ID
    with db() as con:
        row=con.execute("SELECT seed,state FROM qpu_jobs WHERE job_id=?", (jid,)).fetchone()
        if row is None or int(row[0]) != 8:
            raise RuntimeError(f"Expected saved seed-8 IBM job {jid} was not found in this V14 checkpoint")
        total=con.execute("SELECT COUNT(*) FROM qpu_job_items WHERE job_id=?",(jid,)).fetchone()[0]
        missing=con.execute("""SELECT COUNT(*) FROM qpu_job_items i
            LEFT JOIN qpu_counts c ON c.key=i.item_key
            WHERE i.job_id=? AND c.key IS NULL""",(jid,)).fetchone()[0]
    if total == 0:
        # On re-entry after an explicitly archived orphan replacement, the old
        # job must not be recovered or detached again. The new job owns any
        # replacement shots, and normal reservoir replay reads them as saved.
        with db() as con:
            archive_count=con.execute("""SELECT COUNT(*) FROM inaccessible_job_items_archive
                                          WHERE old_job_id=?""",(jid,)).fetchone()[0] if 'inaccessible_job_items_archive' in {
                                             r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")
                                          } else 0
        if row[1]=='closed_batch_orphan_replaced' and archive_count==1:
            V143_SEED8_DEFERRED=False
            print("Seed-8 old closed-batch job already archived; normal checkpoint reuse/resubmission may continue")
        else:
            raise RuntimeError(
                f"Job {jid} has no checkpointed item metadata; no matching audit entry"
            )
    # Even when this particular job is already recovered, verify and cache the
    # old instance so OTHER pending V14.2 jobs remain retrievable.

    from qiskit_ibm_runtime import QiskitRuntimeService
    # get_ibm_service resolves an existing token or prompts once, and stores it in the env.
    get_ibm_service()
    token = os.getenv("IBM_QUANTUM_TOKEN", "").strip()
    if not token:
        raise RuntimeError("IBM_QUANTUM_TOKEN not available to the recovery preflight")
    discovery = QiskitRuntimeService(channel="ibm_quantum_platform", token=token, region="us-east")
    matches=[x for x in discovery.instances() if x.get("name")==SEED8_ORIGINAL_INSTANCE_NAME]
    if len(matches)!=1 or not matches[0].get("crn"):
        raise RuntimeError(f"Cannot uniquely discover {SEED8_ORIGINAL_INSTANCE_NAME} in us-east")
    old_service = QiskitRuntimeService(channel="ibm_quantum_platform",token=token,
                                       instance=matches[0]["crn"])
    # Reuse this VERIFIED instance for later V14.2 pending-job recovery as well.
    with _PREVIOUS_RECOVERY_SERVICE_LOCK:
        _PREVIOUS_RECOVERY_SERVICE = old_service

    if total == 0:
        print("Seed-8 orphan already archived. Continuing experiment.")
        return
    print("Recovery instance verified:",SEED8_ORIGINAL_INSTANCE_NAME,
          "| saved job:",jid,"| missing measurement items:",missing)
    if missing == 0:
        V143_SEED8_DEFERRED = False
        print(f"Seed-8 job {jid}: {total} measurement(s) already saved; no QPU re-submission")
        return

    try:
        job=old_service.job(jid)
    except Exception as direct_error:
        if not _is_ibm_job_not_found(direct_error):
            raise
        # Optional fallback: lookup inside the batch the user verified in the IBM UI.
        # Do not silently replace/submit if the batch listing is incomplete.
        job=None
        for offset in range(0,2000,100):
            page=old_service.jobs(session_id=SEED8_ORIGINAL_BATCH_ID, limit=100, skip=offset)
            for candidate in page:
                if candidate.job_id()==jid:
                    job=candidate
                    break
            if job is not None or len(page)<100:
                break
        if job is None:
            raise RuntimeError(
                f"Could not retrieve job {jid} from {SEED8_ORIGINAL_INSTANCE_NAME} "
                f"or its recorded batch {SEED8_ORIGINAL_BATCH_ID}; no shots resubmitted"
            ) from direct_error

    status=_job_status_text(job,jid)
    if status == "QUEUED" and SEED8_APPROVE_CLOSED_BATCH_ORPHAN_REPLACEMENT:
        from qiskit_ibm_runtime import Batch
        original_batch = Batch.from_id(SEED8_ORIGINAL_BATCH_ID,service=old_service)
        details = original_batch.details()
        _release_only_seed8_closed_batch_orphan(job,details)
        V143_SEED8_DEFERRED = False
        print("Seed 8 unblocked: the next normal dSWAP run will use cached shots "
              "and acquire only the released missing measurement")
        return
    if status in ("INITIALIZING", "PENDING", "QUEUED", "RUNNING"):
        V143_SEED8_DEFERRED = True
        print(f"Seed 8 job {jid} is {status} on the ORIGINAL instance. "
              "The batch-level Completed label does not override this per-job status. "
              "Keeping its original measurement ownership; no duplicate shots. "
              "Seeds 7 and 9 may continue while seed 8 is deferred.")
        return
    if status not in ("DONE","COMPLETED"):
        raise RuntimeError(f"IBM reports seed-8 job {jid} status {status}. "
                           "Inspect this individual job before any replacement; no shots resubmitted")
    recover_qpu_job(jid,initial_job=job)  # V14.3.1 persists counts, manifest and completed status.
    with db() as con:
        missing_after=con.execute("""SELECT COUNT(*) FROM qpu_job_items i
            LEFT JOIN qpu_counts c ON c.key=i.item_key
            WHERE i.job_id=? AND c.key IS NULL""",(jid,)).fetchone()[0]
        saved_state=con.execute("SELECT state FROM qpu_jobs WHERE job_id=?",(jid,)).fetchone()[0]
    if missing_after:
        raise RuntimeError(f"Recovered {jid} but {missing_after} item(s) remain unsaved; stop safely")
    V143_SEED8_DEFERRED = False
    print("Recovered seed-8 batch job into V14 checkpoint:",jid,
          "| items:",total,"| state:",saved_state)

# Runs before the V14.3 QPA wave starts. Safe to rerun after restarting the kernel.
recover_verified_seed8_batch_job()


In [ ]:

# ============================================================
# Live hardware KING/CHALLENGER run with completed-config skipping + incremental exports
# V8: same logical algorithm, guaranteed first-level prefetch and faster physical acquisition.
# ============================================================
def _wilson_interval(successes,n,z=1.959963984540054):
    n=int(n)
    if n<=0: return (np.nan,np.nan)
    phat=float(successes)/n
    den=1.0+z*z/n
    center=(phat+z*z/(2*n))/den
    half=(z/den)*math.sqrt(phat*(1-phat)/n+z*z/(4*n*n))
    return max(0.0,center-half),min(1.0,center+half)


def export_partial_hardware_artifacts():
    """Refresh analysis-ready CSVs from whatever has finished so far."""
    with db() as con:
        hw=pd.read_sql_query("SELECT * FROM hardware_results ORDER BY created_at",con)

    if len(hw):
        hw["calibration_scope"] = np.where(pd.to_numeric(hw["qpa"])>=10000,
            "stageA_high_budget_screened", "low_budget_exploratory_or_underfunded")
    hw.to_csv(OUTDIR/"hardware_algorithm_results.csv",index=False)

    if len(hw):
        hw=hw[hw["status"]=="ok"].copy()
        if hw.empty:
            print("No supported completed configurations to summarize yet.")
            export_checkpoint_ledger()
            return hw
        rows=[]
        for keys,g in hw.groupby(["delta","qpa","method","kR","dd_mode"],dropna=False):
            delta,qpa,method,kR,dd_mode=keys
            ntr=len(g); s=int(g.success.sum()); lo,hi=_wilson_interval(s,ntr)
            reg=np.asarray(g.simple_regret,float)
            reg_mean=float(np.mean(reg)); reg_sem=float(np.std(reg,ddof=1)/math.sqrt(ntr)) if ntr>1 else np.nan
            row={
                "delta":delta,"qpa":int(qpa),"requested_total_queries":int(N*int(qpa)),
                "method":method,"kR":int(kR),"dd_mode":dd_mode,"trials":int(ntr),
                "success_rate":float(s/ntr),"success_ci95_low":lo,"success_ci95_high":hi,
                "mean_simple_regret":reg_mean,"regret_sem":reg_sem,
                "regret_ci95_low":reg_mean-1.96*reg_sem if np.isfinite(reg_sem) else np.nan,
                "regret_ci95_high":reg_mean+1.96*reg_sem if np.isfinite(reg_sem) else np.nan,
                "mean_queries_used":float(g.logical_queries_used.mean()),
                "mean_dswap_queries":float(g.dswap_queries_used.mean()),
                "mean_challenge_queries":float(g.challenge_queries_used.mean()),
                "mean_challenge_fraction":float(g.challenge_fraction.mean()),
                "mean_effective_candidate_fraction":float(g.effective_candidate_fraction.mean()),
                "best_false_close_rate":float(np.mean(np.asarray(g.best_false_close)>0)),
                "mean_max_k_used":float(g.max_k_used.mean()),
            }
            for c in ["prefetch_not_consumed_circuits","prefetch_not_consumed_shots",
                      "prefetch_not_consumed_oracle_queries"]:
                if c in g.columns:
                    row["mean_"+c]=float(pd.to_numeric(g[c],errors="coerce").fillna(0).mean())
            rows.append(row)
        hs=pd.DataFrame(rows).sort_values(["delta","method","kR","qpa"])
        hs.to_csv(OUTDIR/"hardware_summary.csv",index=False)

        q=hw[
            hw.method.isin(["theory_stream_dswap","theory_stream_no_dswap"])
            & (hw.dd_mode=="production")
        ]
        if len(q):
            values=["success","simple_regret","logical_queries_used",
                    "dswap_queries_used","challenge_queries_used",
                    "challenge_fraction","best_false_close"]
            p=q.pivot_table(
                index=["delta","seed","qpa","kR"],
                columns="method",
                values=values
            )
            p.columns=["__".join(c) for c in p.columns]
            p=p.reset_index()
            p.to_csv(OUTDIR/"hardware_paired_dswap_ablation.csv",index=False)

    export_checkpoint_ledger()
    try:
        export_physical_manifest()
    except Exception:
        pass
    return hw


def _store_hardware_result(key,row):
    now=time.time()
    with db() as con:
        con.execute("""
            INSERT OR REPLACE INTO hardware_results(
                key,method,delta,seed,qpa,kR,dd_mode,selected_arm,best_arm,success,simple_regret,
                logical_queries_used,dswap_queries_used,challenge_queries_used,
                close_skips,far_challenges,best_false_close,challenge_fraction,
                effective_candidate_fraction,max_k_used,budget_exhaustions,status,created_at,
                dswap_shots_per_pair,dswap_budget_fraction_target,dswap_budget_fraction_realized,
                prefetch_not_consumed_circuits,prefetch_not_consumed_shots,
                prefetch_not_consumed_oracle_queries,execution_protocol
            ) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """,(
            key,row["method"],float(row["delta"]),int(row["seed"]),int(row["qpa"]),int(row["kR"]),
            row["dd_mode"],int(row["selected_arm"]),int(row["best_arm"]),int(row["success"]),
            float(row["simple_regret"]),int(row["logical_queries_used"]),int(row["dswap_queries_used"]),
            int(row["challenge_queries_used"]),int(row["close_skips"]),int(row["far_challenges"]),
            int(row["best_false_close"]),float(row["challenge_fraction"]),
            float(row["effective_candidate_fraction"]),int(row["max_k_used"]),
            int(row["budget_exhaustions"]),str(row["status"]),now,
            int(row.get("dswap_shots_per_pair",0)),
            float(row.get("dswap_budget_fraction_target",0.0)),
            float(row.get("dswap_budget_fraction_realized",0.0)),
            int(row.get("prefetch_not_consumed_circuits",0)),
            int(row.get("prefetch_not_consumed_shots",0)),
            int(row.get("prefetch_not_consumed_oracle_queries",0)),
            str(row.get("execution_protocol",EXECUTION_PROTOCOL))
        ))
    maybe_export_progress()


def run_hardware_streaming(seed,delta,qpa,kR,use_dswap=True,dd_override=None,strict_dd_override=False):
    if not RUN_QPU:
        raise RuntimeError("Set RUN_QPU=True")

    kR=int(kR)
    method="theory_stream_dswap" if use_dswap else "theory_stream_no_dswap"
    dd_mode="production" if dd_override is None else str(dd_override)
    result_key=hardware_result_key(method,delta,seed,qpa,kR,dd_mode)

    # Only V7 logical results share this key; old V5/V6 results cannot be mislabelled.
    if SKIP_COMPLETED_HARDWARE_CONFIGS:
        existing=load_hardware_result(result_key)
        if existing is not None and str(existing.get("status")) in ("ok","dswap_underfunded","dswap_gate_unsupported"):
            print("RESUME: completed configuration already stored; skipping QPU work:",
                  method,"delta",delta,"seed",seed,"qpa",qpa,"kR",kR,"dd",dd_mode)
            return existing

    probs,meta=generate_instance(N,delta,seed)
    best_arm=int(meta["best_arm"]); best_p=float(meta["best_p"]); Q=int(N*qpa)
    theory=theory_dswap_parameters(delta)
    # Raw physical `11` gate, learned from independent controls in V11.
    # No ideal-theory threshold is used for hardware decisions.
    gate_raw_cutoff=None  # validate gate only after underfunded/unsupported checks

    alloc=nisq_dswap_allocation(Q,N) if use_dswap else {
        "m_d":0,"planned_queries":0,"realized_fraction":0.0
    }
    m_d=int(alloc["m_d"]); planned_d=int(alloc["planned_queries"])
    # Empirical exploratory gate runs across depths/budgets for delta=.01 and (unvalidated) delta=.05.
    # This is NOT a certified theorem-level or hardware-safe best-arm filter.
    gate_supported=bool(not use_dswap or any(np.isclose(float(delta),d) for d in DELTAS))
    if use_dswap and (m_d<MIN_DSWAP_SHOTS or planned_d>Q or not gate_supported):
        # Unsupported physical gates are neither executed nor scored as missed arms.
        unsupported_status=("dswap_underfunded" if m_d<MIN_DSWAP_SHOTS or planned_d>Q
                            else "dswap_gate_unsupported")
        row={"method":method,"delta":delta,"seed":seed,"qpa":qpa,"kR":kR,"dd_mode":dd_mode,
             "selected_arm":-1,"best_arm":best_arm,"success":0,"simple_regret":float("nan"),
             "logical_queries_used":0,"dswap_queries_used":0,"challenge_queries_used":0,
             "close_skips":0,"far_challenges":0,"best_false_close":0,"challenge_fraction":0.0,
             "effective_candidate_fraction":0.0,"max_k_used":-1,"budget_exhaustions":0,
             "dswap_shots_per_pair":m_d,"dswap_budget_fraction_target":DSWAP_BUDGET_FRACTION,
             "dswap_budget_fraction_realized":float(planned_d/max(1,Q)),
             "execution_protocol":EXECUTION_PROTOCOL,"status":unsupported_status}
        _store_hardware_result(result_key,row)
        return row
    if use_dswap:
        gate_raw_cutoff=required_hardware_gate(delta,qpa,dd_override)

    # Fair comparison with V12: reserve the same nominal maximum dSWAP budget
    # for determining the MLAE per-arrival allowance (b). Early stopping
    # lowers ACTUAL d_used, but does NOT silently give MLAE a larger allowance.
    # Unused reserved budget is reported as savings rather than reallocated.
    challenge_cap=Q-planned_d
    b=max(1,challenge_cap//(2*N))
    levels=challenge_level_schedule(b,kR)

    # State is reconstructed deterministically from V7 checkpointed replay draws.
    # Compatible legacy amplitude RESERVOIRS may be reused, but logical trajectories are new.
    pool=ArmMLAEPool(
        sampler,backend,PAIR,probs,seed,delta,config_key=result_key,
        dd_override=dd_override,strict_override=strict_dd_override
    )
    king=0; B=0; d_used=0; c_used=0
    close_skips=0; far=0; false_close=0
    dswap_decision_audit=[]
    max_k_used=-1; budget_exhaustions=0
    prefetch_not_consumed_circuits=0
    prefetch_not_consumed_shots=0
    prefetch_not_consumed_oracle_queries=0
    stable_king_streak=0

    idx=1
    prefetch_size=int(DSWAP_PREFETCH_INITIAL)
    # Algorithm 2 adds b before every challenger. If b >= s_1, all challengers
    # execute level one even when the KING changes and their prior bank is zero.
    guaranteed_no_dswap_first_level=(
        not use_dswap and ENABLE_NO_DSWAP_FIRST_LEVEL_BLOCK_PREFETCH
        and int(b)>=int(levels[0]))
    first_level_shots=max(1,int(levels[0])//mlae_schedule_cost(kR))
    no_dswap_block_size=max(1,min(int(NO_DSWAP_AMP_BLOCK_MAX),
                                   int(MAX_CIRCUITS_PER_JOB)//max(1,kR+1)))

    while idx<N:
        if use_dswap:
            if ENABLE_STABLE_KING_FIRST_BLOCK_PREFETCH and stable_king_streak>=int(DSWAP_STABLE_KING_MIN):
                block_size=int(DSWAP_FIRST_BLOCK_PREFETCH_WINDOW)
            else:
                block_size=1
        else:
            block_size=(no_dswap_block_size if guaranteed_no_dswap_first_level
                        else DSWAP_PREFETCH_INITIAL)
        block=np.arange(idx,min(N,idx+block_size),dtype=int)

        prefetched={}
        if guaranteed_no_dswap_first_level and len(block):
            pool.prefetch_guaranteed_challengers(block,kR,first_level_shots)
        if use_dswap and len(block)>1:
            # Only one 64-shot first block per challenger, all against the
            # CURRENT KING; these can be consumed in stream order or reported
            # as wasted physical shots if refinement changes KING.
            prefetched=prefetch_early_dswap_first_blocks(
                sampler,backend,PAIR,probs,seed,delta,qpa,king,[int(x) for x in block],
                m_d,dd_override=dd_override,strict_override=strict_dd_override)

        king_changed=False
        for pos,challenger in enumerate(block):
            challenger=int(challenger)
            B+=b
            is_far=True

            if use_dswap:
                # The reservoir should already exist from the small adaptive block prefetch. The replay
                # itself is still checkpointed and logically consumes exactly m_d samples.
                adaptive=adaptive_dswap_measure(
                    result_key,sampler,backend,PAIR,probs,seed,delta,
                    king,challenger,m_d,qpa=qpa,dd_override=dd_override,
                    strict_override=strict_dd_override
                )
                counts=adaptive["counts"]
                raw_11_hits=adaptive["raw_11_hits"]
                shots_used=adaptive["shots_used"]
                d_used+=2*oracle_uses_from_k(FIXED_KD)*shots_used
                is_far=bool(adaptive["is_far"])
                dswap_decision_audit.append({
                    "seed":int(seed),"delta":float(delta),"qpa":int(qpa),"kR":int(kR),
                    "challenger":int(challenger),"king_before":int(king),
                    "best_arm":int(best_arm),"true_best_encounter":int(challenger==best_arm),
                    "king_already_best":int(king==best_arm),
                    "raw_11_hits":int(raw_11_hits),"shots":int(shots_used),
                    "shots_cap":int(m_d),"num_blocks":int(adaptive["num_blocks"]),
                    "block_sizes":json.dumps(adaptive["block_sizes"]),
                    "job_ids":json.dumps(adaptive["job_ids"]),
                    "elapsed_wall_seconds":float(adaptive["elapsed_wall_seconds"]),
                    "circuit_sig":adaptive["circuit_sig"],
                    "far_cutoff":int(gate_raw_cutoff),"is_far":int(is_far),
                    "filtered_close":int(not is_far),
                    "best_false_close_this_encounter":int(challenger==best_arm and king!=best_arm and not is_far)
                })
                save_adaptive_dswap_decision(result_key,dswap_decision_audit[-1])
                if not is_far:
                    close_skips+=1
                    if challenger==best_arm and king!=best_arm:
                        false_close+=1

            if is_far:
                far+=1
                old=king
                resolved=False
                S=mlae_schedule_cost(kR)

                for s_l in levels:
                    if B<s_l:
                        king=challenger
                        B=0
                        budget_exhaustions+=1
                        resolved=True
                        break

                    shots=max(1,int(s_l)//S)
                    actual_per_arm=shots*S

                    # V6 batches the 2*(kR+1) CURRENTLY REQUIRED amplitude circuits. It does not
                    # pre-measure later challenge levels, so the logical algorithm is unchanged.
                    pk,pc=pool.estimate_pair(king,challenger,kR,shots)

                    c_used+=2*actual_per_arm
                    B-=int(s_l)
                    max_k_used=max(max_k_used,kR)

                    if pk>=pc:
                        resolved=True
                        break

                if not resolved:
                    king=challenger

                if king!=old:
                    # Any later dSWAP circuits in this block were measured against the old KING and are
                    # NOT consumed by the logical algorithm. Record only shots newly acquired by this
                    # prefetch call; pre-existing V5 data do not create new physical waste now.
                    if use_dswap:
                        for later in block[pos+1:]:
                            rec=prefetched.get(int(later),{})
                            newshots=int(rec.get("new_physical_shots",0))
                            if newshots>0:
                                prefetch_not_consumed_circuits+=1
                                prefetch_not_consumed_shots+=newshots
                                prefetch_not_consumed_oracle_queries+=2*oracle_uses_from_k(FIXED_KD)*newshots
                    idx=challenger+1
                    king_changed=True
                    prefetch_size=int(DSWAP_PREFETCH_INITIAL)
                    stable_king_streak=0
                    break

            idx=challenger+1
            if use_dswap:
                stable_king_streak+=1

        if not king_changed:
            idx=int(block[-1])+1
            if use_dswap:
                prefetch_size=min(
                    int(DSWAP_PREFETCH_MAX),
                    max(int(DSWAP_PREFETCH_INITIAL),int(prefetch_size)*int(DSWAP_PREFETCH_GROWTH))
                )

        if d_used+c_used>Q:
            raise RuntimeError("Logical query cap exceeded; inspect allocation.")

    row={
        "method":method,
        "delta":delta,"seed":seed,"qpa":qpa,"kR":kR,"dd_mode":dd_mode,
        "selected_arm":king,"best_arm":best_arm,"success":int(king==best_arm),
        "simple_regret":float(best_p-probs[king]),
        "logical_queries_used":int(d_used+c_used),
        "dswap_queries_used":int(d_used),
        "challenge_queries_used":int(c_used),
        "close_skips":int(close_skips),
        "far_challenges":int(far),
        "best_false_close":int(false_close),
        "challenge_fraction":float(far/max(1,N-1)),
        "effective_candidate_fraction":float((1+far)/N),
        "max_k_used":int(max_k_used),
        "budget_exhaustions":int(budget_exhaustions),
        "dswap_shots_per_pair":int(m_d),  # maximum, NOT realized average
        "dswap_budget_fraction_target":float(DSWAP_BUDGET_FRACTION) if use_dswap else 0.0,
        "dswap_budget_fraction_realized":float(planned_d/Q) if Q>0 else 0.0,
        "prefetch_not_consumed_circuits":int(prefetch_not_consumed_circuits),
        "prefetch_not_consumed_shots":int(prefetch_not_consumed_shots),
        "prefetch_not_consumed_oracle_queries":int(prefetch_not_consumed_oracle_queries),
        "execution_protocol":EXECUTION_PROTOCOL,
        "status":"ok"
    }

    _store_hardware_result(result_key,row)
    if use_dswap:
        # Additional cost audit: per-pair physical and logical shots are equal
        # by construction (one QPU histogram per adaptive block, no prefetched dSWAP).
        assert sum(int(r["shots"]) for r in dswap_decision_audit) * (2*oracle_uses_from_k(FIXED_KD)) == d_used
        audit_path=OUTDIR/f"hardware_pilot_dswap_decisions_delta{delta:.2f}_seed{seed}_qpa{qpa}_kr{kR}.csv"
        pd.DataFrame(dswap_decision_audit).to_csv(audit_path,index=False)
        print("Early-stop audit:",audit_path,"| best_false_close:",false_close,
              "| skipped:",close_skips,"| sent to challenge:",far,
              "| unused first-block physical shots:",prefetch_not_consumed_shots)
    return row


# ============================================================
# Checkpoint-compatible selective-baseline scheduler (historical adaptive selector disabled)
# Same run_hardware_streaming, hardware_result_key, SQLite tables, QPU reservoirs,
# and execution protocol as V14.2. New selection ledger is additive only.
# Each QPA wave finishes all 11 kR values for seeds 7, 8, and 9 BEFORE
# selecting the lowest-query successful dSWAP configurations for baseline.
# ============================================================
_CAMPAIGN_STOP=threading.Event()


def _v143_setup_ledger():
    with db() as con:
        con.execute("""CREATE TABLE IF NOT EXISTS v143_baseline_trials (
            delta REAL NOT NULL,
            seed INTEGER NOT NULL,
            qpa INTEGER NOT NULL,
            kR INTEGER NOT NULL,
            policy TEXT NOT NULL,
            state TEXT NOT NULL,
            ds_queries INTEGER,
            baseline_queries INTEGER,
            dswap_win INTEGER,
            updated_at REAL NOT NULL,
            PRIMARY KEY(delta, seed, qpa, kR)
        )""")


def _v143_trial_mark(delta,seed,qpa,kR,state,ds_queries=None,baseline_queries=None,dswap_win=None):
    with db() as con:
        con.execute("""INSERT INTO v143_baseline_trials
            (delta,seed,qpa,kR,policy,state,ds_queries,baseline_queries,dswap_win,updated_at)
            VALUES(?,?,?,?,?,?,?,?,?,?)
            ON CONFLICT(delta,seed,qpa,kR) DO UPDATE SET
                state=excluded.state,ds_queries=excluded.ds_queries,
                baseline_queries=excluded.baseline_queries,dswap_win=excluded.dswap_win,
                updated_at=excluded.updated_at""",
            (float(delta),int(seed),int(qpa),int(kR),"lowest_successful_dswap_queries",
             str(state),ds_queries,baseline_queries,dswap_win,time.time()))


def _v143_result(method,delta,seed,qpa,kR):
    key=hardware_result_key(str(method),float(delta),int(seed),int(qpa),int(kR),"production")
    return load_hardware_result(key)


def _v143_is_correct(result):
    return (result is not None and str(result.get("status"))=="ok"
            and int(result.get("success") or 0)==1)


def _v143_win(ds,baseline):
    return (_v143_is_correct(ds) and _v143_is_correct(baseline)
            and int(ds["logical_queries_used"])<int(baseline["logical_queries_used"]))


def _v143_existing_win():
    """Respect favorable matched results from V14.2 or an interrupted V14.3.

    Only evaluation seeds/QPA/kR and the unchanged production protocol qualify.
    """
    for delta in DELTAS:
        for seed in HARDWARE_EVAL_SEEDS:
            for qpa in HARDWARE_QPA_GRID:
                for kR in HARDWARE_KR_VALUES:
                    ds=_v143_result("theory_stream_dswap",delta,seed,qpa,kR)
                    if not _v143_is_correct(ds):
                        continue
                    base=_v143_result("theory_stream_no_dswap",delta,seed,qpa,kR)
                    if _v143_win(ds,base):
                        return (float(delta),int(seed),int(qpa),int(kR),
                                int(ds["logical_queries_used"]),int(base["logical_queries_used"]))
    return None


def _v143_run_dswap_seed_for_qpa(delta,seed,qpa):
    completed=[]
    for kR in HARDWARE_KR_VALUES:
        if _CAMPAIGN_STOP.is_set():
            break
        print(f"[Hardware dSWAP delta={delta} seed={seed}] qpa={qpa} kR={kR}")
        row=run_hardware_streaming(seed,delta,qpa,kR,True)
        completed.append(row)
    return completed


def _v143_run_group_dswap_wave(delta, seed_group, qpa):
    """Complete one QPA across this group (max 3 seeds), all kR per seed.

    Exactly one owner per seed and no concurrent delta waves. Existing checkpoints
    short-circuit individual completed configs. A pending seed-8 historical IBM
    job may be deferred, but the wave cannot be declared complete without it.
    """
    pairs=[(float(delta),int(seed)) for seed in seed_group
           if not (V143_SEED8_DEFERRED and int(seed)==8)]
    if V143_SEED8_DEFERRED and 8 in seed_group:
        print("Seed 8 deferred until its original job is recovered")
    rows=[]
    if MAX_PARALLEL_STREAMS==1:
        for d,s in pairs:
            rows.extend(_v143_run_dswap_seed_for_qpa(d,s,qpa))
    else:
        with ThreadPoolExecutor(max_workers=min(MAX_PARALLEL_STREAMS,len(pairs)),
                                thread_name_prefix="nighthawk-v1436-group") as executor:
            futures={executor.submit(_v143_run_dswap_seed_for_qpa,d,s,qpa):(d,s)
                     for d,s in pairs}
            for future in as_completed(futures):
                try:
                    rows.extend(future.result())
                except Exception as exc:
                    _CAMPAIGN_STOP.set()
                    for f in futures:
                        f.cancel()
                    print("Hardware group-wave error", futures[future], repr(exc),
                          "- SQLite checkpoint preserved")
                    raise
    return rows



def _v143_screen_baselines_for_qpa(qpa):
    if not V143_RUN_NEW_BASELINES:
        raise RuntimeError("Legacy adaptive no-dSWAP selector DISABLED. Use fixed V14.3.7 list only.")
    """Run at most a small number of NEW comparisons per QPA wave.

    Incomplete V14.3 trials are resumed first; remaining candidates are ranked
    by measured dSWAP query cost. Existing V14.2 completed baselines are reused.
    """
    if V143_STOP_BASELINES_AFTER_FIRST_WIN:
        winner=_v143_existing_win()
        if winner is not None:
            print("V14.3: existing matched dSWAP win found:",winner,
                  "| no more automatic no-dSWAP runs")
            return winner
    candidates=[]
    with db() as con:
        trial_states={(int(s),int(k)):str(st) for s,k,st in con.execute(
            "SELECT seed,kR,state FROM v143_baseline_trials WHERE delta=? AND qpa=?",
            (float(DELTAS[0]),int(qpa))).fetchall()}
    for delta in DELTAS:
        for seed in HARDWARE_EVAL_SEEDS:
            for kR in HARDWARE_KR_VALUES:
                ds=_v143_result("theory_stream_dswap",delta,seed,qpa,kR)
                if _v143_is_correct(ds):
                    candidates.append((int(ds["logical_queries_used"]),float(delta),int(seed),int(kR),ds))
    # Previously started V14.3 baselines receive resume priority; otherwise
    # prioritize the lowest successful dSWAP query count as requested.
    candidates.sort(key=lambda x:(0 if trial_states.get((x[2],x[3])) in
                                      ("started","interrupted") else 1,
                                  x[0],x[2],x[3]))
    new_trials=0
    for ds_queries,delta,seed,kR,ds in candidates:
        if _CAMPAIGN_STOP.is_set():break
        baseline=_v143_result("theory_stream_no_dswap",delta,seed,qpa,kR)
        if baseline is not None and str(baseline.get("status")) in (
                "ok","dswap_underfunded","dswap_gate_unsupported"):
            won=_v143_win(ds,baseline)
            _v143_trial_mark(delta,seed,qpa,kR,"matched_completed",ds_queries,
                             int(baseline.get("logical_queries_used") or 0),int(won))
            if won and V143_STOP_BASELINES_AFTER_FIRST_WIN:
                return (delta,seed,int(qpa),kR,ds_queries,
                        int(baseline["logical_queries_used"]))
            continue
        was_started=trial_states.get((seed,kR)) in ("started","interrupted")
        if not was_started and new_trials>=V143_BASELINE_TRIALS_PER_QPA:
            break
        if not was_started:
            new_trials+=1
        print(f"[Selected no-dSWAP] seed={seed} qpa={qpa} kR={kR} "
              f"dswap_success_queries={ds_queries:,} "
              f"({'RESUME' if was_started else 'NEW'})")
        _v143_trial_mark(delta,seed,qpa,kR,"started",ds_queries)
        try:
            # Same V14.2 runner: reuse completed results, replay draws,
            # circuit histograms, and any recoverable pending Runtime job ID.
            baseline=run_hardware_streaming(seed,delta,qpa,kR,False)
        except Exception:
            _v143_trial_mark(delta,seed,qpa,kR,"interrupted",ds_queries)
            raise
        won=_v143_win(ds,baseline)
        _v143_trial_mark(delta,seed,qpa,kR,"matched_completed",ds_queries,
                         int(baseline.get("logical_queries_used") or 0),int(won))
        if won and V143_STOP_BASELINES_AFTER_FIRST_WIN:
            print("V14.3 MATCHED WIN: dSWAP saved",
                  int(baseline["logical_queries_used"])-ds_queries,
                  "logical queries; future automatic baselines disabled")
            return (delta,seed,int(qpa),kR,ds_queries,
                    int(baseline["logical_queries_used"]))
    return None


def _v1437_targeted_pair_summary():
    """Read-only view: the exact six planned pairs, including pending entries."""
    records=[]
    for seed in V1437_TARGET_SEEDS:
        for kR in V1437_TARGET_KR_VALUES:
            ds = _v143_result("theory_stream_dswap",V1437_TARGET_DELTA,seed,V1437_TARGET_QPA,kR)
            nd = _v143_result("theory_stream_no_dswap",V1437_TARGET_DELTA,seed,V1437_TARGET_QPA,kR)
            ds_ok = ds is not None and str(ds.get("status"))=="ok"
            nd_ok = nd is not None and str(nd.get("status"))=="ok"
            ds_q = int(ds["logical_queries_used"]) if ds_ok else None
            nd_q = int(nd["logical_queries_used"]) if nd_ok else None
            records.append({
                "delta":V1437_TARGET_DELTA,"qpa":V1437_TARGET_QPA,
                "seed":int(seed),"kR":int(kR),
                "dswap_status": None if ds is None else str(ds.get("status")),
                "no_dswap_status":None if nd is None else str(nd.get("status")),
                "dswap_correct":int(ds["success"]) if ds_ok else None,
                "no_dswap_correct":int(nd["success"]) if nd_ok else None,
                "dswap_queries":ds_q,"no_dswap_queries":nd_q,
                "no_dswap_minus_dswap_queries":nd_q-ds_q if ds_ok and nd_ok else None,
                "dswap_saving_fraction_of_no_dswap":
                  (nd_q-ds_q)/nd_q if ds_ok and nd_ok and nd_q>0 else None,
                "state":"matched_completed" if ds_ok and nd_ok else "pending",
            })
    return pd.DataFrame.from_records(records)


def _v1437_export_targeted_pair_summary():
    """Atomic CSV refresh, never implying unfinished controls are measured."""
    df=_v1437_targeted_pair_summary()
    target=OUTDIR/"v1437_six_fixed_no_dswap_comparisons_delta005_qpa20000.csv"
    temp=target.with_suffix(".tmp")
    with _CSV_EXPORT_LOCK:
        df.to_csv(temp,index=False)
        os.replace(temp,target)
    return df


def _v1437_run_six_fixed_no_dswap_controls():
    """One-time, resumable fixed ablation after group-1 delta=.05 dSWAP sweep.

    ALL six matching dSWAP results must already be saved with status=ok.
    Existing completed no-dSWAP results are skipped via run_hardware_streaming;
    no cherry-picking, no win-based early stopping, no other seed/QPA/depth.
    """
    assert RUN_QPU and V1437_ENABLE_TARGETED_NO_DSWAP
    assert not V143_RUN_NEW_BASELINES, "Legacy adaptive baseline mode must remain off"
    assert (V1437_TARGET_DELTA,V1437_TARGET_QPA,V1437_TARGET_SEEDS,V1437_TARGET_KR_VALUES)==(
        0.05,20000,(7,8,9),(2,4)), "Unexpected extra baseline configurations"
    missing=[]
    for seed in V1437_TARGET_SEEDS:
        for kR in V1437_TARGET_KR_VALUES:
            ds=_v143_result("theory_stream_dswap",0.05,seed,20000,kR)
            if ds is None or str(ds.get("status"))!="ok":
                missing.append((seed,kR,None if ds is None else ds.get("status")))
    if missing:
        _v1437_export_targeted_pair_summary()
        raise RuntimeError("Refusing no-dSWAP baseline before all six matching dSWAP "
                           f"results are complete: {missing}")
    print("Starting/resuming six predefined no-dSWAP controls:",
          "delta=0.05, QPA=20000, seeds=(7,8,9), kR=(2,4)")
    for seed in V1437_TARGET_SEEDS:
        for kR in V1437_TARGET_KR_VALUES:
            if _CAMPAIGN_STOP.is_set():
                raise RuntimeError("Campaign stop requested before all fixed controls completed")
            print(f"[Fixed no-dSWAP] delta=0.05 seed={seed} QPA=20000 kR={kR}")
            nd=run_hardware_streaming(seed,0.05,20000,kR,False)
            # CHECKPOINT already commits hardware row. Export the latest comparison.
            _v1437_export_targeted_pair_summary()
            if str(nd.get("status"))!="ok":
                raise RuntimeError(f"No-dSWAP baseline not complete for seed={seed}, kR={kR}: "
                                   f"status={nd.get('status')}; checkpoint preserved")
    out=_v1437_export_targeted_pair_summary()
    if not bool((out["state"]=="matched_completed").all()):
        raise RuntimeError("Some fixed baseline comparisons are still incomplete")
    print("Completed all six predefined no-dSWAP comparisons")
    display(out)
    return out


def run_hardware_campaign():
    """Group -> .01/.05 full dSWAP sweeps -> 6 controls after group 1 -> other groups."""
    _CAMPAIGN_STOP.clear()
    _v143_setup_ledger()
    print("Group-major seed schedule:",HARDWARE_SEED_GROUPS)
    print("Within each group, deltas:",DELTAS,"| QPA:",HARDWARE_QPA_EXECUTION_ORDER,
          "| kR:",HARDWARE_KR_VALUES)
    print("Maximum concurrent seed streams:",MAX_PARALLEL_STREAMS,
          "| maximum in-flight IBM submissions:",MAX_INFLIGHT_SUBMISSIONS)
    print("Legacy adaptive no-dSWAP disabled:",not V143_RUN_NEW_BASELINES)
    print("Six predefined no-dSWAP controls enabled:",V1437_ENABLE_TARGETED_NO_DSWAP)
    assert not V143_RUN_NEW_BASELINES, "Legacy adaptive no-dSWAP runs must remain disabled"
    assert len(HARDWARE_EVAL_SEEDS)==20 and set(HARDWARE_EVAL_SEEDS)==set(range(7,27))
    for group_idx,seed_group in enumerate(HARDWARE_SEED_GROUPS,1):
        if _CAMPAIGN_STOP.is_set(): break
        print(f"\n===== SEED GROUP {group_idx}/{len(HARDWARE_SEED_GROUPS)}: {seed_group} =====")
        for delta in DELTAS:
            if _CAMPAIGN_STOP.is_set(): break
            print(f"\n----- BEGIN DELTA {delta:.2f}, GROUP {seed_group}, clustered_lowvar -----")
            for qpa in HARDWARE_QPA_EXECUTION_ORDER:
                if _CAMPAIGN_STOP.is_set(): break
                print(f"\nHardware group={group_idx} seeds={seed_group} delta={delta:.2f} QPA={qpa}")
                _v143_run_group_dswap_wave(delta,seed_group,qpa)
                # If seed 8 was deferred, check its original job AFTER peers finish.
                # Do not advance into the next QPA/delta/group until it is recovered.
                if V143_SEED8_DEFERRED and 8 in seed_group:
                    print("Rechecking verified original seed-8 batch job")
                    recover_verified_seed8_batch_job()
                    if not V143_SEED8_DEFERRED:
                        _v143_run_dswap_seed_for_qpa(delta,8,qpa)
                with _CSV_EXPORT_LOCK:
                    export_partial_hardware_artifacts()
                if V143_SEED8_DEFERRED and 8 in seed_group:
                    print("Seed 8 remains pending; all results saved. Stop before advancing wave.")
                    return export_partial_hardware_artifacts()
            print(f"----- FINISHED DELTA {delta:.2f}, GROUP {seed_group} -----")
            # Strict new ablation barrier: only AFTER all 11 QPAs and all 11 depths
            # have finished for group 1 at delta=.05. No other groups get baselines.
            if (group_idx==1 and np.isclose(float(delta),V1437_TARGET_DELTA)
                    and V1437_ENABLE_TARGETED_NO_DSWAP):
                _v1437_run_six_fixed_no_dswap_controls()
                with _CSV_EXPORT_LOCK:
                    export_partial_hardware_artifacts()
    return export_partial_hardware_artifacts()


# Wrap CSV export now that all downstream reporting functions are defined.
_raw_export_partial_hardware_artifacts=export_partial_hardware_artifacts
def export_partial_hardware_artifacts(*args,**kwargs):
    with _CSV_EXPORT_LOCK: return _raw_export_partial_hardware_artifacts(*args,**kwargs)

if RUN_QPU and RUN_HARDWARE_TRACE:
    try:
        hw=run_hardware_campaign()
        display(hw)
    finally:
        # The batch is closed only when every worker has completed or failed.
        # Completed/pending job IDs are safe in SQLite for the next resume.
        print("Hardware campaign ended; saved checkpoint:",DB_PATH)


### Matched dSWAP comparisons

The experiment includes predefined matched no-dSWAP controls. Report favorable and unfavorable results, including actual query use and success/regret.


In [ ]:
# Read-only summary: safe to run or rerun; does NOT submit to IBM.
if RUN_QPU:
    _v1437_targeted_status = _v1437_export_targeted_pair_summary()
    display(_v1437_targeted_status)


In [ ]:
# ============================================================
# Optional end-to-end DD vs no-DD ablation
# ============================================================
def run_end_to_end_dd_ablation():
    """Small paired algorithm-level ablation. A requested XX/XY4 run is accepted only if
    the complete sequence is physically inserted on every circuit that needs it; otherwise that
    configuration is recorded as unsupported rather than silently becoming no-DD.
    """
    if not RUN_QPU:
        raise RuntimeError("Set RUN_QPU=True")
    rows=[]
    for delta in DD_ABLATION_DELTAS:
        for seed in DD_ABLATION_SEEDS:
            for kR in DD_ABLATION_KR_VALUES:
                for tr in ["no_dd","xx","xy4"]:
                    try:
                        r=run_hardware_streaming(seed,delta,DD_ABLATION_QPA,kR,True,
                            dd_override=tr,strict_dd_override=True)
                        r["dd_ablation_supported"]=1
                    except Exception as exc:
                        r={"method":"theory_stream_dswap","delta":delta,"seed":seed,"qpa":DD_ABLATION_QPA,
                           "kR":kR,"dd_mode":tr,"status":"unsupported_dd_insertion",
                           "dd_ablation_supported":0,"error":repr(exc)}
                    rows.append(r)
    df=pd.DataFrame(rows); df.to_csv(OUTDIR/"end_to_end_dd_ablation.csv",index=False); display(df); return df

if RUN_QPU and RUN_END_TO_END_DD_ABLATION:
    dd_e2e=run_end_to_end_dd_ablation()


In [ ]:
# ============================================================
# Matched classical baselines for hardware instances
# ============================================================
def hardware_instance_classical_baselines(qpa_grid=HARDWARE_QPA_GRID):
    rows=[]
    for delta in DELTAS:
        for seed in HARDWARE_EVAL_SEEDS:
            probs,meta=generate_instance(N,delta,seed)
            for qpa in qpa_grid:
                Q=N*int(qpa)
                rng=np.random.default_rng(seed*100000+qpa+777); pred,used=sequential_halving(probs,Q,rng)
                rows.append({"method":"sequential_halving","delta":delta,"seed":seed,"qpa":qpa,"kR":-1,
                             "requested_total_queries":Q,"selected_arm":pred,"best_arm":meta["best_arm"],"success":int(pred==meta["best_arm"]),
                             "simple_regret":float(meta["best_p"]-probs[pred]),"logical_queries_used":used})
                rng=np.random.default_rng(seed*100000+qpa+999); pred,used=uniform_sampling(probs,Q,rng)
                rows.append({"method":"uniform_sampling","delta":delta,"seed":seed,"qpa":qpa,"kR":-1,
                             "requested_total_queries":Q,"selected_arm":pred,"best_arm":meta["best_arm"],"success":int(pred==meta["best_arm"]),
                             "simple_regret":float(meta["best_p"]-probs[pred]),"logical_queries_used":used})
    df=pd.DataFrame(rows); df.to_csv(OUTDIR/"hardware_instance_classical_baselines.csv",index=False); return df

classical_hw=hardware_instance_classical_baselines()
display(classical_hw)


In [ ]:
# ============================================================
# Paper-style hardware query curves (safe to rerun from partial checkpoints)
# ============================================================
def _load_hardware_curve_frames():
    hsp=OUTDIR/"hardware_summary.csv"
    if not hsp.exists():
        export_partial_hardware_artifacts()
    hs=pd.read_csv(hsp) if hsp.exists() else pd.DataFrame()
    cp=OUTDIR/"hardware_instance_classical_baselines.csv"
    cb=pd.read_csv(cp) if cp.exists() else hardware_instance_classical_baselines()
    if len(cb):
        cb=(cb.groupby(["delta","qpa","method"],as_index=False)
              .agg(trials=("success","size"),success_rate=("success","mean"),
                   mean_simple_regret=("simple_regret","mean")))
        cb["requested_total_queries"]=N*cb["qpa"].astype(int)
    return hs,cb


def plot_hardware_paper_curves():
    hs,cb=_load_hardware_curve_frames()
    if len(hs)==0:
        print("No completed hardware algorithm rows yet.")
        return []
    made=[]
    prod=hs[(hs.method=="theory_stream_dswap") & (hs.dd_mode=="production")].copy()
    for delta in DELTAS:
        q=prod[prod.delta==delta]
        if len(q)==0: continue

        fig,ax=plt.subplots(figsize=(8.0,5.2))
        for kR in KR_PLOT_VALUES:
            g=q[q.kR==kR].sort_values("requested_total_queries")
            if len(g):
                ax.plot(g.requested_total_queries,g.success_rate,marker="o",label=f"kR={kR}")
        for method,ls,label in [("sequential_halving","--","Seq. Halving"),("uniform_sampling",":","Uniform")]:
            g=cb[(cb.delta==delta)&(cb.method==method)].sort_values("requested_total_queries")
            if len(g): ax.plot(g.requested_total_queries,g.success_rate,ls=ls,linewidth=2,label=label)
        ax.set_xscale("log"); ax.set_ylim(-0.02,1.02)
        ax.set_xlabel("Total oracle-query budget Q"); ax.set_ylabel("Best-arm success rate")
        ax.set_title(f"IBM hardware: success, Δ={delta}, n={N}")
        ax.grid(True,which="both",alpha=.25); ax.legend(ncol=2,fontsize=8)
        fig.tight_layout(); p=FIGDIR/f"hardware_success_delta_{delta:.2f}.png"; fig.savefig(p,dpi=220); plt.show(); made.append(p)

        fig,ax=plt.subplots(figsize=(8.0,5.2))
        for kR in KR_PLOT_VALUES:
            g=q[q.kR==kR].sort_values("requested_total_queries")
            if len(g):
                ax.plot(g.requested_total_queries,g.mean_simple_regret,marker="o",label=f"kR={kR}")
        for method,ls,label in [("sequential_halving","--","Seq. Halving"),("uniform_sampling",":","Uniform")]:
            g=cb[(cb.delta==delta)&(cb.method==method)].sort_values("requested_total_queries")
            if len(g): ax.plot(g.requested_total_queries,g.mean_simple_regret,ls=ls,linewidth=2,label=label)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("Total oracle-query budget Q"); ax.set_ylabel("Mean simple regret")
        ax.set_title(f"IBM hardware: simple regret, Δ={delta}, n={N}")
        ax.grid(True,which="both",alpha=.25); ax.legend(ncol=2,fontsize=8)
        fig.tight_layout(); p=FIGDIR/f"hardware_regret_delta_{delta:.2f}.png"; fig.savefig(p,dpi=220); plt.show(); made.append(p)
    return made

hardware_curve_files=plot_hardware_paper_curves()


In [ ]:
# ============================================================
# Reviewer-facing physical resource summary
# ============================================================
def summarize_physical_resources():
    df=export_physical_manifest()
    if df is None or len(df)==0:
        print("No QPU manifest yet. Run the QPU cells first.")
        return pd.DataFrame()
    numeric=[c for c in ["depth","size","oneq_gates","twoq_gates","measurements","barriers","max_idle_us","total_idle_us",
                         "scheduled_duration_us","min_t1_t2_us","min_t2_us","rho_t2","rho_t1_t2","retention_proxy_t2",
                         "logical_oracle_queries","circuit_executions","estimated_scheduled_qpu_time_sec","submit_wall_sec"] if c in df.columns]
    summary=df.groupby(["role","treatment"],as_index=False)[numeric].agg(["count","mean","median","min","max"])
    summary.to_csv(OUTDIR/"physical_resource_summary.csv")

    total=pd.DataFrame([{
        "backend": IBM_BACKEND_NAME,
        "selected_pair": "" if PAIR is None else f"{PAIR[0]}-{PAIR[1]}",
        "n_arms": N,
        "max_simultaneous_logical_arm_qubits": 2,
        "submitted_circuit_records": int(len(df)),
        "total_circuit_executions_shots": int(df["circuit_executions"].sum()),
        "total_logical_oracle_queries_executed_in_bank": int(df["logical_oracle_queries"].sum()),
        "estimated_total_scheduled_qpu_time_sec": float(df["estimated_scheduled_qpu_time_sec"].sum()),
        "observed_sum_submit_to_result_wall_sec": float(df.groupby("job_id")["submit_wall_sec"].max().sum()),
    }])
    total.to_csv(OUTDIR/"reviewer_physical_resource_totals.csv",index=False)
    display(total)
    return summary

physical_summary=summarize_physical_resources()


## Results interpretation

The QPU results use an empirical early-stopping rule and must not be interpreted as a universal speedup or as a direct validation of the active-memory theorem.


In [ ]:
# ============================================================
# Final paper tables: safe to run at any time, including mid-campaign
# ============================================================
def build_paper_tables():
    outputs={}
    sim_path=OUTDIR/"simulator_summary.csv"
    if sim_path.exists():
        sim=pd.read_csv(sim_path)
        outputs["simulator_summary"]=sim
        display(sim)

    hw=export_partial_hardware_artifacts()
    if len(hw):
        hsp=OUTDIR/"hardware_summary.csv"
        if hsp.exists():
            outputs["hardware_summary"]=pd.read_csv(hsp)
            display(outputs["hardware_summary"])

        pp=OUTDIR/"hardware_paired_dswap_ablation.csv"
        if pp.exists():
            outputs["hardware_paired_dswap_ablation"]=pd.read_csv(pp)
            display(outputs["hardware_paired_dswap_ablation"])

    manifest=export_physical_manifest()
    if len(manifest):
        outputs["physical_manifest"]=manifest

    ddp=OUTDIR/"dd_calibration_raw.csv"
    if ddp.exists():
        outputs["dd_calibration_raw"]=pd.read_csv(ddp)

    ledger=export_checkpoint_ledger()
    outputs["qpu_job_ledger"]=ledger["jobs"]
    outputs["hardware_results_partial"]=ledger["hardware_results"]
    return outputs


paper_outputs=build_paper_tables()
print("Artifacts written under:",OUTDIR.resolve())
print("You may run this cell at any time while/after the campaign; it only reads checkpointed data.")


In [ ]:
# ============================================================
# Checkpoint performance instrumentation (does not submit to IBM)
# ============================================================
def checkpoint_performance_report():
    """Inspect existing checkpoint and estimate where job overhead is accumulating.

    qpu_jobs.shots=0 denotes heterogeneous per-PUB shots; individual stats_json
    contains each circuit's actual count. Reports physical acquisition, not
    logically charged replay sampling, and never claims measured speedups.
    """
    with db() as con:
        jobs=pd.read_sql_query("SELECT job_id,role,seed,delta,shots,item_count,submitted_at,completed_at,state FROM qpu_jobs",con)
        by_item=pd.read_sql_query("SELECT q.job_id,q.shots,q.role,q.circuit_executions,q.logical_oracle_queries FROM physical_manifest q",con)
        hw=pd.read_sql_query("SELECT status,close_skips,far_challenges,dswap_queries_used,challenge_queries_used,prefetch_not_consumed_shots FROM hardware_results",con)
    if jobs.empty:
        print("No QPU jobs in checkpoint; run the live pilot first.")
        return {}
    done=jobs[jobs.state.str.lower().isin(['completed','done'])].copy()
    report={
      'jobs_total':int(len(jobs)),
      'jobs_completed':int(len(done)),
      'circuits_per_job_median':float(jobs.item_count.median()),
      'circuits_per_job_p90':float(jobs.item_count.quantile(.9)),
      'mixed_shot_jobs':int((jobs.shots==0).sum()),
      'job_wall_seconds_median':float((done.completed_at-done.submitted_at).median()) if len(done) else None,
      'physical_circuit_executions':int(by_item.circuit_executions.sum()) if len(by_item) else 0,
      'physical_queries_acquired':int(by_item.logical_oracle_queries.sum()) if len(by_item) else 0,
      'unused_prefetch_shots_recorded':int(hw.prefetch_not_consumed_shots.fillna(0).sum()) if len(hw) else 0,
    }
    for k,v in report.items(): print(f'{k}: {v}')
    return report

if not RUN_QPU:
    # Empty/new offline checkpoints are valid; don't add network traffic.
    checkpoint_performance_report()


In [ ]:
# Close the accepting Batch only after the hardware and optional DD ablation finish.
if RUN_QPU and hasattr(sampler,"close"):
    sampler.close()
    print("Closed IBM Runtime Batch after completed notebook workflow.")


## Reproducing hardware experiments

Live continuation requires the original SQLite checkpoint and compatible IBM backend configuration. Do not run two processes against one checkpoint.


## Reporting conventions

Underfunded trials must be identified separately. Do not equate nominal budgets with actual oracle uses or physical QPU execution time.

## Historical identifiers and checkpoint compatibility

This is a single continuation experiment, not a collection of separate notebook releases. Some older identifiers deliberately remain in executable code and archival filenames to preserve the SQLite schema, job signatures, raw hardware counts, IBM recovery paths, and experiment provenance. In particular, do not rename `REVISION`, `EXECUTION_PROTOCOL`, `LEGACY_AMPLITUDE_REVISION`, `V143*`/`V1437*` variables, `v143_baseline_trials`, or historical audit paths without a checkpoint migration. These identifiers are not separate experimental configurations.

The final cells are read-only: checkpoint performance report, campaign dashboard, transport/scientific invariants, and recovery/checkpoint invariants. They do not submit IBM QPU jobs.


## Runtime and checkpoints

Hardware jobs may be expensive and subject to backend availability. The public repository includes aggregated results rather than private job ledgers and physical measurement banks.


In [ ]:
# Resumable campaign dashboard: read-only, no new QPU jobs submitted.
def campaign_dashboard():
    with db() as con:
        finished=pd.read_sql_query("SELECT * FROM hardware_results",con)
        progress=pd.read_sql_query("""SELECT config_key,MAX(challenger) AS last_challenger,
            SUM(1-is_far) AS filtered,SUM(shots_used) AS dswap_shots
            FROM adaptive_dswap_decisions GROUP BY config_key ORDER BY last_challenger DESC""",con)
        jobs=pd.read_sql_query("SELECT state,COUNT(*) n FROM qpu_jobs GROUP BY state",con)
    target=len(DELTAS)*len(HARDWARE_EVAL_SEEDS)*len(HARDWARE_QPA_GRID)*len(HARDWARE_KR_VALUES)
    ds=finished[(finished.method=="theory_stream_dswap") & (finished.seed.isin(HARDWARE_EVAL_SEEDS))] if len(finished) else finished
    ok=int((ds.status=="ok").sum()) if len(ds) else 0
    under=int(ds.status.str.contains("underfunded|unsupported",na=False).sum()) if len(ds) else 0
    print(f"Completed dSWAP: {ok:,}/{target:,} configurations ({ok/target*100:.3f}%)")
    print(f"Unsupported / underfunded: {under:,}; incomplete or not yet run: {target-len(ds):,}")
    print("Completed matched baseline results:",int((finished.method=="theory_stream_no_dswap").sum()) if len(finished) else 0)
    print("Physical Runtime jobs by checkpoint state:")
    display(jobs)
    if len(progress):
        print("Most advanced currently checkpointed streams:")
        display(progress.head(10))
    return {"completed":ok,"target":target,"underfunded":under,"jobs":jobs,"progress":progress}

campaign_dashboard()

In [ ]:
# Transport and scientific-configuration invariants; no IBM jobs required.
assert MAX_CIRCUITS_PER_JOB == 240
assert NO_DSWAP_AMP_BLOCK_MAX == 240
assert MAX_EXECUTIONS_PER_JOB == 9_000_000
assert (FIXED_KD, DSWAP_BUDGET_FRACTION, PILOT_RAW_FAR_CUTOFF) == (5,0.25,2)
assert DSWAP_EARLY_BLOCK_SHOTS == 64
print("Transport limits and frozen scientific settings verified; no QPU submission")

In [ ]:
# Recovery and checkpoint identity invariants; no IBM jobs submitted.
assert RUNTIME_RECOVERY_ATTEMPTS >= 1
assert AUTO_RESUBMIT_CONFIRMED_TERMINAL_JOBS
assert REVISION == "theory_faithful_system1_v14_kd5_f25_t2_early64_full_delta001"
assert FIXED_KD == 5 and abs(DSWAP_BUDGET_FRACTION-0.25) < 1e-12
print("Recovery settings verified; existing checkpoint identities remain unchanged.")